<div style="background-color: black;">
    <hr style="border: 3px solid skyblue;">
<div style="text-align: center; margin-left: 0em; font-weight: bold; font-size: 20px; font-family: TimesNewRoman; color: skyblue;">
    TECHNO - ECONOMIC PARAMETERS DATA PROCESSING
    <br>
    POWER PLANTS
</div>
<div style="text-align: center; margin-left: 0em; font-weight: bold; font-size: 18px; font-family: TimesNewRoman; color:skyblue">
    Main Formatting Notebook
    <br>
    from PYPSA to DISPA-SET
</div>
<br>
<div style="text-align: justify; margin-left: 0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
    Each part of the following script was used to proccess the raw data from the PyPSA model sumulations results for power plants units of the Dispa-SET_Unleash project.
    <br>
    Read explanation text cells to follow and understand all the process until final results were got stept by step.
</div>
<br>
<hr style="border: 2px solid skyblue;">
<div style="text-align: justify; margin-left: 3.0em; font-weight: bold; font-size: 18px; font-family: TimesNewRoman; color:skyblue">
    1. Notebook Set Up
</div>
<div style="text-align: justify; margin-left: 0.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
    Importing needed libraries.
<hr style="border: 1px solid skyblue;">
</div>
</div>

In [256]:
import os
import csv
from datetime import datetime
import requests
import pandas as pd
from shutil import move
import numpy as np
import shutil
from bs4 import BeautifulSoup
import re
import io
import plotly.graph_objects as go
from typing import List, Dict, Tuple
import re

<div style="background-color: black;">
<hr style="border: 0.5px solid skyblue;">
<div style="text-align: right; margin-left: 3.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman;color:skyblue">
    Auxiliar Code
</div>
    <div style="text-align: right; margin-left: 1.50em; font-weight: unbold; font-size: 13px; font-family: TimesNewRoman;color:skyblue">
    This cell has the purpose to create the correponding folders with the name of all the EU countries available in the ENTSOE data base.
    <br>
    Uncomment it to use it just if needed.
</div>
<hr style="border: 0.5px solid skyblue;">
</div>

In [257]:
"""
# List of countries with their acronyms in parentheses

countries = [
    "Albania (AL)", "Armenia (AM)", "Austria (AT)", "Azerbaijan (AZ)",
    "Belarus (BY)", "Belgium (BE)", "Bosnia and Herz. (BA)", "Bulgaria (BG)",
    "Croatia (HR)", "Cyprus (CY)", "Czech Republic (CZ)", "Denmark (DK)",
    "Estonia (EE)", "Finland (FI)", "France (FR)", "Georgia (GE)",
    "Germany (DE)", "Greece (EL)", "Hungary (HU)", "Iceland (IS)",
    "Ireland (IE)", "Italy (IT)", "Kosovo (XK)", "Latvia (LV)",
    "Lithuania (LT)", "Luxembourg (LU)", "Malta (MT)", "Moldova (MD)",
    "Montenegro (ME)", "Netherlands (NL)", "North Macedonia (MK)",
    "Norway (NO)", "Poland (PL)", "Portugal (PT)", "Romania (RO)",
    "Russia (RU)", "Russia Legacy (RU)", "Serbia (RS)", "Slovakia (SK)",
    "Slovenia (SI)", "Spain (ES)", "Sweden (SE)", "Switzerland (CH)",
    "Turkey (TR)", "Ukraine (UA)", "United Kingdom (UK)"
]

# Set the path where you want to create the folders
# Replace 'C:/Your/Path/Here' with your desired directory
base_path = '/home/ray/Dispa-SET_Unleash/Database_PyPSA/Suficiency_Scenario/PowerPlants'

# Ensure the base path exists
os.makedirs(base_path, exist_ok=True)

# Loop through the list of countries
for country_string in countries:
    # Use a regular expression to find the acronym inside the parentheses
    match = re.search(r'(.∗?)(.*?)', country_string)
    
    # If a match is found, extract the acronym
    if match:
        acronym = match.group(1)
        folder_path = os.path.join(base_path, acronym)
        
        # Check if the folder already exists to avoid errors
        if not os.path.exists(folder_path):
            os.makedirs(folder_path)
            print(f"Created folder: {folder_path}")
        else:
            print(f"Folder already exists: {folder_path}")

print("\nAll folders created successfully!")
"""

'\n# List of countries with their acronyms in parentheses\n\ncountries = [\n    "Albania (AL)", "Armenia (AM)", "Austria (AT)", "Azerbaijan (AZ)",\n    "Belarus (BY)", "Belgium (BE)", "Bosnia and Herz. (BA)", "Bulgaria (BG)",\n    "Croatia (HR)", "Cyprus (CY)", "Czech Republic (CZ)", "Denmark (DK)",\n    "Estonia (EE)", "Finland (FI)", "France (FR)", "Georgia (GE)",\n    "Germany (DE)", "Greece (EL)", "Hungary (HU)", "Iceland (IS)",\n    "Ireland (IE)", "Italy (IT)", "Kosovo (XK)", "Latvia (LV)",\n    "Lithuania (LT)", "Luxembourg (LU)", "Malta (MT)", "Moldova (MD)",\n    "Montenegro (ME)", "Netherlands (NL)", "North Macedonia (MK)",\n    "Norway (NO)", "Poland (PL)", "Portugal (PT)", "Romania (RO)",\n    "Russia (RU)", "Russia Legacy (RU)", "Serbia (RS)", "Slovakia (SK)",\n    "Slovenia (SI)", "Spain (ES)", "Sweden (SE)", "Switzerland (CH)",\n    "Turkey (TR)", "Ukraine (UA)", "United Kingdom (UK)"\n]\n\n# Set the path where you want to create the folders\n# Replace \'C:/Your/Path/Her

<div style="background-color: black;">
<hr style="border: 2px solid skyblue;">
<div style="text-align: justify; margin-left: 3.0em; font-weight: bold; font-size: 18px; font-family: TimesNewRoman; color:skyblue">
    2. Dispa-SET_Unleash Folder Path
</div>
<div style="text-align: justify; margin-left: 0.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
    Determinning dynamically the zone_folder_path based on the location of the "Dispa-SET_Unleash" folder relative to the current working directory. 
<br>
    If the "Dispa-SET_Unleash" folder is copied to a different machine or location, the dispaSET_unleash_folder_path variable will automatically adjust accordingly.
</div>
<hr style="border: 1px solid skyblue;">
</div>

In [258]:
# Get the current working directory
current_directory = os.getcwd()

# Navigate to the parent directory of "Dispa-SET_Unleash"
dispaSET_unleash_parent_directory = os.path.dirname(current_directory)

# Get the path to the "Dispa-SET_Unleash" folder
dispaSET_unleash_folder_path = os.path.dirname(dispaSET_unleash_parent_directory)

# Construct the dispaSET_unleash_folder_name variable
dispaSET_unleash_folder_name = os.path.basename(dispaSET_unleash_folder_path)

print("dispaSET_unleash_folder_name:", dispaSET_unleash_folder_name)
print("dispaSET_unleash_folder_path:", dispaSET_unleash_folder_path)




# Get the path to the "Dispa-SET_Unleash_Power_Plants" folder of the data used as base
additional_path_1 = "/Database/PowerPlants"
power_plants_base_data_folder_path = dispaSET_unleash_folder_path + additional_path_1

# Construct the dispaSET_unleash_power_plants_folder_name variable of the data used as reference
power_plants_base_data_folder_name = os.path.basename(power_plants_base_data_folder_path)

print("power_plants_base_data_folder_name:", power_plants_base_data_folder_name)
print("power_plants_base_data_folder_path:", power_plants_base_data_folder_path)




# Get the path to the "Dispa-SET_Unleash_Power_Plants" folder of the PyPSA raw data
additional_path_2 = "/RawData_PyPSA/Reference_Scenario/PowerPlants"
#additional_path_2 = "/RawData/Suficiency_Scenario/PowerPlants"
power_plants_pypsa_raw_data_folder_path = dispaSET_unleash_folder_path + additional_path_2

# Construct the dispaSET_unleash_power_plants_folder_name variable of the PyPSA raw data
power_plants_pypsa_raw_data_folder_name = os.path.basename(power_plants_pypsa_raw_data_folder_path)

print("power_plants_pypsa_raw_data_folder_name:", power_plants_pypsa_raw_data_folder_name)
print("power_plants_pypsa_raw_data_folder_path:", power_plants_pypsa_raw_data_folder_path)




# Get the path to the "Dispa-SET_Unleash_Power_Plants" folder of the PyPSA formated data
additional_path_3 = "/Database_PyPSA/Reference_Scenario/PowerPlants"
#additional_path_3 = "/Database_PyPSA/Suficiency_Scenario/PowerPlants"

power_plants_pypsa_formated_data_folder_path = dispaSET_unleash_folder_path + additional_path_3

# Construct the dispaSET_unleash_power_plants_folder_name variable of the PyPSA formated data
power_plants_pypsa_formated_data_folder_name = os.path.basename(power_plants_pypsa_formated_data_folder_path)

print("power_plants_pypsa_formated_data_folder_name:", power_plants_pypsa_formated_data_folder_name)
print("power_plants_pypsa_formated_data_folder_path:", power_plants_pypsa_formated_data_folder_path)

dispaSET_unleash_folder_name: Dispa-SET_Unleash
dispaSET_unleash_folder_path: /home/ray/Dispa-SET_Unleash
power_plants_base_data_folder_name: PowerPlants
power_plants_base_data_folder_path: /home/ray/Dispa-SET_Unleash/Database/PowerPlants
power_plants_pypsa_raw_data_folder_name: PowerPlants
power_plants_pypsa_raw_data_folder_path: /home/ray/Dispa-SET_Unleash/RawData_PyPSA/Reference_Scenario/PowerPlants
power_plants_pypsa_formated_data_folder_name: PowerPlants
power_plants_pypsa_formated_data_folder_path: /home/ray/Dispa-SET_Unleash/Database_PyPSA/Reference_Scenario/PowerPlants


<div style="background-color: black;">
<hr style="border: 2px solid skyblue;">
<div style="text-align: justify; margin-left: 3.0em; font-weight: bold; font-size: 18px; font-family: TimesNewRoman; color:skyblue">
    3. Zone(s) Creation
</div>
<div style="text-align: justify; margin-left: 0.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
    Entering the zone name or names (comment those ones that are not available data) where all data related to the corresponding zone are going to be storage
<br>
For European country names use the ISO 3166-1 standars i.e. AT, BE, BG, CH.... etc. to give the zone_name.
</div>
<hr style="border: 1px solid skyblue;">

In [259]:
# List of folder names to be addressed
zone_names = [
                #"AL",
                #"AM",
                #"AT",
                #"AZ",
                #"BY",
                "BE",
                #"BA",
                #"BG",
                #"HR",
                #"CY",
                #"CZ",
                #"DK",
                #"EE",
                #"FI",
                "FR",
                #"GE",
                "DE",
                #"EL",
                #"HU",
                #"IS",
                #"IE",
                #"IT",
                #"XK",
                #"LV",
                #"LT",
                #"LU",
                #"MT",
                #"MD",
                #"ME",
                "NL",
                #"MK",
                #"NO",
                #"PL",
                #"PT",
                #"RO",
                #"RU",
                #"RS",
                #"SK",
                #"SI",
                #"ES",
                #"SE",
                #"CH",
                #"TR",
                #"UA",
                "UK"
             ]

print("zone_names:", zone_names)

zone_names: ['BE', 'FR', 'DE', 'NL', 'UK']


<div style="background-color: black;">
<hr style="border: 2px solid skyblue;">
<div style="text-align: justify; margin-left: 3.0em; font-weight: bold; font-size: 18px; font-family: TimesNewRoman; color:skyblue">
    4. Data Reference Year 
</div>
<div style="text-align: justify; margin-left: 0.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
    Setting the variable on the target year which formatting data is wanted to
</div>
<hr style="border: 1px solid skyblue;">

In [260]:
# Year to which data is formating to:
data_target_year = '2030'

<div style="background-color: black;">
<hr style="border: 2px solid skyblue;">
<div style="text-align: justify; margin-left: 3.0em; font-weight: bold; font-size: 18px; font-family: TimesNewRoman; color:skyblue">
    5. Power Plant Data Frame
</div>
<div style="text-align: justify; margin-left: 0.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
    Creating the data frame with all the corresponding headers according the Dispa-SET nomenclature.
</div>
<hr style="border: 1px solid skyblue;">

In [261]:
# convert data year to integer
data_target_year = int(data_target_year)  

# Dictionary to store created DataFrames
power_plants_dfs_dict = {}

for zone in zone_names:
    zone_folder = os.path.join(power_plants_base_data_folder_path, zone)
    if not os.path.isdir(zone_folder):
        continue  # skip if zone folder doesn't exist

    # List all csv files in the folder
    csv_files = [f for f in os.listdir(zone_folder) if f.endswith(".csv")]

    # Extract years from filenames (must be digits only)
    available_years = []
    for f in csv_files:
        name, ext = os.path.splitext(f)
        if name.isdigit():  # e.g., "2004"
            available_years.append(int(name))

    if not available_years:
        continue  # skip if no year-based CSVs exist

    # Find closest year to target
    closest_year = min(available_years, key=lambda y: abs(y - data_target_year))

    # Path to the chosen file
    chosen_file = os.path.join(zone_folder, f"{closest_year}.csv")

    # Read only the header
    headers = pd.read_csv(chosen_file, nrows=0).columns.tolist()

    # Create an empty DataFrame with those headers
    df_name = f"{zone}_{data_target_year}"
    power_plants_dfs_dict[df_name] = pd.DataFrame(columns=headers)

    # Optionally, assign to globals() if you want actual variables:
    globals()[df_name] = power_plants_dfs_dict[df_name]

    print(f"Zone {zone}: picked {closest_year}.csv for target {data_target_year}")

print("Power Plant DataFrames:", list(power_plants_dfs_dict.keys()))

Zone BE: picked 2024.csv for target 2030
Zone FR: picked 2024.csv for target 2030
Zone DE: picked 2024.csv for target 2030
Zone NL: picked 2024.csv for target 2030
Zone UK: picked 2024.csv for target 2030
Power Plant DataFrames: ['BE_2030', 'FR_2030', 'DE_2030', 'NL_2030', 'UK_2030']


<div style="background-color: black;">
<hr style="border: 0.5px solid skyblue;">
<div style="text-align: right; margin-left: 3.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman;color:skyblue">
    Tracking Variables.
</div>
    <div style="text-align: right; margin-left: 1.50em; font-weight: unbold; font-size: 13px; font-family: TimesNewRoman;color:skyblue">
    This cells are just to confirm all the file names, file paths and other information related to the data being processed.
    <br>
  Also are used to ensure the inputs for next cells in order to avoid to re-enter the same information each time.
</div>
<hr style="border: 0.5px solid skyblue;">
</div>

In [262]:
print (f"Name of the DispaSET Unleash folder:                       {dispaSET_unleash_folder_name}\n")
print (f"Path to the DispaSET Unleash folder:                       {dispaSET_unleash_folder_path}\n")
print (f"Name of the Power Plants Base data folder:                 {power_plants_base_data_folder_name}\n")
print (f"Path to the Power Plants Base data folder:                 {power_plants_base_data_folder_path}\n")
print (f"Name of Power Plants_Pypsa Raw data folder:                {power_plants_pypsa_raw_data_folder_name}\n")
print (f"Path to the Power Plants_Pypsa Raw data folder:            {power_plants_pypsa_raw_data_folder_path}\n")
print (f"Name of the Power Plants_Pypsa Formated data folder:       {power_plants_pypsa_formated_data_folder_name}\n")
print (f"Path to the Power Plants_Pypsa Formated data folder:       {power_plants_pypsa_formated_data_folder_path}\n")
print (f"Name of the zones:                                         {zone_names}")
print (f"Target year:                                               {data_target_year}")
print (f"Name of the Power Plant DataFrames (dictionary):           {list(power_plants_dfs_dict.keys())}")

Name of the DispaSET Unleash folder:                       Dispa-SET_Unleash

Path to the DispaSET Unleash folder:                       /home/ray/Dispa-SET_Unleash

Name of the Power Plants Base data folder:                 PowerPlants

Path to the Power Plants Base data folder:                 /home/ray/Dispa-SET_Unleash/Database/PowerPlants

Name of Power Plants_Pypsa Raw data folder:                PowerPlants

Path to the Power Plants_Pypsa Raw data folder:            /home/ray/Dispa-SET_Unleash/RawData_PyPSA/Reference_Scenario/PowerPlants

Name of the Power Plants_Pypsa Formated data folder:       PowerPlants

Path to the Power Plants_Pypsa Formated data folder:       /home/ray/Dispa-SET_Unleash/Database_PyPSA/Reference_Scenario/PowerPlants

Name of the zones:                                         ['BE', 'FR', 'DE', 'NL', 'UK']
Target year:                                               2030
Name of the Power Plant DataFrames (dictionary):           ['BE_2030', 'FR_2030', 'DE_20

<div style="background-color: black;">
<hr style="border: 2px solid skyblue;">
<div style="text-align: justify; margin-left: 3.0em; font-weight: bold; font-size: 18px; font-family: TimesNewRoman; color:skyblue">
6. Nomenclature Technology Dictionary creation
</div>
<div style="text-align: justify; margin-left: 0.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
This is the equivalence table between PyPSA and Dispa-SET technologies nomenclature
</div>
<hr style="border: 2px dashed skyblue;">
<div style="text-align: justify; margin-left: 3.0em; font-weight: bold; font-size: 17px; font-family: TimesNewRoman; color:skyblue">
6.1. Dispa-SET Technologies Nomenclature
</div>
<div style="text-align: justify; margin-left: 0.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
The dispa-SET technologies nomenclature are loaded from "<a href="https://github.com/energy-modelling-toolkit/Dispa-SET/blob/master/dispaset/common.py" style="color:skyblue">https://github.com/energy-modelling-toolkit/Dispa-SET/blob/master/dispaset/common.py</a>"
</div>
<hr style="border: 0.5px solid skyblue;">
</div> 

In [263]:
# Define all the Dispa-SET technology lists from the common.py script of Dispa-SET core scripts
tech_master_list =         []

tech_renewables =          ['HROR' , 'PHOT' , 'WAVE' , 'WTOF' , 'WTON' , 'SOTH']

tech_conventional =        ['HDAM' , 'COMC' , 'GTUR' , 'STUR' , 'BATS' , 'ICEN']

tech_batteries =           ['BATS']

tech_storage =             ['BATS' , 'HDAM' , 'HPHS' , 'BEVS' , 'CAES' , 'SCSP' , 'H2ST' , 'HPHSC', 'THMS']

tech_p2bs =                ['P2GS' , 'ALKE' , 'PEME' , 'SOXE' , 'P2BS' , 'PEFC' , 'DMFC' , 'ALFC' , 'PAFC' , 'MCFC' , 'SOFC' ,
                            'REFC' , 'HDAMC', 'HRORC', 'HDLZ' , 'COMCX', 'GTURX', 'ICENX', 'STURX', 'P2HT' , 'ASHP' , 'GSHP' , 
                            'HYHP' , 'WSHP' , 'REHE']

tech_bs2p =                ['BSPG']

tech_boundary_sector =     ['BSPG' , 'GETH' , 'HOBO' , 'SOTH' , 'ABHP' , 'HOBOX', 'P2BS' , 'HBBS' , 'WHEN']

# Combine all lists into a single set to get unique values
all_technologies_set = set(tech_master_list + tech_renewables + tech_conventional +
                           tech_batteries + tech_storage + tech_p2bs + tech_bs2p +
                           tech_boundary_sector)

# Convert the set back to a sorted list
all_technologies_list = sorted(list(all_technologies_set))

# Create a DataFrame with the single column
dispaSET_tech_list = pd.DataFrame(all_technologies_list, columns=['Dispa-SET Technologies'])

# Print the final DataFrame
dispaSET_tech_list

,Dispa-SET Technologies
0,ABHP
1,ALFC
2,ALKE
3,ASHP
4,BATS
5,BEVS
6,BSPG
7,CAES
8,COMC
9,COMCX


<div style="background-color: black;">
<hr style="border: 0.5px solid skyblue;">
<div style="text-align: justify; margin-left: 3.0em; font-weight: bold; font-size: 17px; font-family: TimesNewRoman; color:skyblue">
6.2. Dispa-SET Fuels Nomenclature
</div>
<div style="text-align: justify; margin-left: 0.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
Additionally the dispa-SET fuelss nomenclature are loaded from "<a href="https://github.com/energy-modelling-toolkit/Dispa-SET/blob/master/dispaset/common.py" style="color:skyblue">https://github.com/energy-modelling-toolkit/Dispa-SET/blob/master/dispaset/common.py</a>"
</div>
<hr style="border: 0.5px solid skyblue;">
</div>

In [264]:
# Define all the Dispa-SET fuel lists from the common.py script of Dispa-SET core scripts
dispaSET_fuel_list =      [ 'AIR', 'AMO', 'BIO', 'GAS', 'HRD', 'LIG', 'NUC', 'OIL', 'PEA', 'SUN', 
                            'WAT', 'WIN', 'WST', 'OTH', 'GEO', 'HYD', 'WHT', 'ELE', 'THE', 'UNK'  ]

# Create a DataFrame with the single column
dispaSET_fuel_list = pd.DataFrame(dispaSET_fuel_list, columns=['Dispa-SET Fuels'])

# Print the final DataFrame
dispaSET_fuel_list

,Dispa-SET Fuels
0,AIR
1,AMO
2,BIO
3,GAS
4,HRD
5,LIG
6,NUC
7,OIL
8,PEA
9,SUN


<div style="background-color: black;">
<hr style="border: 2px solid skyblue;">
<div style="text-align: justify; margin-left: 3.0em; font-weight: bold; font-size: 17px; font-family: 'Times New Roman', serif; color: skyblue;">
6.2. PyPSA vs Dispaset Nomenclature
</div>
<div style="text-align: justify; margin-left: 0.0em; font-weight: normal; font-size: 14px; font-family: 'Times New Roman', serif; color: skyblue;">
All those technologies from PyPSA which can be represented in Dispa-SET have to be identified.
</div>
<hr style="border: 2.0px dashed skyblue;">
<div style="text-align: justify; margin-left: 3.0em; font-weight: bold; font-size: 16px; font-family: 'Times New Roman', serif; color: skyblue;">
6.2.1. PyPSA Energy System Flow Diagram
</div>
<div style="text-align: justify; margin-left: 0.0em; font-weight: normal; font-size: 14px; font-family: 'Times New Roman', serif; color: skyblue;">
The next chart represents graphically how all the Energy sector inside PyPSA is structured.<br>
This is used to get the equivalent diagram for Dispaset.
</div>
<div style="text-align: center; margin: 20px 0;">
  <img src="Images/PyPSA_multisector_figure_1.png" 
       alt="PyPSA Multisector Flow Diagram" 
       style="max-width:35%; height: auto;">
</div>
<div style="text-align: center; margin-top: 10px; font-size: 12px; font-family: 'Times New Roman', serif; color: skyblue;">
  Source: <a href="https://pypsa-eur.readthedocs.io/en/latest/" target="_blank" style="color: skyblue; text-decoration: underline;">PyPSA-Eur Documentation</a>
</div>
<hr style="border: 2.0px dashed skyblue;">
<div style="text-align: justify; margin-left: 3.0em; font-weight: bold; font-size: 16px; font-family: 'Times New Roman', serif; color: skyblue;">
6.2.2. Equivalent Dispa-SET Energy System Flow Diagram
</div>
<div style="text-align: justify; margin-left: 0.0em; font-weight: normal; font-size: 14px; font-family: 'Times New Roman', serif; color:skyblue">
A correlation has been established between the PyPSA parameters and their corresponding Dispa-SET equivalents:
</div>
<div style="text-align: center; margin: 20px 0;">
  <img src="Images/PyPSA_sectors_as_Dispaset_Flow_work_1.svg" 
       alt="Equivalent Dispa-SET Energy System Flow Diagram" 
       style="max-width: auto; height: auto;">
</div>
<div style="text-align: center; margin-top: 10px; font-size: 12px; font-family: 'Times New Roman', serif; color: skyblue;">
  Source: Adapted for Dispa-SET_Unleash
</div>
<div style="text-align: justify; margin-left: 0.0em; font-weight: normal; font-size: 14px; font-family: 'Times New Roman', serif; color:skyblue">
Due to feature limitations, not all elements from PyPSA can be represented in Dispa-SET.
<br>
However, for those compatible technologies, the following chart graphically illustrates how they are connected within the Dispa-SET environment logic:
</div>
<div style="text-align: center; margin: 20px 0;">
  <img src="Images/PyPSA_sectors_as_Dispaset_Flow_work_Filtered_1.svg" 
       alt="Equivalent Dispa-SET Energy System Flow Diagram" 
       style="max-width: auto; height: auto;">
</div>
<div style="text-align: center; margin-top: 10px; font-size: 12px; font-family: 'Times New Roman', serif; color: skyblue;">
  Source: Adapted for Dispa-SET_Unleash
</div>
<hr style="border: 2.0px dashed skyblue;">
<div style="text-align: justify; margin-left: 3.0em; font-weight: bold; font-size: 17px; font-family: 'Times New Roman', serif; color: skyblue;">
6.2.3. Technologies & Demmands Nomenclature - PyPSA vs Dispaset 
</div>
<div style="text-align: justify; margin-left: 0.0em; font-weight: normal; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
The PyPSA technologies and demmands nomenclature and their correlation with their homologous from Dispa-SET are described as follows:
</div>
<table style="width: 95%; margin-left: auto; margin-right: auto; border-collapse: collapse; font-family: TimesNewRoman; font-size: 12px; color: skyblue;">
  <thead>
    <tr style="background-color: #1E1E1E; color: skyblue; border-bottom: 1px solid skyblue;">
      <th style="width: 8%; padding: 8px; text-align: left; border: 1px solid #444;">PyPSA Element</th>
      <th style="width: 15%; padding: 8px; text-align: left; border: 1px solid #444;">Technology</th>
      <th style="width: 10%; padding: 8px; text-align: left; border: 1px solid #444;">Sector Classification</th>
      <th style="width: 34%; padding: 8px; text-align: left; border: 1px solid #444;">Description</th>
      <th style="width: 5%; padding: 8px; text-align: left; border: 1px solid #444;">Sector Relation</th>
      <th style="width: 15%; padding: 8px; text-align: left; border: 1px solid #444;">Dispa-SET Element</th>
      <th style="width: 5%; padding: 8px; text-align: left; border: 1px solid #444;">Element Type</th>
      <th style="width: 5%; padding: 8px; text-align: left; border: 1px solid #444;">May Modeled?</th>
    </tr>
  </thead>
  <tbody>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">DC</td>
      <td style="padding: 8px; border: 1px solid #444;">Power Sector</td>
      <td style="padding: 8px; border: 1px solid #444;">Represents the DC (HVDC) transmission network for electricity</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">NTC</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">OCGT</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Open-Cycle Gas Turbine producing electricity from gas.</td>
      <td style="padding: 8px; border: 1px solid #444;">X2P2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">CCGT</td>
      <td style="padding: 8px; border: 1px solid #444;">Power Sector</td>
      <td style="padding: 8px; border: 1px solid #444;">Combined-Cycle Gas Turbine</td>
      <td style="padding: 8px; border: 1px solid #444;">X2P2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">EV charger</td>
      <td style="padding: 8px; border: 1px solid #444;">Power Sector</td>
      <td style="padding: 8px; border: 1px solid #444;">Interface between the grid and electric vehicles</td>
      <td style="padding: 8px; border: 1px solid #444;">P&S</td>
      <td style="padding: 8px; border: 1px solid #444;">STOMaxChargingPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">V2G</td>
      <td style="padding: 8px; border: 1px solid #444;">Power Sector</td>
      <td style="padding: 8px; border: 1px solid #444;">Vehicle-to-Grid---allows EVs to discharge electricity</td>
      <td style="padding: 8px; border: 1px solid #444;">P&S</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">battery charger</td>
      <td style="padding: 8px; border: 1px solid #444;">Power Sector</td>
      <td style="padding: 8px; border: 1px solid #444;">Converts electricity to stored energy in batteries (charging link)</td>
      <td style="padding: 8px; border: 1px solid #444;">P&S</td>
      <td style="padding: 8px; border: 1px solid #444;">STOMaxChargingPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">BioSNG</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Produces synthetic natural gas (bio-methane)---fuel synthesis process</td>
      <td style="padding: 8px; border: 1px solid #444;">X2X2X</td>
      <td style="padding: 8px; border: 1px solid #444;">STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & N</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">DAC</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Direct Air Capture---captures CO<sub>2</sub> for storage or utilization</td>
      <td style="padding: 8px; border: 1px solid #444;">P2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">Fischer-Tropsch</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Converts H<sub>2</sub> + CO<sub>2</sub> to liquid hydrocarbons; fuel synthesis</td>
      <td style="padding: 8px; border: 1px solid #444;">X2X2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & N</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">H<sub>2</sub> Electrolysis</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Electricity into hydrogen cross-sector conversion</td>
      <td style="padding: 8px; border: 1px solid #444;">P2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">H<sub>2</sub> Fuel Cell</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Converts hydrogen back to electricity</td>
      <td style="padding: 8px; border: 1px solid #444;">X2P</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">H<sub>2</sub> pipeline</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Transports hydrogen between regions or sectors</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">Boundary Sector Interconnections</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">H<sub>2</sub> pipeline retrofitted</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Existing pipelines adapted for H<sub>2</sub> transport</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">Boundary Sector Interconnections</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">H<sub>2</sub> turbine</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Generates electricity or heat from hydrogen---boundary technology</td>
      <td style="padding: 8px; border: 1px solid #444;">X2P</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">Haber-Bosch</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Converts H<sub>2</sub> + N<sub>2</sub> into ammonia---chemical/fertilizer industry</td>
      <td style="padding: 8px; border: 1px solid #444;">PX2X2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">SMR</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Steam Methane Reforming---gas to hydrogen conversion</td>
      <td style="padding: 8px; border: 1px solid #444;">X2X2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & N</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">SMR CC</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">SMR with Carbon Capture---industrial hydrogen with CC</td>
      <td style="padding: 8px; border: 1px solid #444;">X2X2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & N</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">Sabatier</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">H<sub>2</sub> + CO<sub>2</sub> → CH<sub>4</sub>---synthetic methane production.</td>
      <td style="padding: 8px; border: 1px solid #444;">X2X2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & N</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">agriculture machinery oil</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Oil used in agricultural machinery---transport/fuel</td>
      <td style="padding: 8px; border: 1px solid #444;">X2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & N</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">ammonia cracker</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Converts ammonia back into hydrogen</td>
      <td style="padding: 8px; border: 1px solid #444;">P2X2X2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">battery discharger</td>
      <td style="padding: 8px; border: 1px solid #444;">Power Sector</td>
      <td style="padding: 8px; border: 1px solid #444;">Discharges stored electricity from batteries</td>
      <td style="padding: 8px; border: 1px solid #444;">P&S</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">electricity distribution grid</td>
      <td style="padding: 8px; border: 1px solid #444;">Power Sector</td>
      <td style="padding: 8px; border: 1px solid #444;">Represents distribution-level power flow---low voltage</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">N & N</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">coal</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Represents coal-based electricity generation</td>
      <td style="padding: 8px; border: 1px solid #444;">X2P2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">lignite</td>
      <td style="padding: 8px; border: 1px solid #444;">Power Sector</td>
      <td style="padding: 8px; border: 1px solid #444;">Represents coal-based electricity generation or conversion node</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">nuclear</td>
      <td style="padding: 8px; border: 1px solid #444;">Power Sector</td>
      <td style="padding: 8px; border: 1px solid #444;">Nuclear-to-electricity conversion within the power system.</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">oil</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Oil-fired electricity generation or conversion node</td>
      <td style="padding: 8px; border: 1px solid #444;">X2P2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">biogas to gas</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Upgrades raw biogas into pipeline-quality methane</td>
      <td style="padding: 8px; border: 1px solid #444;">X2X2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & N</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">biogas to gas CC</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Biogas upgrading with carbon capture---industrial fuel conversion</td>
      <td style="padding: 8px; border: 1px solid #444;">X2X2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & N</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">biomass to liquid</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Converts biomass into liquid fuels---synthetic fuel process.</td>
      <td style="padding: 8px; border: 1px solid #444;">P2X2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & N</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">CO<sub>2</sub> sequestered</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Represents captured Tons of CO<sub>2</sub> / hour transported or stored</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">OutputEmissions</td>
      <td style="padding: 8px; border: 1px solid #444;">output</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">coal for industry</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Use of coal as industrial feedstock/fuel</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">SectorXDemand</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">gas for industry</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Supplies natural gas to industrial demand</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">PowerXx2p</td>
      <td style="padding: 8px; border: 1px solid #444;">output</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">gas for industry CC</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Industrial gas use with carbon capture</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">PowerXx2p</td>
      <td style="padding: 8px; border: 1px solid #444;">output</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">gas pipeline</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Transports natural gas---energy carrier infrastructure</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">Boundary Sector Interconnections</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">gas pipeline new</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Expansion of natural gas transport capacity</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">Boundary Sector Interconnections</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">kerosene for aviation</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Aviation fuel consumption---transport sector</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">N & N</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">land transport oil</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Oil use for land transport---transport fuel consumption</td>
      <td style="padding: 8px; border: 1px solid #444;">X2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & N</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">methanolisation</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Synthesizes methanol (CO<sub>2</sub> + H<sub>2</sub> → CH<sub>3</sub>OH)</td>
      <td style="padding: 8px; border: 1px solid #444;">X2X2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & N</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">naphtha for industry</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Provides naphtha feedstock to industrial processes</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">N & N</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">process emissions</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Industrial process CO<sub>2</sub> emissions. Tons of CO<sub>2</sub> / hour</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">OutputEmissions</td>
      <td style="padding: 8px; border: 1px solid #444;">output</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">process emissions CC</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Industrial CO<sub>2</sub> emissions with capture — Tons of CO<sub>2</sub> / hour</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">OutputEmissions</td>
      <td style="padding: 8px; border: 1px solid #444;">output</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">residential rural air heat pump</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Converts electricity to heat for rural homes</td>
      <td style="padding: 8px; border: 1px solid #444;">P2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & N</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">residential rural biomass boiler</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Converts biomass to heat---residential fuel use</td>
      <td style="padding: 8px; border: 1px solid #444;">X2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">residential rural gas boiler</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Burns gas for heating---non-electric final energy use</td>
      <td style="padding: 8px; border: 1px solid #444;">X2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">residential rural oil boiler</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Burns oil for household heating---outside power generation</td>
      <td style="padding: 8px; border: 1px solid #444;">X2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">residential rural resistive heater</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Uses electricity directly for heating---part of demand side</td>
      <td style="padding: 8px; border: 1px solid #444;">P2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & N</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">residential rural water tanks charger</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Converts electricity to thermal energy in storage</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">OutputSectorXStorageInput</td>
      <td style="padding: 8px; border: 1px solid #444;">output</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">residential rural water tanks discharger</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Discharges stored heat — part of residential heating loop</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">OutputSectorXStorageInput</td>
      <td style="padding: 8px; border: 1px solid #444;">output</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">residential urban decentral air heat pump</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Decentralized air-source heat pump for urban buildings</td>
      <td style="padding: 8px; border: 1px solid #444;">P2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & N</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">residential urban decentral biomass boiler</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Biomass-to-heat conversion for urban residential areas</td>
      <td style="padding: 8px; border: 1px solid #444;">X2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">residential urban decentral gas boiler</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Urban gas boilers---distributed thermal devices</td>
      <td style="padding: 8px; border: 1px solid #444;">X2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">residential urban decentral oil boiler</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Decentralized oil boilers for urban homes</td>
      <td style="padding: 8px; border: 1px solid #444;">X2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">residential urban decentral resistive heater</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Electric heating devices---boundary heat sector</td>
      <td style="padding: 8px; border: 1px solid #444;">P2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & N</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">residential urban decentral water tanks charger</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Transfers electric energy to thermal storage (heat)</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">OutputSectorXStorageInput</td>
      <td style="padding: 8px; border: 1px solid #444;">output</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">residential urban decentral biomass boiler</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Biomass-to-heat conversion for urban residential areas</td>
      <td style="padding: 8px; border: 1px solid #444;">X2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">residential urban decentral gas boiler</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Urban gas boilers---distributed thermal devices</td>
      <td style="padding: 8px; border: 1px solid #444;">X2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">residential urban decentral oil boiler</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Decentralized oil boilers for urban homes</td>
      <td style="padding: 8px; border: 1px solid #444;">X2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">link & residential rural ground heat pump</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Electric heat using stable ground temperature</td>
      <td style="padding: 8px; border: 1px solid #444;">X2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & N</td>
    </tr>
     <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">residential urban decentral water tanks charger</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Transfers electric energy to thermal storage (heat)</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">OutputSectorXStorageInput</td>
      <td style="padding: 8px; border: 1px solid #444;">output</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">residential urban decentral water tanks discharger</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Discharges heat from thermal storage</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">OutputSectorXStorageInput</td>
      <td style="padding: 8px; border: 1px solid #444;">output</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">services rural air heat pump</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Service sector rural building heat pump</td>
      <td style="padding: 8px; border: 1px solid #444;">P2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & N</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">services rural biomass boiler</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Provides space or process heat for service buildings</td>
      <td style="padding: 8px; border: 1px solid #444;">X2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">services rural gas boiler</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Burns gas for heating---final energy use</td>
      <td style="padding: 8px; border: 1px solid #444;">X2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">services rural ground heat pump</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Electricity-to-heat for service buildings---heat sector</td>
      <td style="padding: 8px; border: 1px solid #444;">P2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & N</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">services rural oil boiler</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Oil heating in rural service buildings.</td>
      <td style="padding: 8px; border: 1px solid #444;">X2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">services rural resistive heater</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Electric heating for service buildings---final energy use</td>
      <td style="padding: 8px; border: 1px solid #444;">P2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & N</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">services rural water tanks charger</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Converts electricity to heat for service‐sector thermal storage</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">OutputSectorXStorageInput</td>
      <td style="padding: 8px; border: 1px solid #444;">output</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">services rural water tanks discharger</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Discharges stored heat to buildings---part of the heat sector</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">OutputSectorXStorageInput</td>
      <td style="padding: 8px; border: 1px solid #444;">output</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">services urban decentral air heat pump</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Local electric heat production for service buildings</td>
      <td style="padding: 8px; border: 1px solid #444;">P2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & N</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">services urban decentral biomass boiler</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Biomass‐to‐heat conversion---end-use heating</td>
      <td style="padding: 8px; border: 1px solid #444;">X2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">services urban decentral gas boiler</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Burns oil for household heating---outside power generation</td>
      <td style="padding: 8px; border: 1px solid #444;">X2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">services urban decentral oil boiler</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Oil heating---final energy use</td>
      <td style="padding: 8px; border: 1px solid #444;">X2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">services urban decentral resistive heater</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Direct electric heating for service buildings</td>
      <td style="padding: 8px; border: 1px solid #444;">P2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & N</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">services urban decentral water tanks charger</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Converts power to stored heat</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">OutputSectorXStorageInput</td>
      <td style="padding: 8px; border: 1px solid #444;">output</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">services urban decentral water tanks discharger</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Releases stored thermal energy</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">OutputSectorXStorageInput</td>
      <td style="padding: 8px; border: 1px solid #444;">output</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">shipping methanol</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Methanol use in maritime transport---fuel demand</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">PowerXx2p</td>
      <td style="padding: 8px; border: 1px solid #444;">output</td>
      <td style="padding: 8px; border: 1px solid #444;">N & N</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">shipping oil</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Oil consumption for ships---transport sector</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">PowerXx2p</td>
      <td style="padding: 8px; border: 1px solid #444;">output</td>
      <td style="padding: 8px; border: 1px solid #444;">N & N</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">solid biomass for Industry</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Biomass fuel use for industrial heat/processes</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">PowerXx2p</td>
      <td style="padding: 8px; border: 1px solid #444;">output</td>
      <td style="padding: 8px; border: 1px solid #444;">N & N</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">solid biomass for industry CC</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Same as above but with carbon capture</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">PowerXx2p</td>
      <td style="padding: 8px; border: 1px solid #444;">output</td>
      <td style="padding: 8px; border: 1px solid #444;">N & N</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">solid biomass transport</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Represents biomass logistics between regions/sectors</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">FlowXmaximum & FlowXminimum</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & N</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">urban central air heat pump</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Centralized district heat pump---heat sector interfac.</td>
      <td style="padding: 8px; border: 1px solid #444;">P2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & N</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">urban central gas CHP</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Combined heat + power supplying district heating</td>
      <td style="padding: 8px; border: 1px solid #444;">X2CHP</td>
      <td style="padding: 8px; border: 1px solid #444;">Power Plant Data</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">urban central gas CHP CC</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Gas CHP with carbon capture---district heating system</td>
      <td style="padding: 8px; border: 1px solid #444;">X2CHP</td>
      <td style="padding: 8px; border: 1px solid #444;">Power Plant Data</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">urban central gas boiler</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Centralized gas heating for urban networks</td>
      <td style="padding: 8px; border: 1px solid #444;">X2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">urban central resistive heater</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Electric boiler for district heating---end-use conversion</td>
      <td style="padding: 8px; border: 1px solid #444;">P2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & N</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">urban central solid biomass CHP</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Biomass combined heat + power---heat boundary process</td>
      <td style="padding: 8px; border: 1px solid #444;">X2CHP</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">urban central solid biomass CHP CC</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Same with carbon capture---boundary sector</td>
      <td style="padding: 8px; border: 1px solid #444;">X2CHP</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">urban central water tanks charger</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Converts electricity to heat in district storage</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">OutputSectorXStorageInput</td>
      <td style="padding: 8px; border: 1px solid #444;">output</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">urban central water tanks discharger</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Discharges stored heat to the district network</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">OutputSectorXStorageInput</td>
      <td style="padding: 8px; border: 1px solid #444;">output</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">generator</td>
      <td style="padding: 8px; border: 1px solid #444;">coal</td>
      <td style="padding: 8px; border: 1px solid #444;">Power Sector</td>
      <td style="padding: 8px; border: 1px solid #444;">Fossil fuel-based electricity generation</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">OutputPower</td>
      <td style="padding: 8px; border: 1px solid #444;">output</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">generator</td>
      <td style="padding: 8px; border: 1px solid #444;">gas</td>
      <td style="padding: 8px; border: 1px solid #444;">Power Sector</td>
      <td style="padding: 8px; border: 1px solid #444;">Natural gas–fired power generation</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">OutputPower</td>
      <td style="padding: 8px; border: 1px solid #444;">output</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">generator</td>
      <td style="padding: 8px; border: 1px solid #444;">lignite</td>
      <td style="padding: 8px; border: 1px solid #444;">Power Sector</td>
      <td style="padding: 8px; border: 1px solid #444;">Coal variant used for power generation</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">OutputPower</td>
      <td style="padding: 8px; border: 1px solid #444;">output</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
        <td style="padding: 8px; border: 1px solid #444;">generator</td>
      <td style="padding: 8px; border: 1px solid #444;">oil</td>
      <td style="padding: 8px; border: 1px solid #444;">Power Sector</td>
      <td style="padding: 8px; border: 1px solid #444;">Oil-fired electricity generation</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">Powerx2p</td>
      <td style="padding: 8px; border: 1px solid #444;">output</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">generator</td>
      <td style="padding: 8px; border: 1px solid #444;">onwind</td>
      <td style="padding: 8px; border: 1px solid #444;">Power Sector</td>
      <td style="padding: 8px; border: 1px solid #444;">Onshore wind turbine generation</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">generator</td>
      <td style="padding: 8px; border: 1px solid #444;">offwind</td>
      <td style="padding: 8px; border: 1px solid #444;">Power Sector</td>
      <td style="padding: 8px; border: 1px solid #444;">Offshore wind turbine generation</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">generator</td>
      <td style="padding: 8px; border: 1px solid #444;">offwind-ac</td>
      <td style="padding: 8px; border: 1px solid #444;">Power Sector</td>
      <td style="padding: 8px; border: 1px solid #444;">Offshore wind with AC connection</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">generator</td>
      <td style="padding: 8px; border: 1px solid #444;">offwind-dc</td>
      <td style="padding: 8px; border: 1px solid #444;">Power Sector</td>
      <td style="padding: 8px; border: 1px solid #444;">Offshore wind with DC connection</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">generator</td>
      <td style="padding: 8px; border: 1px solid #444;">solar</td>
      <td style="padding: 8px; border: 1px solid #444;">Power Sector</td>
      <td style="padding: 8px; border: 1px solid #444;">Utility-scale photovoltaic generation</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">generator</td>
      <td style="padding: 8px; border: 1px solid #444;">solar rooftop</td>
      <td style="padding: 8px; border: 1px solid #444;">Power Sector</td>
      <td style="padding: 8px; border: 1px solid #444;">Distributed PV connected to power grid</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">generator</td>
      <td style="padding: 8px; border: 1px solid #444;">ror</td>
      <td style="padding: 8px; border: 1px solid #444;">Power Sector</td>
      <td style="padding: 8px; border: 1px solid #444;">Run-of-river hydro power plant</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">generator</td>
      <td style="padding: 8px; border: 1px solid #444;">uranium</td>
      <td style="padding: 8px; border: 1px solid #444;">Power Sector</td>
      <td style="padding: 8px; border: 1px solid #444;">Nuclear fuel input for electricity generation</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">OutputPower</td>
      <td style="padding: 8px; border: 1px solid #444;">output</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">generator</td>
      <td style="padding: 8px; border: 1px solid #444;">residential rural solar thermal</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Produces heat for households (not electricity)</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">generator</td>
      <td style="padding: 8px; border: 1px solid #444;">residential urban decentral solar thermal</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Decentralized solar heating for buildings</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">generator</td>
      <td style="padding: 8px; border: 1px solid #444;">services rural solar thermal</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Solar thermal for service-sector heat demand</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">generator</td>
      <td style="padding: 8px; border: 1px solid #444;">services urban decentral solar thermal</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Urban service-sector solar heating</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">generator</td>
      <td style="padding: 8px; border: 1px solid #444;">urban central solar thermal</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Centralized solar thermal for district heating</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">generator</td>
      <td style="padding: 8px; border: 1px solid #444;">load</td>
      <td style="padding: 8px; border: 1px solid #444;">Power Sector</td>
      <td style="padding: 8px; border: 1px solid #444;">Represents total shredding energy</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">Load Shedding</td>
      <td style="padding: 8px; border: 1px solid #444;">output</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">storage_units</td>
      <td style="padding: 8px; border: 1px solid #444;">hydro</td>
      <td style="padding: 8px; border: 1px solid #444;">Power Sector</td>
      <td style="padding: 8px; border: 1px solid #444;">Conventional hydro reservoir</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">storage_units</td>
      <td style="padding: 8px; border: 1px solid #444;">PHS</td>
      <td style="padding: 8px; border: 1px solid #444;">Power Sector</td>
      <td style="padding: 8px; border: 1px solid #444;">Pumped Hydro Storage, a grid-scale electricity storage technology</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">stores</td>
      <td style="padding: 8px; border: 1px solid #444;">battery</td>
      <td style="padding: 8px; border: 1px solid #444;">Power Sector</td>
      <td style="padding: 8px; border: 1px solid #444;">Electrical energy storage — directly coupled with the grid.</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">STOCapacity</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
            <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">stores</td>
      <td style="padding: 8px; border: 1px solid #444;">uranium</td>
      <td style="padding: 8px; border: 1px solid #444;">Power Sector</td>
      <td style="padding: 8px; border: 1px solid #444;">Nuclear fuel stock for electricity generation</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">STOCapacity</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & N</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">stores</td>
      <td style="padding: 8px; border: 1px solid #444;">H<sub>2</sub></td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Hydrogen storage---chemical energy carrier</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">STOCapacity</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">stores</td>
      <td style="padding: 8px; border: 1px solid #444;">NH<sub>3</sub></td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Ammonia storage---chemical/fertilizer or fuel vector.</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">STOCapacity</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">stores</td>
      <td style="padding: 8px; border: 1px solid #444;">biogas</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Biomethane stock for heating or industry</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">STOCapacity</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & N</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">stores</td>
      <td style="padding: 8px; border: 1px solid #444;">CO<sub>2</sub></td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Captured CO<sub>2</sub> pool---used in synthesis or stored</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">OutputEmissions</td>
      <td style="padding: 8px; border: 1px solid #444;">output</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">stores</td>
      <td style="padding: 8px; border: 1px solid #444;">CO<sub>2</sub> sequestered</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Permanent CO<sub>2</sub> storage---Tons of CO<sub>2</sub></td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">OutputEmissions</td>
      <td style="padding: 8px; border: 1px solid #444;">output</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>      
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">stores</td>
      <td style="padding: 8px; border: 1px solid #444;">CO<sub>2</sub> stored</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Intermediate or final CO<sub>2</sub> reservoir---Tons of CO<sub>2</sub></td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">OutputEmissions</td>
      <td style="padding: 8px; border: 1px solid #444;">output</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">stores</td>
      <td style="padding: 8px; border: 1px solid #444;">coal</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Coal stock for industrial/fuel processes</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">STOCapacity</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">stores</td>
      <td style="padding: 8px; border: 1px solid #444;">gas</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Natural gas (CH<sub>4</sub>) stock — cross-sector energy carrier</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">STOCapacity</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & N</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">stores</td>
      <td style="padding: 8px; border: 1px solid #444;">lignite</td>
      <td style="padding: 8px; border: 1px solid #444;">Power Sector</td>
      <td style="padding: 8px; border: 1px solid #444;">Fuel storage for thermal use — outside grid operations</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">STOCapacity</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & N</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">stores</td>
      <td style="padding: 8px; border: 1px solid #444;">methanol</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Liquid fuel stock — used in transport or synthesis chains</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">STOCapacity</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & N</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">stores</td>
      <td style="padding: 8px; border: 1px solid #444;">oil</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Oil (synthetic hydrocarbons) stock for transport/industrial fuels</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">STOCapacity</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">stores</td>
      <td style="padding: 8px; border: 1px solid #444;">solid biomass</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Biomass stock for heating/industrial use</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">STOCapacity</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & N</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">stores</td>
      <td style="padding: 8px; border: 1px solid #444;">residential rural water tanks</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Thermal storage for rural households — heat sector</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">STOCapacity</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1评审44;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">stores</td>
      <td style="padding: 8px; border: 1px solid #444;">residential urban decentral water tanks</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Distributed heat storage in urban residences</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">STOCapacity</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">stores</td>
      <td style="padding: 8px; border: 1px solid #444;">services rural water tanks</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Heat storage for rural service buildings</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">STOCapacity</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">stores</td>
      <td style="padding: 8px; border: 1px solid #444;">services urban decentral water tanks</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Thermal storage for urban service buildings</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">STOCapacity</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">stores</td>
      <td style="padding: 8px; border: 1px solid #444;">urban central water tanks</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">District heating storage — boundary heat network</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">STOCapacity</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">oil</td>
      <td style="padding: 8px; border: 1px solid #444;">Oil for industry</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Final oil demand in the industrial sector</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">SectorXDemand</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">heat</td>
      <td style="padding: 8px; border: 1px solid #444;">Residential and tertiary DH demand</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">District heating demand for residential and service buildings</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">SectorXDemand</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">heat</td>
      <td style="padding: 8px; border: 1px solid #444;">Residential and tertiary heat demand</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Decentralized residential space/water heating demand</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">SectorXDemand</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">electricity</td>
      <td style="padding: 8px; border: 1px solid #444;">agriculture electricity</td>
      <td style="padding: 8px; border: 1px solid #444;">Power Sector</td>
      <td style="padding: 8px; border: 1px solid #444;">Electricity consumption in lighting, irrigation, machinery)</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">Demand</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">heat</td>
      <td style="padding: 8px; border: 1px solid #444;">agriculture heat</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Heat energy required in agricultural processes (drying, greenhouses)</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">SectorXDemand</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">oil</td>
      <td style="padding: 8px; border: 1px solid #444;">agriculture oil</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Oil consumption in agricultural machinery and vehicles</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">SectorXDemand</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">oil</td>
      <td style="padding: 8px; border: 1px solid #444;">aviation oil demand</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Jet fuel (kerosene) demand for aviation transport</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">SectorXDemand</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">electricity</td>
      <td style="padding: 8px; border: 1px solid #444;">electricity demand for rail network</td>
      <td style="padding: 8px; border: 1px solid #444;">Power Sector</td>
      <td style="padding: 8px; border: 1px solid #444;">Traction electricity used by rail and metro transport systems</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">Demand</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">electricity</td>
      <td style="padding: 8px; border: 1px solid #444;">electricity demand of residential and tertairy</td>
      <td style="padding: 8px; border: 1px solid #444;">Power Sector</td>
      <td style="padding: 8px; border: 1px solid #444;">Electricity consumption in households and service-sector buildings</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">Demand</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">electricity</td>
      <td style="padding: 8px; border: 1px solid #444;">electricity for Industry</td>
      <td style="padding: 8px; border: 1px solid #444;">Power Sector</td>
      <td style="padding: 8px; border: 1px solid #444;">Electricity use for machinery, processes, and electrified production</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">Demand</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">methane</td>
      <td style="padding: 8px; border: 1px solid #444;">gas for Industry</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Natural gas or synthetic methane Industrial demand</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">SectorXDemand</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">hydrogen</td>
      <td style="padding: 8px; border: 1px solid #444;">hydrogen for industry</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Hydrogen demand for industrial refining, ammonia, steelmaking</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">SectorXDemand</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">electricity</td>
      <td style="padding: 8px; border: 1px solid #444;">land transport EV</td>
      <td style="padding: 8px; border: 1px solid #444;">Power Sector</td>
      <td style="padding: 8px; border: 1px solid #444;">Electricity consumption by electric vehicles in road transport</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">Demand</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">hydrogen</td>
      <td style="padding: 8px; border: 1px solid #444;">land transport hydrogen demand</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Hydrogen fuel demand for road transport (fuel-cell vehicles)</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">SectorXDemand</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">heat</td>
      <td style="padding: 8px; border: 1px solid #444;">low-temperature heat for industry</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">(below $\sim 200^{\circ}$C), typically supplied by boilers or heat pumps</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">SectorXDemand</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">Non-energy</td>
      <td style="padding: 8px; border: 1px solid #444;">naphtha for non-energy</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Naphtha used as a chemical feedstock e.g., plastics, petrochemicals</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">N & N</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">oil</td>
      <td style="padding: 8px; border: 1px solid #444;">oil to transport demand</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Oil demand for conventional land gasoline and diesel vehicles</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">SectorXDemand</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">hydrogen</td>
      <td style="padding: 8px; border: 1px solid #444;">shipping hydrogen</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Hydrogen demand, maritime transport fuel-cell/ combustion ships</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">SectorXDemand</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">oil</td>
      <td style="padding: 8px; border: 1px solid #444;">shipping oil</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Conventional marine oil fuel demand (HFO, MGO) for ships</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">SectorXDemand</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">solid biomass</td>
      <td style="padding: 8px; border: 1px solid #444;">solid biomass for Industry</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Biomass demand in industrial processes for heat or material use</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">SectorXDemand</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
  </tbody>
</table>
<div style="text-align: center; margin-top: 10px; font-size: 12px; font-family: 'Times New Roman', serif; color: skyblue;">
  Source: Adapted for Dispa-SET_Unleash
</div>
<hr style="border: 0.5px solid skyblue;">
<div style="text-align: justify; margin-left: 3.0em; font-weight: bold; font-size: 17px; font-family: 'Times New Roman', serif; color: skyblue;">
6.3. Dispa-SET vs PyPSA Technologies Equivalences Dictionary
</div>
<div style="text-align: justify; margin-left: 0.0em; font-weight: normal; font-size: 14px; font-family: 'Times New Roman', serif; color: skyblue;">
A dictionary of Dispa-SET and PyPSA technology equivalences for power plant units is developed based on the specifications in the last table</div>
<div style="text-align: justify; margin-left:2em; font-weight: unbold; font-size: 12px; font-family: TimesNewRoman; color:skyblue">
    * Notes: &nbsp;&nbsp; keep values of the first column identical to the 'dispaSET_tech_list' list, since it depends of the Dipsa-SET core code.
    <br>
    &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;
    If the list in the <code>commons.py</code> script changes, the 'tech_equivalences' list has to be updated accordingly.
</div>
<hr style="border: 0.5px solid skyblue;">
</div>

In [265]:
# Dictionary mapping Dispa-SET acronyms to PyPSA tech names

tech_equivalences_dict = {
    
# -----------------------
# Renewable Power Units
# -----------------------
"generators offwind":                        {"tech": ["WTOF"] ,                                                           "fuel": ["WIN"]},
"generators offwind-ac":                     {"tech": ["WTOF"] ,                                                           "fuel": ["WIN"]},
"generators offwind-dc":                     {"tech": ["WTOF"] ,                                                           "fuel": ["WIN"]},
    
"generators onwind":                         {"tech": ["WTON"] ,                                                           "fuel": ["WIN"]},

"generators solar":                          {"tech": ["PHOT"] ,                                                           "fuel": ["SUN"]},
"generators solar rooftop":                  {"tech": ["PHOT"] ,                                                           "fuel": ["SUN"]},

"generators ror":                            {"tech": ["HROR"] ,                                                           "fuel": ["WAT"]},
# -----------------------
# Conventional Power Units
# -----------------------
"links Lignite":                             {"tech": ["STUR"] ,                                                           "fuel": ["LIG"]},
    
"links nuclear":                             {"tech": ["STUR"] ,                                                           "fuel": ["NUC"]},

"links CCGT":                                {"tech": ["COMC"] ,                                                           "fuel": ["GAS"  , "HYD"  , "BIO"]},
    
"links OCGT":                                {"tech": ["GTUR"] ,                                                           "fuel": ["GAS"  , "HYD"  , "OIL"  , "AMO" , "BIO" , "WST" , "OTH"]},

"links oil":                                 {"tech": ["GTUR"  , "ICEN", "STUR"] ,                                         "fuel": ["OIL"]},

"links coal":                                {"tech": ["STUR"  , "COMC", "GTUR"] ,                                         "fuel": ["HRD"  , "PEA"]},

# -----------------------
# Storage Units
# -----------------------
"links V2G":                                 {"tech": ["BEVS"] ,                                                           "fuel": ["ELE"]},
    
"links battery discharger":                  {"tech": ["BATS"] ,                                                           "fuel": ["ELE"]},
    
"storage_units hydro":                      {"tech": ["HDAM"] ,                                                           "fuel": ["WAT"]},
    
"storage_units PHS":                        {"tech": ["HPHS"] ,                                                           "fuel": ["WAT"]},
# -----------------------
# Combined Heat and Power Units
# -----------------------
"links urban central gas CHP":               {"tech": ["COMC"] ,                                                           "fuel": ["GAS"]},
"links urban central gas CHP CC":            {"tech": ["COMC"] ,                                                           "fuel": ["GAS"]},

"links urban central solid biomass CHP":     {"tech": ["STUR"] ,                                                           "fuel": ["BIO"]},
"links urban central solid biomass CHP CC":  {"tech": ["STUR"] ,                                                           "fuel": ["BIO"]},
# -----------------------
# Sector X Units
# -----------------------
"links H2 Electrolysis":                     {"tech": ["PEME"  , "ALKE"  , "SOXE"] ,                                       "fuel": ["HYD"]},
    
"links H2 Fuel Cell":                        {"tech": ["PEFC"  , "SOFC"  , "MCFC"  , "PAFC" , "ALFC" , "DMFC" , "REFC"] ,  "fuel": ["HYD"]},
"links H2 turbine":                          {"tech": ["GTUR"  , "COMC"  , "ICEN"] ,                                       "fuel": ["HYD"]},

"links Haber-Bosch":                         {"tech": ["HBBS"] ,                                                           "fuel": ["AMO"]},
    
"links DAC":                                 {"tech": ["P2BS"] ,                                                           "fuel": ["OTH"]},

"links methanolisation":                          {"tech": ["P2BS"] ,                                                           "fuel": ["OTH"]}
    
}

tech_equivalences_dict

{'generators offwind': {'tech': ['WTOF'], 'fuel': ['WIN']},
 'generators offwind-ac': {'tech': ['WTOF'], 'fuel': ['WIN']},
 'generators offwind-dc': {'tech': ['WTOF'], 'fuel': ['WIN']},
 'generators onwind': {'tech': ['WTON'], 'fuel': ['WIN']},
 'generators solar': {'tech': ['PHOT'], 'fuel': ['SUN']},
 'generators solar rooftop': {'tech': ['PHOT'], 'fuel': ['SUN']},
 'generators ror': {'tech': ['HROR'], 'fuel': ['WAT']},
 'links Lignite': {'tech': ['STUR'], 'fuel': ['LIG']},
 'links nuclear': {'tech': ['STUR'], 'fuel': ['NUC']},
 'links CCGT': {'tech': ['COMC'], 'fuel': ['GAS', 'HYD', 'BIO']},
 'links OCGT': {'tech': ['GTUR'],
  'fuel': ['GAS', 'HYD', 'OIL', 'AMO', 'BIO', 'WST', 'OTH']},
 'links oil': {'tech': ['GTUR', 'ICEN', 'STUR'], 'fuel': ['OIL']},
 'links coal': {'tech': ['STUR', 'COMC', 'GTUR'], 'fuel': ['HRD', 'PEA']},
 'links V2G': {'tech': ['BEVS'], 'fuel': ['ELE']},
 'links battery discharger': {'tech': ['BATS'], 'fuel': ['ELE']},
 'storage_units hydro': {'tech': ['HDAM'], '

<div style="background-color: black;">
<hr style="border: 0.5px solid skyblue;">
<div style="text-align: justify; margin-left: 3.0em; font-weight: bold; font-size: 17px; font-family: 'Times New Roman', serif; color: skyblue;">
6.4. Dispa-SET Technology - Fuel matching Table
</div>
<div style="text-align: justify; margin-left: 0.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
    In Dispa-SET, each unit must be defined with the pair of values (technology,fuel). The following data is an updated version—Fuel-type proportions table—of its homologous one from the documentation in:
</div>
<div style="text-align: justify; margin-left: 5em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
    <a href=https://www.dispaset.eu/en/latest/data.html style="color:skyblue">https://www.dispaset.eu/en/latest/data.html</a>
</div>
<div style="text-align: justify; margin-left: 3.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
    The items provided in the Dispa-SET core scripts—i.e., <code>commons.py</code>—and further refined using recently compiled and estimated data derived from publicly available sources. These sources include datasets and reports from:
</div>

<div style="text-align: justify; margin-left: 5em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
    <ol style="margin-top: 0; padding-left: 1.5em;">
        <li>
            The Dispa-SET documentation.
            <br>
            <a href="https://www.dispaset.eu/en/latest/data.html" style="color:skyblue">https://www.dispaset.eu/en/latest/data.html</a>
        </li>
        <li>
            The European Energy Storage Inventory by the Joint Research Centre (JRC) of the European Commission.
            <br>
            <a href="https://joint-research-centre.ec.europa.eu/jrc-news-and-updates/new-tool-maps-europes-real-time-sustainable-energy-storage-data-2025-03-20_en" style="color:skyblue">https://joint-research-centre.ec.europa.eu/jrc-news-and-updates/new-tool-maps-europes-real-time-sustainable-energy-storage-data-2025-03-20_en</a>
        </li>
        <li>
            The IEA Energy Statistics Data Browser.
            <br>
            <a href="https://www.iea.org/data-and-statistics/data-tools/energy-statistics-data-browser" style="color:skyblue">https://www.iea.org/data-and-statistics/data-tools/energy-statistics-data-browser</a>
        </li>
        <li>
            The EU Energy Statistical Pocketbook.
            <br>
            <a href="https://energy.ec.europa.eu/data-and-analysis/eu-energy-statistical-pocketbook-and-country-datasheets_en" style="color:skyblue">https://energy.ec.europa.eu/data-and-analysis/eu-energy-statistical-pocketbook-and-country-datasheets_en</a>
        </li>
        <li>
            The European Electricity Review by Ember.
            <br>
            <a href="https://www.ember-climate.org/publications/european-electricity-review-2025/" style="color:skyblue">https://www.ember-climate.org/publications/european-electricity-review-2025/</a>
        </li>
    </ol>
</div>
<div style="text-align: justify; margin-left: 3.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
The technology-to-fuel mappings were additionally informed by modeling frameworks and literature such as:
</div>
<div style="text-align: justify; margin-left: 5em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
    <ol style="margin-top: 0; padding-left: 1.5em;" start="6">
        <li>
            ETSAP-TIMES.
            <br>
            <a href="https://iea-etsap.org/index.php/etsap-tools/model-generator/times" style="color:skyblue">https://iea-etsap.org/index.php/etsap-tools/model-generator/times</a>
        </li>
        <li>
            Dispa-SET technical documentation.
            <br>
            <a href="https://www.dispaset.eu/en/latest/" style="color:skyblue">https://www.dispaset.eu/en/latest/</a>
        </li>
    </ol>
</div>

<div style="text-align: justify; margin-left: 3.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
    And the following sources:
</div>

<div style="text-align: justify; margin-left: 5em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
    <ol style="margin-top: 0; padding-left: 1.5em;" start="8">
        <li>
            IEA Hydrogen Roadmap (2023).
            <br>
            <a href="https://www.iea.org/reports/technology-roadmap-hydrogen-and-fuel-cells" style="color:skyblue">https://www.iea.org/reports/technology-roadmap-hydrogen-and-fuel-cells</a>
        </li>
        <li>
            IRENA Electrolyzers (2023).
            <br>
            <a href="https://www.irena.org/publications" style="color:skyblue">https://www.irena.org/publications</a>
        </li>
        <li>
            US DOE Electrolysis (2023).
            <br>
            <a href="https://www.energy.gov/eere/fuelcells/hydrogen-production-electrolysis" style="color:skyblue">https://www.energy.gov/eere/fuelcells/hydrogen-production-electrolysis</a>
        </li>
        <li>
            EC Electrolyzers (2023).
            <br>
            <a href="https://energy.ec.europa.eu/topics/eus-energy-system/hydrogen_en" style="color:skyblue">https://energy.ec.europa.eu/topics/eus-energy-system/hydrogen_en</a>
        </li>
        <li>
            US DOE Air-Source Heat Pumps (2024).
            <br>
            <a href="https://www.energy.gov/energysaver/air-source-heat-pumps" style="color:skyblue">https://www.energy.gov/energysaver/air-source-heat-pumps</a>
        </li>
        <li>
            Minnesota Air Source Heat Pump Report (2011).
            <br>
            <a href="https://www.leg.mn.gov/docs/2014/other/141021.pdf" style="color:skyblue">https://www.leg.mn.gov/docs/2014/other/141021.pdf</a>
        </li>
        <li>
            North NJ HVAC (2025).
            <br>
            <a href="https://northnjhvac.com/common-sources-thermal-energy-heat-pumps-explained/" style="color:skyblue">https://northnjhvac.com/common-sources-thermal-energy-heat-pumps-explained/</a>
        </li>
        <li>
            US DOE Battery Storage (2024).
            <br>
            <a href="https://www.energy.gov/sites/default/files/2025-01/BESSIE_supply-chain-battery-report_111124_OPENRELEASE_SJ_1.pdf" style="color:skyblue">https://www.energy.gov/sites/default/files/2025-01/BESSIE_supply-chain-battery-report_111124_OPENRELEASE_SJ_1.pdf</a>
        </li>
        <li>
            EIA Battery Storage (2024).
            <br>
            <a href="https://www.eia.gov/todayinenergy/detail.php?id=63025" style="color:skyblue">https://www.eia.gov/todayinenergy/detail.php?id=63025</a>
        </li>
        <li>
            IEA Batteries (2024).
            <br>
            <a href="https://www.iea.org/reports/batteries-and-secure-energy-transitions/executive-summary" style="color:skyblue">https://www.iea.org/reports/batteries-and-secure-energy-transitions/executive-summary</a>
        </li>
        <li>
            Vehicle-to-Grid (2025).
            <br>
            <a href="https://www.mdpi.com/2032-6653/16/3/142" style="color:skyblue">https://www.mdpi.com/2032-6653/16/3/142</a>
        </li>
        <li>
            EIA Transportation Energy (2024).
            <br>
            <a href="https://www.eia.gov/energyexplained/use-of-energy/transportation-in-depth.php" style="color:skyblue">https://www.eia.gov/energyexplained/use-of-energy/transportation-in-depth.php</a>
        </li>
        <li>
            Solar PV Systems (2023).
            <br>
            <a href="https://www.frontiersin.org/articles/10.3389/fenrg.2023.1164494/full" style="color:skyblue">https://www.frontiersin.org/articles/10.3389/fenrg.2023.1164494/full</a>
        </li>
        <li>
            US DOE Solar Opportunities (2024).
            <br>
            <a href="https://www.energy.gov/eere/solar/articles/expanding-solar-energy-opportunities-rooftops-building-integration" style="color:skyblue">https://www.energy.gov/eere/solar/articles/expanding-solar-energy-opportunities-rooftops-building-integration</a>
        </li>
        <li>
            US DOE CAES (2023).
            <br>
            <a href="https://www.energy.gov/sites/default/files/2023-07/Technology%20Strategy%20Assessment%20-%20Compressed%20Air%20Energy%20Storage_0.pdf" style="color:skyblue">https://www.energy.gov/sites/default/files/2023-07/Technology%20Strategy%20Assessment%20-%20Compressed%20Air%20Energy%20Storage_0.pdf</a>
        </li>
        <li>
            Net Zero CAES (2025).
            <br>
            <a href="https://urbanao.com/post/compressed-air-energy-storage-caes-a-comprehensive-2025-overview" style="color:skyblue">https://urbanao.com/post/compressed-air-energy-storage-caes-a-comprehensive-2025-overview</a>
        </li>
        <li>
            EPA CHP Emissions (2021).
            <br>
            <a href="https://www.epa.gov/sites/production/files/2015-07/documents/fuel_and_carbon_dioxide_emissions_savings_calculation_methodology_for_combined_heat_and_power_systems.pdf" style="color:skyblue">https://www.epa.gov/sites/production/files/2015-07/documents/fuel_and_carbon_dioxide_emissions_savings_calculation_methodology_for_combined_heat_and_power_systems.pdf</a>
        </li>
        <li>
            US DOE CHP Guide (2015).
            <br>
            <a href="https://www.energy.gov/sites/default/files/2019/01/f58/CHPGuide2015.pdf" style="color:skyblue">https://www.energy.gov/sites/default/files/2019/01/f58/CHPGuide2015.pdf</a>
        </li>
        <li>
            EIA CHP (2012).
            <br>
            <a href="https://www.eia.gov/todayinenergy/detail.php?id=8250" style="color:skyblue">https://www.eia.gov/todayinenergy/detail.php?id=8250</a>
        </li>
        <li>
            Direct Methanol Fuel Cells (2003).
            <br>
            <a href="https://www1.eere.energy.gov/hydrogenandfuelcells//pdfs/ive16_zelenay.pdf" style="color:skyblue">https://www1.eere.energy.gov/hydrogenandfuelcells//pdfs/ive16_zelenay.pdf</a>
        </li>
        <li>
            Direct Methanol Fuel Cells Review (2015).
            <br>
            <a href="https://link.springer.com/article/10.1557/mre.2015.4" style="color:skyblue">https://link.springer.com/article/10.1557/mre.2015.4</a>
        </li>
        <li>
            Methanol Fuel for DMFC (2023).
            <br>
            <a href="https://www.intechopen.com/chapters/1157844" style="color:skyblue">https://www.intechopen.com/chapters/1157844</a>
        </li>
        <li>
            World Bank Geothermal (2022).
            <br>
            <a href="https://www.esmap.org/sites/default/files/esmap-files/16103-WB_ESMAP%20Direct%20Use-WEB.pdf" style="color:skyblue">https://www.esmap.org/sites/default/files/esmap-files/16103-WB_ESMAP%20Direct%20Use-WEB.pdf</a>
        </li>
        <li>
            UMich Geothermal (2023).
            <br>
            <a href="https://css.umich.edu/publications/factsheets/energy/geothermal-energy-factsheet" style="color:skyblue">https://css.umich.edu/publications/factsheets/energy/geothermal-energy-factsheet</a>
        </li>
        <li>
            ORNL Geothermal (2024).
            <br>
            <a href="https://www.ornl.gov/news/ornl-study-projects-geothermal-heat-pumps-impact-carbon-emissions-and-electrical-grid-2050" style="color:skyblue">https://www.ornl.gov/news/ornl-study-projects-geothermal-heat-pumps-impact-carbon-emissions-and-electrical-grid-2050</a>
        </li>
        <li>
            US DOE Geothermal Heat Pumps (2023).
            <br>
            <a href="https://www.energy.gov/sites/prod/files/guide_to_geothermal_heat_pumps.pdf" style="color:skyblue">https://www.energy.gov/sites/prod/files/guide_to_geothermal_heat_pumps.pdf</a>
        </li>
        <li>
            EnergySage Geothermal (2023).
            <br>
            <a href="https://www.energysage.com/heat-pumps/pros-cons-geothermal-heat-pumps/" style="color:skyblue">https://www.energysage.com/heat-pumps/pros-cons-geothermal-heat-pumps/</a>
        </li>
        <li>
            Wikipedia GSHP (2023).
            <br>
            <a href="https://en.wikipedia.org/wiki/Ground_source_heat_pump" style="color:skyblue">https://en.wikipedia.org/wiki/Ground_source_heat_pump</a>
        </li>
        <li>
            Hydrogen in Gas Turbines (2022).
            <br>
            <a href="https://doi.org/10.1093/ijlct/ctac025" style="color:skyblue">https://doi.org/10.1093/ijlct/ctac025</a>
        </li>
        <li>
            Fuel Composition Impact (2019).
            <br>
            <a href="https://doi.org/10.1115/1.4044238" style="color:skyblue">https://doi.org/10.1115/1.4044238</a>
        </li>
        <li>
            US DOE Hydrogen Review (2022).
            <br>
            <a href="https://www.netl.doe.gov/sites/default/files/publication/A-Literature-Review-of-Hydrogen-and-Natural-Gas-Turbines-081222.pdf" style="color:skyblue">https://www.netl.doe.gov/sites/default/files/publication/A-Literature-Review-of-Hydrogen-and-Natural-Gas-Turbines-081222.pdf</a>
        </li>
        <li>
            Modernizing Gas-Fired District Heating (2024).
            <br>
            <a href="https://doi.org/10.3390/su16041401" style="color:skyblue">https://doi.org/10.3390/su16041401</a>
        </li>
        <li>
            Hydrogen Storage Review (2023).
            <br>
            <a href="https://doi.org/10.4236/wjet.2023.113033" style="color:skyblue">https://doi.org/10.4236/wjet.2023.113033</a>
        </li>
        <li>
            Hydrogen Energy Storage (2022).
            <br>
            <a href="https://www.sandia.gov/app/uploads/sites/163/2022/03/ESHB_Ch11_Hydrogen_Headley.pdf" style="color:skyblue">https://www.sandia.gov/app/uploads/sites/163/2022/03/ESHB_Ch11_Hydrogen_Headley.pdf</a>
        </li>
        <li>
            Biomass for Energy (2024).
            <br>
            <a href="https://www.energy.gov/sites/default/files/2024-03/beto-2023-billion-ton-report_2-current_0.pdf" style="color:skyblue">https://www.energy.gov/sites/default/files/2024-03/beto-2023-billion-ton-report_2-current_0.pdf</a>
        </li>
        <li>
            Sustainable Biomass in Residential Sector (2018).
            <br>
            <a href="https://publications.jrc.ec.europa.eu/repository/bitstream/JRC113417/kjna29542enn.pdf" style="color:skyblue">https://publications.jrc.ec.europa.eu/repository/bitstream/JRC113417/kjna29542enn.pdf</a>
        </li>
        <li>
            EIA Hydropower (2023).
            <br>
            <a href="https://www.eia.gov/energyexplained/hydropower/index.php" style="color:skyblue">https://www.eia.gov/energyexplained/hydropower/index.php</a>
        </li>
        <li>
            Britannica Hydroelectric (2025).
            <br>
            <a href="https://www.britannica.com/science/hydroelectric-power" style="color:skyblue">https://www.britannica.com/science/hydroelectric-power</a>
        </li>
        <li>
            Pumped Hydroelectric Storage (2024).
            <br>
            <a href="https://www.sandia.gov/app/uploads/sites/163/2024/08/ESHB_Ch9_PHS_Bera.pdf" style="color:skyblue">https://www.sandia.gov/app/uploads/sites/163/2024/08/ESHB_Ch9_PHS_Bera.pdf</a>
        </li>
        <li>
            EPA Fuel Oil Combustion (2020).
            <br>
            <a href="https://www.epa.gov/sites/production/files/2020-09/documents/1.3_fuel_oil_combustion.pdf" style="color:skyblue">https://www.epa.gov/sites/production/files/2020-09/documents/1.3_fuel_oil_combustion.pdf</a>
        </li>
        <li>
            Pumped Storage Hydropower (2022).
            <br>
            <a href="https://www.esmap.org/sites/default/files/ESP/WB_PSH_16Jun22.pdf" style="color:skyblue">https://www.esmap.org/sites/default/files/ESP/WB_PSH_16Jun22.pdf</a>
        </li>
        <li>
            Combustion Efficiency (Engineering Toolbox).
            <br>
            <a href="https://www.engineeringtoolbox.com/boiler-combustion-efficiency-d_271.html" style="color:skyblue">https://www.engineeringtoolbox.com/boiler-combustion-efficiency-d_271.html</a>
        </li>
        <li>
            EIA Hydropower Generation (2023).
            <br>
            <a href="https://www.eia.gov/energyexplained/hydropower/where-hydropower-is-generated.php" style="color:skyblue">https://www.eia.gov/energyexplained/hydropower/where-hydropower-is-generated.php</a>
        </li>
        <li>
            Run-of-River Hydropower Design (2019).
            <br>
            <a href="https://doi.org/10.1016/j.envsoft.2018.08.018" style="color:skyblue">https://doi.org/10.1016/j.envsoft.2018.08.018</a>
        </li>
        <li>
            Hybrid Heat Pump System (2020).
            <br>
            <a href="https://qsel.columbia.edu/assets/uploads/blog/2020/publications/cost-optimal-sizing-and-operation-of-a-hybrid-heat-pump-system-using-numerical-simulation.pdf" style="color:skyblue">https://qsel.columbia.edu/assets/uploads/blog/2020/publications/cost-optimal-sizing-and-operation-of-a-hybrid-heat-pump-system-using-numerical-simulation.pdf</a>
        </li>
        <li>
            Biomass CHP Technologies (2007).
            <br>
            <a href="https://www.epa.gov/sites/production/files/2015-07/documents/biomass_combined_heat_and_power_catalog_of_technologies_v.1.1.pdf" style="color:skyblue">https://www.epa.gov/sites/production/files/2015-07/documents/biomass_combined_heat_and_power_catalog_of_technologies_v.1.1.pdf</a>
        </li>
        <li>
            CHP Internal Combustion Engines (2017).
            <br>
            <a href="https://www.wef.org/globalassets/assets-wef/direct-download-library/public/03---resources/wsec-2017-tr-002-rbc-internal-combustion-engines---9.2017.pdf" style="color:skyblue">https://www.wef.org/globalassets/assets-wef/direct-download-library/public/03---resources/wsec-2017-tr-002-rbc-internal-combustion-engines---9.2017.pdf</a>
        </li>
        <li>
            Molten Carbonate Fuel Cell Analysis (2015).
            <br>
            <a href="https://www.hydrogen.energy.gov/docs/hydrogenprogramlibraries/pdfs/progress15/ix_4_ahmed_2015.pdf" style="color:skyblue">https://www.hydrogen.energy.gov/docs/hydrogenprogramlibraries/pdfs/progress15/ix_4_ahmed_2015.pdf</a>
        </li>
        <li>
            Electrolyzer-Methanation Integration (2022).
            <br>
            <a href="https://doi.org/10.1016/j.apenergy.2022.120268" style="color:skyblue">https://doi.org/10.1016/j.apenergy.2022.120268</a>
        </li>
        <li>
            Electric Resistance Heating (2020).
            <br>
            <a href="https://www.energy.gov/energysaver/electric-resistance-heating" style="color:skyblue">https://www.energy.gov/energysaver/electric-resistance-heating</a>
        </li>
        <li>
            Regenerative Fuel Cell Analysis (2020).
            <br>
            <a href="https://ntrs.nasa.gov/api/citations/20205000357/downloads/TM-20205000357.pdf" style="color:skyblue">https://ntrs.nasa.gov/api/citations/20205000357/downloads/TM-20205000357.pdf</a>
        </li>
        <li>
            Reversible Fuel Cells (2025).
            <br>
            <a href="https://doi.org/10.4236/jpee.2025.136001" style="color:skyblue">https://doi.org/10.4236/jpee.2025.136001</a>
        </li>
        <li>
            Solar PV Energy (2024).
            <br>
            <a href="https://css.umich.edu/publications/factsheets/energy/solar-pv-energy-factsheet" style="color:skyblue">https://css.umich.edu/publications/factsheets/energy/solar-pv-energy-factsheet</a>
        </li>
        <li>
            PEM Electrolysis Targets (2022).
            <br>
            <a href="https://www.energy.gov/eere/fuelcells/technical-targets-proton-exchange-membrane-electrolysis" style="color:skyblue">https://www.energy.gov/eere/fuelcells/technical-targets-proton-exchange-membrane-electrolysis</a>
        </li>
        <li>
            Phosphoric Acid Fuel Cells (2024).
            <br>
            <a href="https://fuelcellz.com/phosphoric-acid-fuel-cells-pafc/" style="color:skyblue">https://fuelcellz.com/phosphoric-acid-fuel-cells-pafc/</a>
        </li>
        <li>
            PEM Fuel Cell Review (2022).
            <br>
            <a href="https://pubs.rsc.org/en/content/articlelanding/2022/ee/d2ee00790h" style="color:skyblue">https://pubs.rsc.org/en/content/articlelanding/2022/ee/d2ee00790h</a>
        </li>
        <li>
            High-Temperature Electrolysis (2024).
            <br>
            <a href="https://www.energy.gov/eere/fuelcells/technical-targets-high-temperature-electrolysis" style="color:skyblue">https://www.energy.gov/eere/fuelcells/technical-targets-high-temperature-electrolysis</a>
        </li>
        <li>
            Solar Thermal for Urban Sustainability (2025).
            <br>
            <a href="https://doi.org/10.3389/frsc.2025.1583316" style="color:skyblue">https://doi.org/10.3389/frsc.2025.1583316</a>
        </li>
        <li>
            Solid Oxide Fuel Cells (2025).
            <br>
            <a href="https://doi.org/10.3390/pr13041145" style="color:skyblue">https://doi.org/10.3390/pr13041145</a>
        </li>
        <li>
            Solar Thermal Energy (2023).
            <br>
            <a href="https://www.energysage.com/about-clean-energy/solar/solar-thermal-what-you-need-to-know/" style="color:skyblue">https://www.energysage.com/about-clean-energy/solar/solar-thermal-what-you-need-to-know/</a>
        </li>
        <li>
            Solar Heating for Residential/Industrial (2015).
            <br>
            <a href="https://energy.mit.edu/wp-content/uploads/2015/04/MITEI-WP-2015-04.pdf" style="color:skyblue">https://energy.mit.edu/wp-content/uploads/2015/04/MITEI-WP-2015-04.pdf</a>
        </li>
        <li>
            Biomass Combustion in Steam Power Plants (2023).
            <br>
            <a href="https://doi.org/10.1093/ce/zkad049" style="color:skyblue">https://doi.org/10.1093/ce/zkad049</a>
        </li>
        <li>
            Waste Heat to Power Systems (2015).
            <br>
            <a href="https://www.epa.gov/sites/default/files/2015-07/documents/waste_heat_to_power_systems.pdf" style="color:skyblue">https://www.epa.gov/sites/default/files/2015-07/documents/waste_heat_to_power_systems.pdf</a>
        </li>
        <li>
            Thermal Hot Water Storage (2025).
            <br>
            <a href="https://www.energystoragenl.nl/wp-content/uploads/2025/01/Thermische-HotWater.pdf" style="color:skyblue">https://www.energystoragenl.nl/wp-content/uploads/2025/01/Thermische-HotWater.pdf</a>
        </li>
        <li>
            Wave Power (2024).
            <br>
            <a href="https://www.eia.gov/energyexplained/hydropower/wave-power.php" style="color:skyblue">https://www.eia.gov/energyexplained/hydropower/wave-power.php</a>
        </li>
        <li>
            Water-Source Heat Pumps (2007).
            <br>
            <a href="https://www.trane.com/content/dam/Trane/Commercial/global/products-systems/education-training/engineers-newsletters/energy-environment/admapn024en_0507.pdf" style="color:skyblue">https://www.trane.com/content/dam/Trane/Commercial/global/products-systems/education-training/engineers-newsletters/energy-environment/admapn024en_0507.pdf</a>
        </li>
        <li>
            Offshore Wind Market Report (2023).
            <br>
            <a href="https://www.energy.gov/sites/default/files/2023-09/doe-offshore-wind-market-report-2023-edition.pdf" style="color:skyblue">https://www.energy.gov/sites/default/files/2023-09/doe-offshore-wind-market-report-2023-edition.pdf</a>
        </li>
        <li>
            IEA Wind Electricity (2024).
            <br>
            <a href="https://www.iea.org/energy-system/renewables/wind" style="color:skyblue">https://www.iea.org/energy-system/renewables/wind</a>
        </li>
        <li>
            Decarbonizing Building Thermal Systems (2024).
            <br>
            <a href="https://www.nrel.gov/docs/fy24osti/87812.pdf" style="color:skyblue">https://www.nrel.gov/docs/fy24osti/87812.pdf</a>
        </li>
    </ol>
</div>

<div style="text-align: justify; margin-left: 0.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
    This matrix—where rows correspond to different Energy Technologies and columns correspond to different Fuel Types—shows the values as fractions (ranging from 0.0000 to 1.0000), indicating the proportional contribution of each fuel/resource to the total energy input for a specific technology.
</div>
</div>
<hr style="border: 0.5px solid skyblue;">
</div>

In [266]:
# Data with 19 values for each column to match the index length
overal_fuel_technologies_match = {
    'AIR': [0.05, 0, 0, 0.95, 0, 0, 0, 0.25, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0.75, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 
    'AMO': [0, 0, 0, 0, 0, 0, 0, 0, 0.00201207243460765, 0, 0, 0, 0, 0.00343642611683849, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 
    'BIO': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0.15, 0, 0, 0, 0.00687285223367698, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0.136986301369863, 0.15, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0.0668924640135478, 0.15, 0, 0, 0, 0, 0, 0],
    'ELE': [0, 0, 1, 0, 1, 0, 0, 0.75, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0.75, 0.9, 0, 1, 0, 0, 0.95, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0],
    'GAS': [0, 0, 0, 0, 0, 0, 0, 0, 0.975855130784708, 0.75, 0, 0, 0, 0.646048109965636, 0.98, 0, 0, 0, 0, 0, 0.75, 0.9, 0, 0, 0, 0, 0, 0.383561643835616, 0.7, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0.0821337849280271, 0.7, 0, 0, 0, 0, 0, 0],
    'GEO': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0.00846740050804403, 0, 0, 0, 0, 0, 0, 0],
    'HRD': [0, 0, 0, 0, 0, 0, 0, 0, 0.00804828973843059, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0.329381879762913, 0, 0, 0, 0, 0, 0, 0],
    'HYD': [0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0.00343642611683849, 0.02, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0.25, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0, 0.000846740050804403, 0, 0, 0, 0, 0, 0, 0],
    'LIG': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0.210838272650296, 0, 0, 0, 0, 0, 0, 0],
    'NUC': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0.116850127011008, 0, 0, 0, 0, 0, 0, 0],
    'OIL': [0, 0, 0, 0, 0, 0, 0, 0, 0.0140845070422535, 0.1, 0, 0, 0, 0.323024054982818, 0, 0, 0, 0, 0, 0, 0.25, 0.1, 0, 0, 0, 0, 0, 0.36986301369863, 0.15, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0.123624047417443, 0.15, 0, 0, 0, 0, 0, 0],
    'OTH': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0.00687285223367698, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0.0136986301369863, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0.000846740050804403, 0, 0, 0, 0, 0, 0, 0],
    'PEA': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0.0143945808636749, 0, 0, 0, 0, 0, 0, 0],
    'SUN': [0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 1, 0, 0.00592718035563082, 0, 0, 0, 0, 0, 0, 0],
    'THE': [0.95, 0, 0, 0.05, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0.25, 0, 0, 0, 0, 0, 0, 0.1, 0, 0, 0, 0, 0.05, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0],
    'WAT': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 0, 0, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0.000846740050804403, 0, 0, 1, 0, 0, 0, 0],
    'WHT': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0],
    'WIN': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1],
    'WST': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0.0103092783505155, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0.0958904109589041, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0.0389500423370025, 0, 0, 0, 0, 0, 0, 0]
}

index = ['ABHP','ALFC','ALKE','ASHP','BATS','BEVS','BSPG','CAES','COMC','COMCX','DMFC','GETH','GSHP','GTUR','GTURX','H2ST','HBBS','HDAM','HDAMC','HDLZ','HOBO','HOBOX','HPHS','HPHSC','HROR','HRORC','HYHP','ICEN','ICENX','MCFC','PAFC','P2BS','P2GS','P2HT','PEFC','PEME','PHOT','REFC','REHE','SCSP','SOFC','SOTH','SOXE','STUR','STURX','THMS','WAVE','WHEN','WTOF','WSHP','WTON']

# Create DataFrame
overal_fuel_technologies_match_df = pd.DataFrame(overal_fuel_technologies_match, index=index)

# Display the DataFrame
overal_fuel_technologies_match_df

,AIR,AMO,BIO,ELE,GAS,GEO,HRD,HYD,LIG,NUC,OIL,OTH,PEA,SUN,THE,WAT,WHT,WIN,WST
ABHP,0.05,0.000000,0.000000,0.00,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.95,0.000000,0,0,0.000000
ALFC,0.00,0.000000,0.000000,0.00,0.000000,0.000000,0.000000,1.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.00,0.000000,0,0,0.000000
ALKE,0.00,0.000000,0.000000,1.00,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.00,0.000000,0,0,0.000000
ASHP,0.95,0.000000,0.000000,0.00,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.05,0.000000,0,0,0.000000
BATS,0.00,0.000000,0.000000,1.00,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.00,0.000000,0,0,0.000000
BEVS,0.00,0.000000,0.000000,0.00,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,1.000000,0.00,0.000000,0,0,0.000000
BSPG,0.00,0.000000,0.000000,0.00,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,1.000000,0.00,0.000000,0,0,0.000000
CAES,0.25,0.000000,0.000000,0.75,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.00,0.000000,0,0,0.000000
COMC,0.00,0.002012,0.000000,0.00,0.975855,0.000000,0.008048,0.000000,0.000000,0.00000,0.014085,0.000000,0.000000,0.000000,0.00,0.000000,0,0,0.000000
COMCX,0.00,0.000000,0.150000,0.00,0.750000,0.000000,0.000000,0.000000,0.000000,0.00000,0.100000,0.000000,0.000000,0.000000,0.00,0.000000,0,0,0.000000


<div style="background-color: black;">
<hr style="border: 0.5px solid skyblue;">
<div style="text-align: justify; margin-left: 0.0em; font-weight: normal; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
The Dispa-SET Technology - Fuel matching table for Albania.
<div style="text-align: justify; margin-left: 3.0em; font-weight: normal; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
Although the previous table can be used as support for all countries since it provides overall values from around the world, the following table specifically focuses on the Alba Energy System.
The same was built using the following additional sources:
</div>
<div style="text-align: justify; margin-left: 5em; font-weight: normal; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
<ol style="margin-top: 0; padding-left: 1.5em;">
<li>
xxxxxxxxxxxxxxxxxxxxxx. 
<br>
<a href="xxxxxxxxxxxxxxxxxxx" style="color:skyblue">xxxxxxxxxxxxxxxxxxxxxxi</a>
</li>
</ol>
</div>
</div>
<hr style="border: 0.5px solid skyblue;">
</div>

In [267]:
AL_fuel_technologies_match = {
    'AIR': [],
    'AMO': [],
    'BIO': [],
    'ELE': [],
    'GAS': [],
    'GEO': [],
    'HRD': [],
    'HYD': [],
    'LIG': [],
    'NUC': [],
    'OIL': [],
    'OTH': [],
    'PEA': [],
    'SUN': [],
    'THE': [],
    'WAT': [],
    'WHT': [],
    'WIN': [],
    'WST': []
}
index = ['ABHP','ALFC','ALKE','ASHP','BATS','BEVS','BSPG','CAES','COMC','COMCX','DMFC','GETH','GSHP','GTUR','GTURX','H2ST','HBBS','HDAM','HDAMC','HDLZ','HOBO','HOBOX','HPHS','HPHSC','HROR','HRORC','HYHP','ICEN','ICENX','MCFC','PAFC','P2BS','P2GS','P2HT','PEFC','PEME','PHOT','REFC','REHE','SCSP','SOFC','SOTH','SOXE','STUR','STURX','THMS','WAVE','WHEN','WTOF','WSHP','WTON']

# Create DataFrame
AL_fuel_technologies_match_df = pd.DataFrame('N/A', index=index, columns=list(AL_fuel_technologies_match.keys()))

# Display the DataFrame
AL_fuel_technologies_match_df

,AIR,AMO,BIO,ELE,GAS,GEO,HRD,HYD,LIG,NUC,OIL,OTH,PEA,SUN,THE,WAT,WHT,WIN,WST
ABHP,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
ALFC,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
ALKE,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
ASHP,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
BATS,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
BEVS,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
BSPG,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
CAES,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
COMC,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
COMCX,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A


<div style="background-color: black;">
<hr style="border: 0.5px solid skyblue;">
<div style="text-align: justify; margin-left: 0.0em; font-weight: normal; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
The Dispa-SET Technology - Fuel matching table for Armenia.
<div style="text-align: justify; margin-left: 3.0em; font-weight: normal; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
Although the previous table can be used as support for all countries since it provides overall values from around the world, the following table specifically focuses on the Arme Energy System.
The same was built using the following additional sources:
</div>
<div style="text-align: justify; margin-left: 5em; font-weight: normal; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
<ol style="margin-top: 0; padding-left: 1.5em;">
<li>
xxxxxxxxxxxxxxxxxxxxxx. 
<br>
<a href="xxxxxxxxxxxxxxxxxxx" style="color:skyblue">xxxxxxxxxxxxxxxxxxxxxxi</a>
</li>
</ol>
</div>
</div>
<hr style="border: 0.5px solid skyblue;">
</div>

In [268]:
AM_fuel_technologies_match = {
    'AIR': [],
    'AMO': [],
    'BIO': [],
    'ELE': [],
    'GAS': [],
    'GEO': [],
    'HRD': [],
    'HYD': [],
    'LIG': [],
    'NUC': [],
    'OIL': [],
    'OTH': [],
    'PEA': [],
    'SUN': [],
    'THE': [],
    'WAT': [],
    'WHT': [],
    'WIN': [],
    'WST': []
}
index = ['ABHP','ALFC','ALKE','ASHP','BATS','BEVS','BSPG','CAES','COMC','COMCX','DMFC','GETH','GSHP','GTUR','GTURX','H2ST','HBBS','HDAM','HDAMC','HDLZ','HOBO','HOBOX','HPHS','HPHSC','HROR','HRORC','HYHP','ICEN','ICENX','MCFC','PAFC','P2BS','P2GS','P2HT','PEFC','PEME','PHOT','REFC','REHE','SCSP','SOFC','SOTH','SOXE','STUR','STURX','THMS','WAVE','WHEN','WTOF','WSHP','WTON']

# Create N/A DataFrame
AM_fuel_technologies_match_df = pd.DataFrame('N/A', index=index, columns=list(AM_fuel_technologies_match.keys()))

# Display the DataFrame
AM_fuel_technologies_match_df

,AIR,AMO,BIO,ELE,GAS,GEO,HRD,HYD,LIG,NUC,OIL,OTH,PEA,SUN,THE,WAT,WHT,WIN,WST
ABHP,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
ALFC,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
ALKE,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
ASHP,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
BATS,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
BEVS,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
BSPG,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
CAES,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
COMC,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
COMCX,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A


<div style="background-color: black;">
<hr style="border: 0.5px solid skyblue;">
<div style="text-align: justify; margin-left: 0.0em; font-weight: normal; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
The Dispa-SET Technology - Fuel matching table for Azerbaijan.
<div style="text-align: justify; margin-left: 3.0em; font-weight: normal; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
Although the previous table can be used as support for all countries since it provides overall values from around the world, the following table specifically focuses on the Azer Energy System.
The same was built using the following additional sources:
</div>
<div style="text-align: justify; margin-left: 5em; font-weight: normal; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
<ol style="margin-top: 0; padding-left: 1.5em;">
<li>
xxxxxxxxxxxxxxxxxxxxxx. 
<br>
<a href="xxxxxxxxxxxxxxxxxxx" style="color:skyblue">xxxxxxxxxxxxxxxxxxxxxxi</a>
</li>
</ol>
</div>
</div>
<hr style="border: 0.5px solid skyblue;">
</div>

In [269]:
AZ_fuel_technologies_match = {
    'AIR': [],
    'AMO': [],
    'BIO': [],
    'ELE': [],
    'GAS': [],
    'GEO': [],
    'HRD': [],
    'HYD': [],
    'LIG': [],
    'NUC': [],
    'OIL': [],
    'OTH': [],
    'PEA': [],
    'SUN': [],
    'THE': [],
    'WAT': [],
    'WHT': [],
    'WIN': [],
    'WST': []
}
index = ['ABHP','ALFC','ALKE','ASHP','BATS','BEVS','BSPG','CAES','COMC','COMCX','DMFC','GETH','GSHP','GTUR','GTURX','H2ST','HBBS','HDAM','HDAMC','HDLZ','HOBO','HOBOX','HPHS','HPHSC','HROR','HRORC','HYHP','ICEN','ICENX','MCFC','PAFC','P2BS','P2GS','P2HT','PEFC','PEME','PHOT','REFC','REHE','SCSP','SOFC','SOTH','SOXE','STUR','STURX','THMS','WAVE','WHEN','WTOF','WSHP','WTON']

# Create N/A DataFrame
AZ_fuel_technologies_match_df = pd.DataFrame('N/A', index=index, columns=list(AZ_fuel_technologies_match.keys()))

# Display the DataFrame
AZ_fuel_technologies_match_df

,AIR,AMO,BIO,ELE,GAS,GEO,HRD,HYD,LIG,NUC,OIL,OTH,PEA,SUN,THE,WAT,WHT,WIN,WST
ABHP,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
ALFC,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
ALKE,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
ASHP,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
BATS,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
BEVS,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
BSPG,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
CAES,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
COMC,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
COMCX,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A


<div style="background-color: black;">
<hr style="border: 0.5px solid skyblue;">
<div style="text-align: justify; margin-left: 0.0em; font-weight: normal; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
The Dispa-SET Technology - Fuel matching table for Belarus.
<div style="text-align: justify; margin-left: 3.0em; font-weight: normal; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
Although the previous table can be used as support for all countries since it provides overall values from around the world, the following table specifically focuses on the Bela Energy System.
The same was built using the following additional sources:
</div>
<div style="text-align: justify; margin-left: 5em; font-weight: normal; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
<ol style="margin-top: 0; padding-left: 1.5em;">
<li>
xxxxxxxxxxxxxxxxxxxxxx. 
<br>
<a href="xxxxxxxxxxxxxxxxxxx" style="color:skyblue">xxxxxxxxxxxxxxxxxxxxxxi</a>
</li>
</ol>
</div>
</div>
<hr style="border: 0.5px solid skyblue;">
</div>

In [270]:
BY_fuel_technologies_match = {
    'AIR': [],
    'AMO': [],
    'BIO': [],
    'ELE': [],
    'GAS': [],
    'GEO': [],
    'HRD': [],
    'HYD': [],
    'LIG': [],
    'NUC': [],
    'OIL': [],
    'OTH': [],
    'PEA': [],
    'SUN': [],
    'THE': [],
    'WAT': [],
    'WHT': [],
    'WIN': [],
    'WST': []
}
index = ['ABHP','ALFC','ALKE','ASHP','BATS','BEVS','BSPG','CAES','COMC','COMCX','DMFC','GETH','GSHP','GTUR','GTURX','H2ST','HBBS','HDAM','HDAMC','HDLZ','HOBO','HOBOX','HPHS','HPHSC','HROR','HRORC','HYHP','ICEN','ICENX','MCFC','PAFC','P2BS','P2GS','P2HT','PEFC','PEME','PHOT','REFC','REHE','SCSP','SOFC','SOTH','SOXE','STUR','STURX','THMS','WAVE','WHEN','WTOF','WSHP','WTON']

# Create N/A DataFrame
BY_fuel_technologies_match_df = pd.DataFrame('N/A', index=index, columns=list(BY_fuel_technologies_match.keys()))

# Display the DataFrame
BY_fuel_technologies_match_df

,AIR,AMO,BIO,ELE,GAS,GEO,HRD,HYD,LIG,NUC,OIL,OTH,PEA,SUN,THE,WAT,WHT,WIN,WST
ABHP,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
ALFC,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
ALKE,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
ASHP,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
BATS,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
BEVS,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
BSPG,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
CAES,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
COMC,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
COMCX,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A


<div style="background-color: black;">
<hr style="border: 0.5px solid skyblue;">
<div style="text-align: justify; margin-left: 0.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
    The Dispa-SET Technology - Fuel matching table for Belgium
    <br>
<div style="text-align: justify; margin-left: 3.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
Although the previous table can be used as support for all countries since it provides overall values from around the world, the following table specifically focuses on the Belg Energy System.
The same was built using the following additional sources:
</div>
<div style="text-align: justify; margin-left: 5em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
    <ol style="margin-top: 0; padding-left: 1.5em;">
        <li>
            ELIA Group - Transparency platform (Accessed: 2024-11-27)
            <br>
            <a href="https://www.elia.be/en/grid-data" style="color:skyblue">https://www.elia.be/en/grid-data</a>
        </li>
        <li>
            International Energy Agency - Belgium (Accessed: 2024-11-27)
            <br>
            <a href="https://www.iea.org/countries/belgium" style="color:skyblue">https://www.iea.org/countries/belgium</a>
        </li>
        <li>
            FPS Economy - Energy (Accessed: 2024-11-27)
            <br>
            <a href="https://economie.fgov.be/en/themes/energy" style="color:skyblue">https://economie.fgov.be/en/themes/energy</a>
        </li>
    </ol>
</div>
</div>
<hr style="border: 0.5px solid skyblue;">
</div>

In [271]:
BE_fuel_technologies_match = {
    'AIR': [0.05, 0, 0, 0.95, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    'AMO': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0.1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    'BIO': [0, 0, 0, 0, 0, 0, 0, 0, 0.05, 0.15, 0, 0, 0, 0.05, 0.1, 0, 1, 0, 0, 0, 0.15, 0.2, 0, 0, 0, 0, 0.1, 0.05, 0.1, 0.1, 0.1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    'ELE': [0, 1, 1, 0.05, 1, 1, 0, 1, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0.95, 0.95, 0, 0, 0.9, 0, 0, 0, 0, 0.9, 1, 1, 0.1, 1, 0, 1, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0],
    'GAS': [0, 0, 0, 0, 0, 0, 0, 0, 0.95, 0.8, 0, 0, 0, 0.9, 0.85, 0, 0, 0, 0, 0, 0.75, 0.65, 0, 0, 0, 0, 0, 0.6, 0.65, 0.85, 0.85, 0, 0, 0, 0.9, 0, 0, 0, 0, 0, 0, 0, 0, 0.1, 0.2, 0, 0, 0, 0, 0, 0],
    'GEO': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    'HRD': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    'HYD': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    'LIG': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    'NUC': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0.8, 0.8, 0, 0, 0, 0, 0, 0],
    'OIL': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0.05, 0.05, 0, 0, 0, 0, 0, 0.1, 0.05, 0, 0, 0, 0, 0, 0.35, 0.25, 0.05, 0.05, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0.05, 0, 0, 0, 0, 0, 0, 0],
    'OTH': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    'PEA': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    'SUN': [0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    'THE': [0.95, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0],
    'WAT': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 0, 0, 0.05, 0.05, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0],
    'WHT': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    'WIN': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1],
    'WST': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0.05, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0]
}
index = ['ABHP','ALFC','ALKE','ASHP','BATS','BEVS','BSPG','CAES','COMC','COMCX','DMFC','GETH','GSHP','GTUR','GTURX','H2ST','HBBS','HDAM','HDAMC','HDLZ','HOBO','HOBOX','HPHS','HPHSC','HROR','HRORC','HYHP','ICEN','ICENX','MCFC','PAFC','P2BS','P2GS','P2HT','PEFC','PEME','PHOT','REFC','REHE','SCSP','SOFC','SOTH','SOXE','STUR','STURX','THMS','WAVE','WHEN','WTOF','WSHP','WTON']

# Create DataFrame
BE_fuel_technologies_match_df = pd.DataFrame(BE_fuel_technologies_match, index=index)

# Display the DataFrame
BE_fuel_technologies_match_df

,AIR,AMO,BIO,ELE,GAS,GEO,HRD,HYD,LIG,NUC,OIL,OTH,PEA,SUN,THE,WAT,WHT,WIN,WST
ABHP,0.05,0.0,0.00,0.00,0.00,0,0,0,0,0.0,0.00,0,0,0,0.95,0.00,0,0,0.00
ALFC,0.00,0.0,0.00,1.00,0.00,0,0,0,0,0.0,0.00,0,0,0,0.00,0.00,0,0,0.00
ALKE,0.00,0.0,0.00,1.00,0.00,0,0,0,0,0.0,0.00,0,0,0,0.00,0.00,0,0,0.00
ASHP,0.95,0.0,0.00,0.05,0.00,0,0,0,0,0.0,0.00,0,0,0,0.00,0.00,0,0,0.00
BATS,0.00,0.0,0.00,1.00,0.00,0,0,0,0,0.0,0.00,0,0,0,0.00,0.00,0,0,0.00
BEVS,0.00,0.0,0.00,1.00,0.00,0,0,0,0,0.0,0.00,0,0,0,0.00,0.00,0,0,0.00
BSPG,0.00,0.0,0.00,0.00,0.00,0,0,0,0,0.0,0.00,0,0,1,0.00,0.00,0,0,0.00
CAES,0.00,0.0,0.00,1.00,0.00,0,0,0,0,0.0,0.00,0,0,0,0.00,0.00,0,0,0.00
COMC,0.00,0.0,0.05,0.00,0.95,0,0,0,0,0.0,0.00,0,0,0,0.00,0.00,0,0,0.00
COMCX,0.00,0.0,0.15,0.00,0.80,0,0,0,0,0.0,0.00,0,0,0,0.00,0.00,0,0,0.00


<div style="background-color: black;">
<hr style="border: 0.5px solid skyblue;">
<div style="text-align: justify; margin-left: 0.0em; font-weight: normal; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
The Dispa-SET Technology - Fuel matching table for Bosnia and Herzegovina.
<div style="text-align: justify; margin-left: 3.0em; font-weight: normal; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
Although the previous table can be used as support for all countries since it provides overall values from around the world, the following table specifically focuses on the Bosn Energy System.
The same was built using the following additional sources:
</div>
<div style="text-align: justify; margin-left: 5em; font-weight: normal; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
<ol style="margin-top: 0; padding-left: 1.5em;">
<li>
xxxxxxxxxxxxxxxxxxxxxx. 
<br>
<a href="xxxxxxxxxxxxxxxxxxx" style="color:skyblue">xxxxxxxxxxxxxxxxxxxxxxi</a>
</li>
</ol>
</div>
</div>
<hr style="border: 0.5px solid skyblue;">
</div>

In [272]:
BA_fuel_technologies_match = {
    'AIR': [],
    'AMO': [],
    'BIO': [],
    'ELE': [],
    'GAS': [],
    'GEO': [],
    'HRD': [],
    'HYD': [],
    'LIG': [],
    'NUC': [],
    'OIL': [],
    'OTH': [],
    'PEA': [],
    'SUN': [],
    'THE': [],
    'WAT': [],
    'WHT': [],
    'WIN': [],
    'WST': []
}
index = ['ABHP','ALFC','ALKE','ASHP','BATS','BEVS','BSPG','CAES','COMC','COMCX','DMFC','GETH','GSHP','GTUR','GTURX','H2ST','HBBS','HDAM','HDAMC','HDLZ','HOBO','HOBOX','HPHS','HPHSC','HROR','HRORC','HYHP','ICEN','ICENX','MCFC','PAFC','P2BS','P2GS','P2HT','PEFC','PEME','PHOT','REFC','REHE','SCSP','SOFC','SOTH','SOXE','STUR','STURX','THMS','WAVE','WHEN','WTOF','WSHP','WTON']

# Create N/A DataFrame
BA_fuel_technologies_match_df = pd.DataFrame('N/A', index=index, columns=list(BA_fuel_technologies_match.keys()))

# Display the DataFrame
BA_fuel_technologies_match_df

,AIR,AMO,BIO,ELE,GAS,GEO,HRD,HYD,LIG,NUC,OIL,OTH,PEA,SUN,THE,WAT,WHT,WIN,WST
ABHP,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
ALFC,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
ALKE,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
ASHP,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
BATS,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
BEVS,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
BSPG,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
CAES,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
COMC,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
COMCX,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A


<div style="background-color: black;">
<hr style="border: 0.5px solid skyblue;">
<div style="text-align: justify; margin-left: 0.0em; font-weight: normal; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
The Dispa-SET Technology - Fuel matching table for Bulgaria.
<div style="text-align: justify; margin-left: 3.0em; font-weight: normal; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
Although the previous table can be used as support for all countries since it provides overall values from around the world, the following table specifically focuses on the Bulg Energy System.
The same was built using the following additional sources:
</div>
<div style="text-align: justify; margin-left: 5em; font-weight: normal; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
<ol style="margin-top: 0; padding-left: 1.5em;">
<li>
xxxxxxxxxxxxxxxxxxxxxx. 
<br>
<a href="xxxxxxxxxxxxxxxxxxx" style="color:skyblue">xxxxxxxxxxxxxxxxxxxxxxi</a>
</li>
</ol>
</div>
</div>
<hr style="border: 0.5px solid skyblue;">
</div>

In [273]:
BG_fuel_technologies_match = {
    'AIR': [],
    'AMO': [],
    'BIO': [],
    'ELE': [],
    'GAS': [],
    'GEO': [],
    'HRD': [],
    'HYD': [],
    'LIG': [],
    'NUC': [],
    'OIL': [],
    'OTH': [],
    'PEA': [],
    'SUN': [],
    'THE': [],
    'WAT': [],
    'WHT': [],
    'WIN': [],
    'WST': []
}
index = ['ABHP','ALFC','ALKE','ASHP','BATS','BEVS','BSPG','CAES','COMC','COMCX','DMFC','GETH','GSHP','GTUR','GTURX','H2ST','HBBS','HDAM','HDAMC','HDLZ','HOBO','HOBOX','HPHS','HPHSC','HROR','HRORC','HYHP','ICEN','ICENX','MCFC','PAFC','P2BS','P2GS','P2HT','PEFC','PEME','PHOT','REFC','REHE','SCSP','SOFC','SOTH','SOXE','STUR','STURX','THMS','WAVE','WHEN','WTOF','WSHP','WTON']

# Create N/A DataFrame
BG_fuel_technologies_match_df = pd.DataFrame('N/A', index=index, columns=list(BG_fuel_technologies_match.keys()))

# Display the DataFrame
BG_fuel_technologies_match_df

,AIR,AMO,BIO,ELE,GAS,GEO,HRD,HYD,LIG,NUC,OIL,OTH,PEA,SUN,THE,WAT,WHT,WIN,WST
ABHP,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
ALFC,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
ALKE,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
ASHP,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
BATS,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
BEVS,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
BSPG,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
CAES,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
COMC,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
COMCX,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A


<div style="background-color: black;">
<hr style="border: 0.5px solid skyblue;">
<div style="text-align: justify; margin-left: 0.0em; font-weight: normal; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
The Dispa-SET Technology - Fuel matching table for Croatia.
<div style="text-align: justify; margin-left: 3.0em; font-weight: normal; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
Although the previous table can be used as support for all countries since it provides overall values from around the world, the following table specifically focuses on the Croat Energy System.
The same was built using the following additional sources:
</div>
<div style="text-align: justify; margin-left: 5em; font-weight: normal; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
<ol style="margin-top: 0; padding-left: 1.5em;">
<li>
xxxxxxxxxxxxxxxxxxxxxx. 
<br>
<a href="xxxxxxxxxxxxxxxxxxx" style="color:skyblue">xxxxxxxxxxxxxxxxxxxxxxi</a>
</li>
</ol>
</div>
</div>
<hr style="border: 0.5px solid skyblue;">
</div>

In [274]:
HR_fuel_technologies_match = {
    'AIR': [],
    'AMO': [],
    'BIO': [],
    'ELE': [],
    'GAS': [],
    'GEO': [],
    'HRD': [],
    'HYD': [],
    'LIG': [],
    'NUC': [],
    'OIL': [],
    'OTH': [],
    'PEA': [],
    'SUN': [],
    'THE': [],
    'WAT': [],
    'WHT': [],
    'WIN': [],
    'WST': []
}
index = ['ABHP','ALFC','ALKE','ASHP','BATS','BEVS','BSPG','CAES','COMC','COMCX','DMFC','GETH','GSHP','GTUR','GTURX','H2ST','HBBS','HDAM','HDAMC','HDLZ','HOBO','HOBOX','HPHS','HPHSC','HROR','HRORC','HYHP','ICEN','ICENX','MCFC','PAFC','P2BS','P2GS','P2HT','PEFC','PEME','PHOT','REFC','REHE','SCSP','SOFC','SOTH','SOXE','STUR','STURX','THMS','WAVE','WHEN','WTOF','WSHP','WTON']

# Create N/A DataFrame
HR_fuel_technologies_match_df = pd.DataFrame('N/A', index=index, columns=list(HR_fuel_technologies_match.keys()))

# Display the DataFrame
HR_fuel_technologies_match_df

,AIR,AMO,BIO,ELE,GAS,GEO,HRD,HYD,LIG,NUC,OIL,OTH,PEA,SUN,THE,WAT,WHT,WIN,WST
ABHP,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
ALFC,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
ALKE,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
ASHP,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
BATS,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
BEVS,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
BSPG,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
CAES,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
COMC,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
COMCX,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A


<div style="background-color: black;">
<hr style="border: 0.5px solid skyblue;">
<div style="text-align: justify; margin-left: 0.0em; font-weight: normal; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
The Dispa-SET Technology - Fuel matching table for Cyprus.
<div style="text-align: justify; margin-left: 3.0em; font-weight: normal; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
Although the previous table can be used as support for all countries since it provides overall values from around the world, the following table specifically focuses on the Cypru Energy System.
The same was built using the following additional sources:
</div>
<div style="text-align: justify; margin-left: 5em; font-weight: normal; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
<ol style="margin-top: 0; padding-left: 1.5em;">
<li>
xxxxxxxxxxxxxxxxxxxxxx. 
<br>
<a href="xxxxxxxxxxxxxxxxxxx" style="color:skyblue">xxxxxxxxxxxxxxxxxxxxxxi</a>
</li>
</ol>
</div>
</div>
<hr style="border: 0.5px solid skyblue;">
</div>

In [275]:
CY_fuel_technologies_match = {
    'AIR': [],
    'AMO': [],
    'BIO': [],
    'ELE': [],
    'GAS': [],
    'GEO': [],
    'HRD': [],
    'HYD': [],
    'LIG': [],
    'NUC': [],
    'OIL': [],
    'OTH': [],
    'PEA': [],
    'SUN': [],
    'THE': [],
    'WAT': [],
    'WHT': [],
    'WIN': [],
    'WST': []
}
index = ['ABHP','ALFC','ALKE','ASHP','BATS','BEVS','BSPG','CAES','COMC','COMCX','DMFC','GETH','GSHP','GTUR','GTURX','H2ST','HBBS','HDAM','HDAMC','HDLZ','HOBO','HOBOX','HPHS','HPHSC','HROR','HRORC','HYHP','ICEN','ICENX','MCFC','PAFC','P2BS','P2GS','P2HT','PEFC','PEME','PHOT','REFC','REHE','SCSP','SOFC','SOTH','SOXE','STUR','STURX','THMS','WAVE','WHEN','WTOF','WSHP','WTON']

# Create N/A DataFrame
CY_fuel_technologies_match_df = pd.DataFrame('N/A', index=index, columns=list(CY_fuel_technologies_match.keys()))

# Display the DataFrame
CY_fuel_technologies_match_df

,AIR,AMO,BIO,ELE,GAS,GEO,HRD,HYD,LIG,NUC,OIL,OTH,PEA,SUN,THE,WAT,WHT,WIN,WST
ABHP,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
ALFC,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
ALKE,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
ASHP,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
BATS,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
BEVS,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
BSPG,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
CAES,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
COMC,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
COMCX,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A


<div style="background-color: black;">
<hr style="border: 0.5px solid skyblue;">
<div style="text-align: justify; margin-left: 0.0em; font-weight: normal; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
The Dispa-SET Technology - Fuel matching table for Czech Republic.
<div style="text-align: justify; margin-left: 3.0em; font-weight: normal; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
Although the previous table can be used as support for all countries since it provides overall values from around the world, the following table specifically focuses on the Cze Energy System.
The same was built using the following additional sources:
</div>
<div style="text-align: justify; margin-left: 5em; font-weight: normal; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
<ol style="margin-top: 0; padding-left: 1.5em;">
<li>
xxxxxxxxxxxxxxxxxxxxxx. 
<br>
<a href="xxxxxxxxxxxxxxxxxxx" style="color:skyblue">xxxxxxxxxxxxxxxxxxxxxxi</a>
</li>
</ol>
</div>
</div>
<hr style="border: 0.5px solid skyblue;">
</div>

In [276]:
CZ_fuel_technologies_match = {
    'AIR': [],
    'AMO': [],
    'BIO': [],
    'ELE': [],
    'GAS': [],
    'GEO': [],
    'HRD': [],
    'HYD': [],
    'LIG': [],
    'NUC': [],
    'OIL': [],
    'OTH': [],
    'PEA': [],
    'SUN': [],
    'THE': [],
    'WAT': [],
    'WHT': [],
    'WIN': [],
    'WST': []
}
index = ['ABHP','ALFC','ALKE','ASHP','BATS','BEVS','BSPG','CAES','COMC','COMCX','DMFC','GETH','GSHP','GTUR','GTURX','H2ST','HBBS','HDAM','HDAMC','HDLZ','HOBO','HOBOX','HPHS','HPHSC','HROR','HRORC','HYHP','ICEN','ICENX','MCFC','PAFC','P2BS','P2GS','P2HT','PEFC','PEME','PHOT','REFC','REHE','SCSP','SOFC','SOTH','SOXE','STUR','STURX','THMS','WAVE','WHEN','WTOF','WSHP','WTON']

# Create N/A DataFrame
CZ_fuel_technologies_match_df = pd.DataFrame('N/A', index=index, columns=list(CZ_fuel_technologies_match.keys()))

# Display the DataFrame
CZ_fuel_technologies_match_df

,AIR,AMO,BIO,ELE,GAS,GEO,HRD,HYD,LIG,NUC,OIL,OTH,PEA,SUN,THE,WAT,WHT,WIN,WST
ABHP,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
ALFC,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
ALKE,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
ASHP,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
BATS,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
BEVS,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
BSPG,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
CAES,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
COMC,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
COMCX,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A


<div style="background-color: black;">
<hr style="border: 0.5px solid skyblue;">
<div style="text-align: justify; margin-left: 0.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
    The Dispa-SET Technology - Fuel matching table for Denmark
    <br>
    <div style="text-align: justify; margin-left: 3.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
        Although the previous table can be used as support for all countries since it provides overall values from around the world, the following table specifically focuses on the Danish Energy System.
        The same was built using the following additional sources:
    </div>
    <div style="text-align: justify; margin-left: 5em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
        <ol style="margin-top: 0; padding-left: 1.5em;">
            <li>
                Statistics Denmark. (2024). <i>Energy statistics – Total 2023</i> (Report No. 2024-03-28).<br>
                <a href="https://www.dst.dk/en/Statistik/emner/energi-og-miljoe/energi" style="color:skyblue">https://www.dst.dk/en/Statistik/emner/energi-og-miljoe/energi</a>
            </li>
            <li>
                Danish Energy Agency. (2024). <i>Annual and monthly statistics</i>.<br>
                <a href="https://ens.dk/en/analyses-and-statistics/annual-and-monthly-statistics" style="color:skyblue">https://ens.dk/en/analyses-and-statistics/annual-and-monthly-statistics</a>
            </li>
            <li>
                International Energy Agency. (2023). <i>Denmark 2023: Energy policy review</i>.<br>
                <a href="https://www.iea.org/reports/denmark-2023" style="color:skyblue">https://www.iea.org/reports/denmark-2023</a>
            </li>
        </ol>
    </div>
</div>
<hr style="border: 0.5px solid skyblue;">
</div>

In [277]:
DK_fuel_technologies_match = {
    'AIR': [0.95, 0, 0, 0.95, 0, 0, 0, 0, 0, 0, 0, 0, 0.95, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0.9, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0.95, 0],
    'AMO': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    'BIO': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0.35, 0, 0, 0, 0, 0.4, 0, 1, 0, 0, 0.35, 0.45, 0.5, 0, 0, 0, 0, 0, 0.3, 0.4, 0, 0, 0.35, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0.25, 0.45, 0, 0, 0, 0, 0, 0],
    'ELE': [0.05, 0, 1, 0.05, 1, 1, 0, 0.65, 0, 0, 0, 0, 0.05, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0.1, 0, 0, 0, 0, 0.5, 1, 1, 0, 1, 0, 0, 1, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0.05, 0],
    'GAS': [0, 0, 0, 0, 0, 0, 0, 0.35, 0.7, 0.55, 0, 0, 0, 0.75, 0.5, 0, 0, 0, 0, 0.55, 0.4, 0.4, 0, 0, 0, 0, 0, 0.25, 0.45, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0.55, 0.45, 0, 0, 0, 0, 0, 0],
    'GEO': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    'HRD': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    'HYD': [0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0.25, 0.1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0.15, 0.15, 0, 0, 0.15, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0.1, 0, 0, 0, 0, 0, 0],
    'LIG': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    'NUC': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    'OIL': [0, 0, 0, 0, 0, 0, 0, 0, 0.3, 0.1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0.1, 0.15, 0.1, 0, 0, 0, 0, 0, 0.3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0.2, 0, 0, 0, 0, 0, 0, 0],
    'OTH': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    'PEA': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    'SUN': [0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    'THE': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0],
    'WAT': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0],
    'WHT': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    'WIN': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1],
    'WST': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
}
index = ['ABHP','ALFC','ALKE','ASHP','BATS','BEVS','BSPG','CAES','COMC','COMCX','DMFC','GETH','GSHP','GTUR','GTURX','H2ST','HBBS','HDAM','HDAMC','HDLZ','HOBO','HOBOX','HPHS','HPHSC','HROR','HRORC','HYHP','ICEN','ICENX','MCFC','PAFC','P2BS','P2GS','P2HT','PEFC','PEME','PHOT','REFC','REHE','SCSP','SOFC','SOTH','SOXE','STUR','STURX','THMS','WAVE','WHEN','WTOF','WSHP','WTON']

# Create DataFrame
DK_fuel_technologies_match_df = pd.DataFrame(DK_fuel_technologies_match, index=index)

# Display the DataFrame
DK_fuel_technologies_match_df

,AIR,AMO,BIO,ELE,GAS,GEO,HRD,HYD,LIG,NUC,OIL,OTH,PEA,SUN,THE,WAT,WHT,WIN,WST
ABHP,0.95,0,0.00,0.05,0.00,0,0,0.00,0,0,0.00,0,0,0,0,0,0,0,0
ALFC,0.00,0,0.00,0.00,0.00,0,0,1.00,0,0,0.00,0,0,0,0,0,0,0,0
ALKE,0.00,0,0.00,1.00,0.00,0,0,0.00,0,0,0.00,0,0,0,0,0,0,0,0
ASHP,0.95,0,0.00,0.05,0.00,0,0,0.00,0,0,0.00,0,0,0,0,0,0,0,0
BATS,0.00,0,0.00,1.00,0.00,0,0,0.00,0,0,0.00,0,0,0,0,0,0,0,0
BEVS,0.00,0,0.00,1.00,0.00,0,0,0.00,0,0,0.00,0,0,0,0,0,0,0,0
BSPG,0.00,0,0.00,0.00,0.00,0,0,0.00,0,0,0.00,0,0,1,0,0,0,0,0
CAES,0.00,0,0.00,0.65,0.35,0,0,0.00,0,0,0.00,0,0,0,0,0,0,0,0
COMC,0.00,0,0.00,0.00,0.70,0,0,0.00,0,0,0.30,0,0,0,0,0,0,0,0
COMCX,0.00,0,0.35,0.00,0.55,0,0,0.00,0,0,0.10,0,0,0,0,0,0,0,0


<div style="background-color: black;">
<hr style="border: 0.5px solid skyblue;">
<div style="text-align: justify; margin-left: 0.0em; font-weight: normal; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
The Dispa-SET Technology - Fuel matching table for Estonia.
<div style="text-align: justify; margin-left: 3.0em; font-weight: normal; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
Although the previous table can be used as support for all countries since it provides overall values from around the world, the following table specifically focuses on the Esto Energy System.
The same was built using the following additional sources:
</div>
<div style="text-align: justify; margin-left: 5em; font-weight: normal; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
<ol style="margin-top: 0; padding-left: 1.5em;">
<li>
xxxxxxxxxxxxxxxxxxxxxx. 
<br>
<a href="xxxxxxxxxxxxxxxxxxx" style="color:skyblue">xxxxxxxxxxxxxxxxxxxxxxi</a>
</li>
</ol>
</div>
</div>
<hr style="border: 0.5px solid skyblue;">
</div>

In [278]:
EE_fuel_technologies_match = {
    'AIR': [],
    'AMO': [],
    'BIO': [],
    'ELE': [],
    'GAS': [],
    'GEO': [],
    'HRD': [],
    'HYD': [],
    'LIG': [],
    'NUC': [],
    'OIL': [],
    'OTH': [],
    'PEA': [],
    'SUN': [],
    'THE': [],
    'WAT': [],
    'WHT': [],
    'WIN': [],
    'WST': []
}
index = ['ABHP','ALFC','ALKE','ASHP','BATS','BEVS','BSPG','CAES','COMC','COMCX','DMFC','GETH','GSHP','GTUR','GTURX','H2ST','HBBS','HDAM','HDAMC','HDLZ','HOBO','HOBOX','HPHS','HPHSC','HROR','HRORC','HYHP','ICEN','ICENX','MCFC','PAFC','P2BS','P2GS','P2HT','PEFC','PEME','PHOT','REFC','REHE','SCSP','SOFC','SOTH','SOXE','STUR','STURX','THMS','WAVE','WHEN','WTOF','WSHP','WTON']

# Create N/A DataFrame
EE_fuel_technologies_match_df = pd.DataFrame('N/A', index=index, columns=list(EE_fuel_technologies_match.keys()))

# Display the DataFrame
EE_fuel_technologies_match_df

,AIR,AMO,BIO,ELE,GAS,GEO,HRD,HYD,LIG,NUC,OIL,OTH,PEA,SUN,THE,WAT,WHT,WIN,WST
ABHP,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
ALFC,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
ALKE,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
ASHP,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
BATS,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
BEVS,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
BSPG,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
CAES,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
COMC,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
COMCX,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A


<div style="background-color: black;">
<hr style="border: 0.5px solid skyblue;">
<div style="text-align: justify; margin-left: 0.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
    The Dispa-SET Technology - Fuel matching table for Finland
    <br>
    <div style="text-align: justify; margin-left: 3.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
        Although the previous table can be used as support for all countries since it provides overall values from around the world, the following table specifically focuses on the Finnish Energy System.
        The same was built using the following additional sources:
    </div>
    <div style="text-align: justify; margin-left: 5em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
        <ol style="margin-top: 0; padding-left: 1.5em;">
            <li>
                Business Finland. (2022). <i>Future Watch: Growth Opportunities in the Hydrogen Economy</i> (Report).<br>
                <a href="https://www.businessfinland.fi/4961e3/globalassets/julkaisut/future-watch-growth-opportunities-in-the-hydrogen-economy.pdf" style="color:skyblue">https://www.businessfinland.fi/4961e3/globalassets/julkaisut/future-watch-growth-opportunities-in-the-hydrogen-economy.pdf</a>
            </li>
            <li>
                Finnish Bioenergy Association (Bioenergia ry). <i>Tietopankki (Knowledge Bank)</i>.<br>
                <a href="https://www.bioenergia.fi/tietopankki/" style="color:skyblue">https://www.bioenergia.fi/tietopankki/</a>
            </li>
            <li>
                Finnish Energy. (2023). <i>Electricity generation</i>.<br>
                <a href="https://energia.fi/en/energy-sector-in-finland/energy-production/electricity-generation/" style="color:skyblue">https://energia.fi/en/energy-sector-in-finland/energy-production/electricity-generation/</a>
            </li>
            <li>
                Renewables Finland (Suomen uusiutuvat ry). (2024). <i>Finnish Wind Power Association expands its activities to solar power</i>.<br>
                <a href="https://suomenuusiutuvat.fi/en/finnish-wind-power-association-expands-its-activities-to-solar-power/" style="color:skyblue">https://suomenuusiutuvat.fi/en/finnish-wind-power-association-expands-its-activities-to-solar-power/</a>
            </li>
            <li>
                International Energy Agency (IEA). (2024). <i>Heat Pumps</i>.<br>
                <a href="https://www.iea.org/energy-system/buildings/heat-pumps" style="color:skyblue">https://www.iea.org/energy-system/buildings/heat-pumps</a>
            </li>
            <li>
                Statistics Finland. (2025). <i>Production of heat</i> (Survey description).<br>
                <a href="https://stat.fi/en/surveys/ene" style="color:skyblue">https://stat.fi/en/surveys/ene</a>
            </li>
            <li>
                Statistics Finland. (2024). <i>Energy supply and consumption</i> (Official Statistics of Finland).<br>
                <a href="https://stat.fi/en/statistics/ehk" style="color:skyblue">https://stat.fi/en/statistics/ehk</a>
            </li>
        </ol>
    </div>
</div>
<hr style="border: 0.5px solid skyblue;">
</div>

In [279]:
FI_fuel_technologies_match = {
    'AIR': [0.95, 0, 0, 0.95, 0, 0, 0, 0, 0, 0, 0, 0, 0.95, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0.9, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0.95, 0],
    'AMO': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    'BIO': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0.5, 0, 0, 0, 0, 0.45, 0, 1, 0, 0, 0, 0.55, 0.6, 0, 0, 0, 0, 0, 0.25, 0.4, 0, 0, 0.4, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0.2, 0.5, 0, 0, 0, 0, 0, 0],
    'ELE': [0.05, 0, 1, 0.05, 1, 1, 0, 0.65, 0, 0, 0, 0, 0.05, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0.1, 0, 0, 0, 0, 0.45, 1, 1, 0, 1, 0, 0, 1, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0.05, 0],
    'GAS': [0, 0, 0, 0, 0, 0, 0, 0.35, 0.6, 0.4, 0, 0, 0, 0.7, 0.45, 0, 0, 0, 0, 0, 0.25, 0.3, 0, 0, 0, 0, 0, 0.2, 0.45, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0.25, 0.35, 0, 0, 0, 0, 0, 0],
    'GEO': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    'HRD': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    'HYD': [0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0.3, 0.1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0.15, 0.15, 0, 0, 0.15, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0.15, 0, 0, 0, 0, 0, 0],
    'LIG': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    'NUC': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0.55, 0, 0, 0, 0, 0, 0, 0],
    'OIL': [0, 0, 0, 0, 0, 0, 0, 0, 0.4, 0.1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0.2, 0.1, 0, 0, 0, 0, 0, 0.4, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    'OTH': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    'PEA': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    'SUN': [0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    'THE': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0],
    'WAT': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0],
    'WHT': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    'WIN': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1],
    'WST': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
}
index = ['ABHP','ALFC','ALKE','ASHP','BATS','BEVS','BSPG','CAES','COMC','COMCX','DMFC','GETH','GSHP','GTUR','GTURX','H2ST','HBBS','HDAM','HDAMC','HDLZ','HOBO','HOBOX','HPHS','HPHSC','HROR','HRORC','HYHP','ICEN','ICENX','MCFC','PAFC','P2BS','P2GS','P2HT','PEFC','PEME','PHOT','REFC','REHE','SCSP','SOFC','SOTH','SOXE','STUR','STURX','THMS','WAVE','WHEN','WTOF','WSHP','WTON']

# Create DataFrame
FI_fuel_technologies_match_df = pd.DataFrame(FI_fuel_technologies_match, index=index)

# Display the DataFrame
FI_fuel_technologies_match_df

,AIR,AMO,BIO,ELE,GAS,GEO,HRD,HYD,LIG,NUC,OIL,OTH,PEA,SUN,THE,WAT,WHT,WIN,WST
ABHP,0.95,0,0.00,0.05,0.00,0,0,0.00,0,0.00,0.0,0,0,0,0,0,0,0,0
ALFC,0.00,0,0.00,0.00,0.00,0,0,1.00,0,0.00,0.0,0,0,0,0,0,0,0,0
ALKE,0.00,0,0.00,1.00,0.00,0,0,0.00,0,0.00,0.0,0,0,0,0,0,0,0,0
ASHP,0.95,0,0.00,0.05,0.00,0,0,0.00,0,0.00,0.0,0,0,0,0,0,0,0,0
BATS,0.00,0,0.00,1.00,0.00,0,0,0.00,0,0.00,0.0,0,0,0,0,0,0,0,0
BEVS,0.00,0,0.00,1.00,0.00,0,0,0.00,0,0.00,0.0,0,0,0,0,0,0,0,0
BSPG,0.00,0,0.00,0.00,0.00,0,0,0.00,0,0.00,0.0,0,0,1,0,0,0,0,0
CAES,0.00,0,0.00,0.65,0.35,0,0,0.00,0,0.00,0.0,0,0,0,0,0,0,0,0
COMC,0.00,0,0.00,0.00,0.60,0,0,0.00,0,0.00,0.4,0,0,0,0,0,0,0,0
COMCX,0.00,0,0.50,0.00,0.40,0,0,0.00,0,0.00,0.1,0,0,0,0,0,0,0,0


<div style="background-color: black;">
<hr style="border: 0.5px solid skyblue;">
<div style="text-align: justify; margin-left: 0.0em; font-weight: normal; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
The Dispa-SET Technology - Fuel matching table for Georgia.
<div style="text-align: justify; margin-left: 3.0em; font-weight: normal; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
Although the previous table can be used as support for all countries since it provides overall values from around the world, the following table specifically focuses on the Georg Energy System.
The same was built using the following additional sources:
</div>
<div style="text-align: justify; margin-left: 5em; font-weight: normal; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
<ol style="margin-top: 0; padding-left: 1.5em;">
<li>
xxxxxxxxxxxxxxxxxxxxxx. 
<br>
<a href="xxxxxxxxxxxxxxxxxxx" style="color:skyblue">xxxxxxxxxxxxxxxxxxxxxxi</a>
</li>
</ol>
</div>
</div>
<hr style="border: 0.5px solid skyblue;">
</div>

In [280]:
GE_fuel_technologies_match = {
    'AIR': [],
    'AMO': [],
    'BIO': [],
    'ELE': [],
    'GAS': [],
    'GEO': [],
    'HRD': [],
    'HYD': [],
    'LIG': [],
    'NUC': [],
    'OIL': [],
    'OTH': [],
    'PEA': [],
    'SUN': [],
    'THE': [],
    'WAT': [],
    'WHT': [],
    'WIN': [],
    'WST': []
}
index = ['ABHP','ALFC','ALKE','ASHP','BATS','BEVS','BSPG','CAES','COMC','COMCX','DMFC','GETH','GSHP','GTUR','GTURX','H2ST','HBBS','HDAM','HDAMC','HDLZ','HOBO','HOBOX','HPHS','HPHSC','HROR','HRORC','HYHP','ICEN','ICENX','MCFC','PAFC','P2BS','P2GS','P2HT','PEFC','PEME','PHOT','REFC','REHE','SCSP','SOFC','SOTH','SOXE','STUR','STURX','THMS','WAVE','WHEN','WTOF','WSHP','WTON']

# Create N/A DataFrame
GE_fuel_technologies_match_df = pd.DataFrame('N/A', index=index, columns=list(GE_fuel_technologies_match.keys()))

# Display the DataFrame
GE_fuel_technologies_match_df

,AIR,AMO,BIO,ELE,GAS,GEO,HRD,HYD,LIG,NUC,OIL,OTH,PEA,SUN,THE,WAT,WHT,WIN,WST
ABHP,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
ALFC,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
ALKE,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
ASHP,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
BATS,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
BEVS,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
BSPG,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
CAES,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
COMC,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
COMCX,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A


<div style="background-color: black;">
<hr style="border: 0.5px solid skyblue;">
<div style="text-align: justify; margin-left: 0.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
    The Dispa-SET Technology - Fuel matching table for Germany
    <br>
<div style="text-align: justify; margin-left: 3.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
Although the previous table can be used as support for all countries since it provides overall values from around the world, the following table specifically focuses on the German Energy System.
    The same was built using the following additional sources:
</div>
<div style="text-align: justify; margin-left: 5em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
    <ol style="margin-top: 0; padding-left: 1.5em;">
        <li>
            Bundesministerium für Wirtschaft und Klimaschutz (BMWK). (2023). Energiedaten: Gesamtausgabe [Energy Data: Complete Edition].
            <br>
            <a href="https://www.bmwk.de/Redaktion/DE/Artikel/Energie/energiedaten-gesamtausgabe.html" style="color:skyblue">https://www.bmwk.de/Redaktion/DE/Artikel/Energie/energiedaten-gesamtausgabe.html</a>
        </li>
        <li>
            Umweltbundesamt (UBA). (2023). Erneuerbare Energien in Zahlen [Renewable Energies in Figures].
            <br>
            <a href="https://www.umweltbundesamt.de/themen/klima-energie/erneuerbare-energien/erneuerbare-energien-in-zahlen" style="color:skyblue">https://www.umweltbundesamt.de/themen/klima-energie/erneuerbare-energien/erneuerbare-energien-in-zahlen</a>
        </li>
        <li>
            International Energy Agency. (2023). Germany 2023: Energy Policy Review. IEA.
            <br>
            <a href="https://www.iea.org/reports/germany-2025" style="color:skyblue">https://www.iea.org/reports/germany-2025</a>
        </li>
        <li>
            Fraunhofer Institute for Solar Energy Systems (ISE). (2024). Energy Charts.
            <br>
            <a href="https://www.energy-charts.info/index.html?l=en&c=DE" style="color:skyblue">https://www.energy-charts.info/index.html?l=en&c=DE</a>
        </li>
        <li>
            Eurostat. (2024). Energy database. European Commission.
            <br>
            <a href="https://ec.europa.eu/eurostat/web/energy/" style="color:skyblue">https://ec.europa.eu/eurostat/web/energy/</a>
        </li>
    </ol>
</div>
</div>
<hr style="border: 0.5px solid skyblue;">
</div>

In [281]:
DE_fuel_technologies_match = {
    'AIR': [0.05, 0, 0, 0.95, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    'AMO': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0.15, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    'BIO': [0, 0, 0, 0, 0, 0, 0, 0, 0.08, 0.2, 0, 0, 0, 0.05, 0.15, 0, 1, 0, 0, 0, 0.2, 0.25, 0, 0, 0, 0, 0.15, 0.05, 0.1, 0.15, 0.15, 0, 0, 0, 0.15, 0, 0, 0, 0, 0, 0.1, 0, 0, 0.05, 0.05, 0, 0, 0, 0, 0, 0],
    'ELE': [0, 1, 1, 0.05, 1, 1, 0, 1, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0.95, 0.95, 0, 0, 0.85, 0, 0, 0, 0, 0.85, 1, 1, 0.1, 1, 0, 1, 1, 0, 0.1, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0],
    'GAS': [0, 0, 0, 0, 0, 0, 0, 0, 0.85, 0.7, 0, 0, 0, 0.8, 0.7, 0, 0, 0, 0, 0, 0.55, 0.5, 0, 0, 0, 0, 0, 0.25, 0.3, 0.8, 0.8, 0, 0, 0, 0.75, 0, 0, 0, 0, 0, 0.75, 0, 0, 0.1, 0.15, 0, 0, 0, 0, 0, 0],
    'GEO': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    'HRD': [0, 0, 0, 0, 0, 0, 0, 0, 0.05, 0.05, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0.1, 0.1, 0, 0, 0, 0, 0, 0.1, 0.15, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0.4, 0.5, 0, 0, 0, 0, 0, 0],
    'HYD': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    'LIG': [0, 0, 0, 0, 0, 0, 0, 0, 0.02, 0.05, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0.05, 0.05, 0, 0, 0, 0, 0, 0.05, 0.1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0.45, 0.3, 0, 0, 0, 0, 0, 0],
    'NUC': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    'OIL': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0.15, 0.15, 0, 0, 0, 0, 0, 0.1, 0.05, 0, 0, 0, 0, 0, 0.55, 0.35, 0.05, 0.05, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0.05, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    'OTH': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    'PEA': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    'SUN': [0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    'THE': [0.95, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0],
    'WAT': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 0, 0, 0.05, 0.05, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0],
    'WHT': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0],
    'WIN': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1],
    'WST': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0.05, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
}

index = ['ABHP','ALFC','ALKE','ASHP','BATS','BEVS','BSPG','CAES','COMC','COMCX','DMFC','GETH','GSHP','GTUR','GTURX','H2ST','HBBS','HDAM','HDAMC','HDLZ','HOBO','HOBOX','HPHS','HPHSC','HROR','HRORC','HYHP','ICEN','ICENX','MCFC','PAFC','P2BS','P2GS','P2HT','PEFC','PEME','PHOT','REFC','REHE','SCSP','SOFC','SOTH','SOXE','STUR','STURX','THMS','WAVE','WHEN','WTOF','WSHP','WTON']

# Create DataFrame
DE_fuel_technologies_match_df = pd.DataFrame(DE_fuel_technologies_match, index=index)

# Display the DataFrame
DE_fuel_technologies_match_df

,AIR,AMO,BIO,ELE,GAS,GEO,HRD,HYD,LIG,NUC,OIL,OTH,PEA,SUN,THE,WAT,WHT,WIN,WST
ABHP,0.05,0.00,0.00,0.00,0.00,0,0.00,0,0.00,0,0.00,0,0,0,0.95,0.00,0,0,0.00
ALFC,0.00,0.00,0.00,1.00,0.00,0,0.00,0,0.00,0,0.00,0,0,0,0.00,0.00,0,0,0.00
ALKE,0.00,0.00,0.00,1.00,0.00,0,0.00,0,0.00,0,0.00,0,0,0,0.00,0.00,0,0,0.00
ASHP,0.95,0.00,0.00,0.05,0.00,0,0.00,0,0.00,0,0.00,0,0,0,0.00,0.00,0,0,0.00
BATS,0.00,0.00,0.00,1.00,0.00,0,0.00,0,0.00,0,0.00,0,0,0,0.00,0.00,0,0,0.00
BEVS,0.00,0.00,0.00,1.00,0.00,0,0.00,0,0.00,0,0.00,0,0,0,0.00,0.00,0,0,0.00
BSPG,0.00,0.00,0.00,0.00,0.00,0,0.00,0,0.00,0,0.00,0,0,1,0.00,0.00,0,0,0.00
CAES,0.00,0.00,0.00,1.00,0.00,0,0.00,0,0.00,0,0.00,0,0,0,0.00,0.00,0,0,0.00
COMC,0.00,0.00,0.08,0.00,0.85,0,0.05,0,0.02,0,0.00,0,0,0,0.00,0.00,0,0,0.00
COMCX,0.00,0.00,0.20,0.00,0.70,0,0.05,0,0.05,0,0.00,0,0,0,0.00,0.00,0,0,0.00


<div style="background-color: black;">
<hr style="border: 0.5px solid skyblue;">
<div style="text-align: justify; margin-left: 0.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
    The Dispa-SET Technology - Fuel matching table for France
    <br>
<div style="text-align: justify; margin-left: 3.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
Although the previous table can be used as support for all countries since it provides overall values from around the world, the following table specifically focuses on the French Energy System.
    The same was built using the following additional sources:
</div>
<div style="text-align: justify; margin-left: 5em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
    <ol style="margin-top: 0; padding-left: 1.5em;">
        <li>
            RTE - Eco2mix Data: Detailed Electricity Generation by Technology (2025).
            <br>
            <a href="https://www.rte-france.com/en/eco2mix" style="color:skyblue">https://www.rte-france.com/en/eco2mix</a>
        </li>
        <li>
            Eurostat - Energy Balances for France (2025).
            <br>
            <a href="https://ec.europa.eu/eurostat/web/energy" style="color:skyblue">https://ec.europa.eu/eurostat/web/energy</a>
        </li>
        <li>
            Ministère de la Transition Énergétique - Chiffres clés de l'énergie et Plan National Énergie-Climat (2025).
            <br>
            <a href="https://www.statistiques.developpement-durable.gouv.fr/energie" style="color:skyblue">https://www.statistiques.developpement-durable.gouv.fr/energie</a>
        </li>
        <li>
            International Energy Agency (IEA) - France Country Review and Energy Policy Analysis (2025).
            <br>
            <a href="https://www.iea.org/countries/france" style="color:skyblue">https://www.iea.org/countries/france</a>
        </li>
    </ol>
</div>
</div>
<hr style="border: 0.5px solid skyblue;">
</div>

In [282]:
FR_fuel_technologies_match = {
    'AIR': [0.05, 0, 0, 0.95, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    'AMO': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0.1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    'BIO': [0, 0, 0, 0, 0, 0, 0, 0, 0.03, 0.1, 0, 0, 0, 0.05, 0.1, 0, 1, 0, 0, 0, 0.15, 0.2, 0, 0, 0, 0, 0.1, 0.05, 0.1, 0.1, 0.1, 0, 0, 0, 0.1, 0, 0, 0, 0, 0, 0.1, 0, 0, 0.05, 0.05, 0, 0, 0, 0, 0, 0],
    'ELE': [0, 1, 1, 0.05, 1, 1, 0, 1, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0.95, 0.95, 0, 0, 0.9, 0, 0, 0, 0, 0.9, 1, 1, 0.1, 1, 0, 1, 1, 0, 0.1, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0],
    'GAS': [0, 0, 0, 0, 0, 0, 0, 0, 0.1, 0.15, 0, 0, 0, 0.4, 0.35, 0, 0, 0, 0, 0, 0.3, 0.25, 0, 0, 0, 0, 0, 0.25, 0.3, 0.7, 0.7, 0, 0, 0, 0.75, 0, 0, 0, 0, 0, 0.75, 0, 0, 0.05, 0.1, 0, 0, 0, 0, 0, 0],
    'GEO': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    'HRD': [0, 0, 0, 0, 0, 0, 0, 0, 0.02, 0.05, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0.1, 0.05, 0, 0, 0, 0, 0, 0.1, 0.15, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0.05, 0.1, 0, 0, 0, 0, 0, 0],
    'HYD': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    'LIG': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0.05, 0.1, 0, 0, 0, 0, 0, 0],
    'NUC': [0, 0, 0, 0, 0, 0, 0, 0, 0.8, 0.65, 0, 0, 0, 0.5, 0.5, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0.8, 0.65, 0, 0, 0, 0, 0, 0],
    'OIL': [0, 0, 0, 0, 0, 0, 0, 0, 0.05, 0.05, 0, 0, 0, 0.05, 0.05, 0, 0, 0, 0, 0, 0.45, 0.45, 0, 0, 0, 0, 0, 0.6, 0.45, 0.2, 0.2, 0, 0, 0, 0.05, 0, 0, 0, 0, 0, 0.05, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    'OTH': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    'PEA': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    'SUN': [0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    'THE': [0.95, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0],
    'WAT': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 0, 0, 0.05, 0.05, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0],
    'WHT': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0],
    'WIN': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1],
    'WST': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0.05, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
}

index = ['ABHP','ALFC','ALKE','ASHP','BATS','BEVS','BSPG','CAES','COMC','COMCX','DMFC','GETH','GSHP','GTUR','GTURX','H2ST','HBBS','HDAM','HDAMC','HDLZ','HOBO','HOBOX','HPHS','HPHSC','HROR','HRORC','HYHP','ICEN','ICENX','MCFC','PAFC','P2BS','P2GS','P2HT','PEFC','PEME','PHOT','REFC','REHE','SCSP','SOFC','SOTH','SOXE','STUR','STURX','THMS','WAVE','WHEN','WTOF','WSHP','WTON']

# Create DataFrame
FR_fuel_technologies_match_df = pd.DataFrame(FR_fuel_technologies_match, index=index)

# Display the DataFrame
FR_fuel_technologies_match_df

,AIR,AMO,BIO,ELE,GAS,GEO,HRD,HYD,LIG,NUC,OIL,OTH,PEA,SUN,THE,WAT,WHT,WIN,WST
ABHP,0.05,0.0,0.00,0.00,0.00,0,0.00,0,0.00,0.00,0.00,0,0,0,0.95,0.00,0,0,0.00
ALFC,0.00,0.0,0.00,1.00,0.00,0,0.00,0,0.00,0.00,0.00,0,0,0,0.00,0.00,0,0,0.00
ALKE,0.00,0.0,0.00,1.00,0.00,0,0.00,0,0.00,0.00,0.00,0,0,0,0.00,0.00,0,0,0.00
ASHP,0.95,0.0,0.00,0.05,0.00,0,0.00,0,0.00,0.00,0.00,0,0,0,0.00,0.00,0,0,0.00
BATS,0.00,0.0,0.00,1.00,0.00,0,0.00,0,0.00,0.00,0.00,0,0,0,0.00,0.00,0,0,0.00
BEVS,0.00,0.0,0.00,1.00,0.00,0,0.00,0,0.00,0.00,0.00,0,0,0,0.00,0.00,0,0,0.00
BSPG,0.00,0.0,0.00,0.00,0.00,0,0.00,0,0.00,0.00,0.00,0,0,1,0.00,0.00,0,0,0.00
CAES,0.00,0.0,0.00,1.00,0.00,0,0.00,0,0.00,0.00,0.00,0,0,0,0.00,0.00,0,0,0.00
COMC,0.00,0.0,0.03,0.00,0.10,0,0.02,0,0.00,0.80,0.05,0,0,0,0.00,0.00,0,0,0.00
COMCX,0.00,0.0,0.10,0.00,0.15,0,0.05,0,0.00,0.65,0.05,0,0,0,0.00,0.00,0,0,0.00


<div style="background-color: black;">
<hr style="border: 0.5px solid skyblue;">
<div style="text-align: justify; margin-left: 0.0em; font-weight: normal; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
The Dispa-SET Technology - Fuel matching table for Greece.
<div style="text-align: justify; margin-left: 3.0em; font-weight: normal; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
Although the previous table can be used as support for all countries since it provides overall values from around the world, the following table specifically focuses on the Gree Energy System.
The same was built using the following additional sources:
</div>
<div style="text-align: justify; margin-left: 5em; font-weight: normal; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
<ol style="margin-top: 0; padding-left: 1.5em;">
<li>
xxxxxxxxxxxxxxxxxxxxxx. 
<br>
<a href="xxxxxxxxxxxxxxxxxxx" style="color:skyblue">xxxxxxxxxxxxxxxxxxxxxxi</a>
</li>
</ol>
</div>
</div>
<hr style="border: 0.5px solid skyblue;">
</div>

In [283]:
EL_fuel_technologies_match = {
    'AIR': [],
    'AMO': [],
    'BIO': [],
    'ELE': [],
    'GAS': [],
    'GEO': [],
    'HRD': [],
    'HYD': [],
    'LIG': [],
    'NUC': [],
    'OIL': [],
    'OTH': [],
    'PEA': [],
    'SUN': [],
    'THE': [],
    'WAT': [],
    'WHT': [],
    'WIN': [],
    'WST': []
}
index = ['ABHP','ALFC','ALKE','ASHP','BATS','BEVS','BSPG','CAES','COMC','COMCX','DMFC','GETH','GSHP','GTUR','GTURX','H2ST','HBBS','HDAM','HDAMC','HDLZ','HOBO','HOBOX','HPHS','HPHSC','HROR','HRORC','HYHP','ICEN','ICENX','MCFC','PAFC','P2BS','P2GS','P2HT','PEFC','PEME','PHOT','REFC','REHE','SCSP','SOFC','SOTH','SOXE','STUR','STURX','THMS','WAVE','WHEN','WTOF','WSHP','WTON']

# Create N/A DataFrame
EL_fuel_technologies_match_df = pd.DataFrame('N/A', index=index, columns=list(EL_fuel_technologies_match.keys()))

# Display the DataFrame
EL_fuel_technologies_match_df

,AIR,AMO,BIO,ELE,GAS,GEO,HRD,HYD,LIG,NUC,OIL,OTH,PEA,SUN,THE,WAT,WHT,WIN,WST
ABHP,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
ALFC,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
ALKE,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
ASHP,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
BATS,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
BEVS,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
BSPG,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
CAES,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
COMC,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
COMCX,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A


<div style="background-color: black;">
<hr style="border: 0.5px solid skyblue;">
<div style="text-align: justify; margin-left: 0.0em; font-weight: normal; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
The Dispa-SET Technology - Fuel matching table for Hungary.
<div style="text-align: justify; margin-left: 3.0em; font-weight: normal; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
Although the previous table can be used as support for all countries since it provides overall values from around the world, the following table specifically focuses on the Hunga Energy System.
The same was built using the following additional sources:
</div>
<div style="text-align: justify; margin-left: 5em; font-weight: normal; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
<ol style="margin-top: 0; padding-left: 1.5em;">
<li>
xxxxxxxxxxxxxxxxxxxxxx. 
<br>
<a href="xxxxxxxxxxxxxxxxxxx" style="color:skyblue">xxxxxxxxxxxxxxxxxxxxxxi</a>
</li>
</ol>
</div>
</div>
<hr style="border: 0.5px solid skyblue;">
</div>

In [284]:
HU_fuel_technologies_match = {
    'AIR': [],
    'AMO': [],
    'BIO': [],
    'ELE': [],
    'GAS': [],
    'GEO': [],
    'HRD': [],
    'HYD': [],
    'LIG': [],
    'NUC': [],
    'OIL': [],
    'OTH': [],
    'PEA': [],
    'SUN': [],
    'THE': [],
    'WAT': [],
    'WHT': [],
    'WIN': [],
    'WST': []
}
index = ['ABHP','ALFC','ALKE','ASHP','BATS','BEVS','BSPG','CAES','COMC','COMCX','DMFC','GETH','GSHP','GTUR','GTURX','H2ST','HBBS','HDAM','HDAMC','HDLZ','HOBO','HOBOX','HPHS','HPHSC','HROR','HRORC','HYHP','ICEN','ICENX','MCFC','PAFC','P2BS','P2GS','P2HT','PEFC','PEME','PHOT','REFC','REHE','SCSP','SOFC','SOTH','SOXE','STUR','STURX','THMS','WAVE','WHEN','WTOF','WSHP','WTON']

# Create N/A DataFrame
HU_fuel_technologies_match_df = pd.DataFrame('N/A', index=index, columns=list(HU_fuel_technologies_match.keys()))

# Display the DataFrame
HU_fuel_technologies_match_df

,AIR,AMO,BIO,ELE,GAS,GEO,HRD,HYD,LIG,NUC,OIL,OTH,PEA,SUN,THE,WAT,WHT,WIN,WST
ABHP,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
ALFC,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
ALKE,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
ASHP,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
BATS,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
BEVS,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
BSPG,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
CAES,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
COMC,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
COMCX,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A


<div style="background-color: black;">
<hr style="border: 0.5px solid skyblue;">
<div style="text-align: justify; margin-left: 0.0em; font-weight: normal; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
The Dispa-SET Technology - Fuel matching table for Iceland.
<div style="text-align: justify; margin-left: 3.0em; font-weight: normal; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
Although the previous table can be used as support for all countries since it provides overall values from around the world, the following table specifically focuses on the Ice Energy System.
The same was built using the following additional sources:
</div>
<div style="text-align: justify; margin-left: 5em; font-weight: normal; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
<ol style="margin-top: 0; padding-left: 1.5em;">
<li>
xxxxxxxxxxxxxxxxxxxxxx. 
<br>
<a href="xxxxxxxxxxxxxxxxxxx" style="color:skyblue">xxxxxxxxxxxxxxxxxxxxxxi</a>
</li>
</ol>
</div>
</div>
<hr style="border: 0.5px solid skyblue;">
</div>

In [285]:
IS_fuel_technologies_match = {
    'AIR': [],
    'AMO': [],
    'BIO': [],
    'ELE': [],
    'GAS': [],
    'GEO': [],
    'HRD': [],
    'HYD': [],
    'LIG': [],
    'NUC': [],
    'OIL': [],
    'OTH': [],
    'PEA': [],
    'SUN': [],
    'THE': [],
    'WAT': [],
    'WHT': [],
    'WIN': [],
    'WST': []
}
index = ['ABHP','ALFC','ALKE','ASHP','BATS','BEVS','BSPG','CAES','COMC','COMCX','DMFC','GETH','GSHP','GTUR','GTURX','H2ST','HBBS','HDAM','HDAMC','HDLZ','HOBO','HOBOX','HPHS','HPHSC','HROR','HRORC','HYHP','ICEN','ICENX','MCFC','PAFC','P2BS','P2GS','P2HT','PEFC','PEME','PHOT','REFC','REHE','SCSP','SOFC','SOTH','SOXE','STUR','STURX','THMS','WAVE','WHEN','WTOF','WSHP','WTON']

# Create N/A DataFrame
IS_fuel_technologies_match_df = pd.DataFrame('N/A', index=index, columns=list(IS_fuel_technologies_match.keys()))

# Display the DataFrame
IS_fuel_technologies_match_df

,AIR,AMO,BIO,ELE,GAS,GEO,HRD,HYD,LIG,NUC,OIL,OTH,PEA,SUN,THE,WAT,WHT,WIN,WST
ABHP,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
ALFC,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
ALKE,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
ASHP,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
BATS,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
BEVS,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
BSPG,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
CAES,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
COMC,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
COMCX,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A


<div style="background-color: black;">
<hr style="border: 0.5px solid skyblue;">
<div style="text-align: justify; margin-left: 0.0em; font-weight: normal; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
    The Dispa-SET Technology - Fuel matching table for Ireland
    <br>
<div style="text-align: justify; margin-left: 3.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
Although the previous table can be used as support for all countries since it provides overall values from around the world, the following table specifically focuses on the Irish Energy System.
    The same was built using the following additional sources:
</div>
<div style="text-align: justify; margin-left: 5em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
    <ol style="margin-top: 0; padding-left: 1.5em;">
        <li>
            Department of the Environment, Climate and Communications. (2023, July 12). National Hydrogen Strategy. Government of Ireland.
            <br>
            <a href="https://assets.gov.ie/static/documents/national-hydrogen-strategy.pdf" style="color:skyblue">https://assets.gov.ie/static/documents/national-hydrogen-strategy.pdf</a>
        </li>
        <li>
            Environmental Protection Agency (EPA). (2024). Circular Economy and Waste Statistics Highlights Report 2022.
            <br>
            <a href="https://www.epa.ie/publications/monitoring--assessment/waste/national-waste-statistics/circular-economy-and-waste-statistics-highlights-report-2022.php" style="color:skyblue">https://www.epa.ie/publications/monitoring--assessment/waste/national-waste-statistics/circular-economy-and-waste-statistics-highlights-report-2022.php</a>
        </li>
        <li>
            EirGrid. (2024). Fuel Mix 2024 [Infographic].
            <br>
            <a href="https://cms.eirgrid.ie/sites/default/files/publications/Fuel-Mix-2024.png" style="color:skyblue">https://cms.eirgrid.ie/sites/default/files/publications/Fuel-Mix-2024.png</a>
        </li>
        <li>
            EirGrid. (n.d.). Generation data from the Smart Grid Dashboard. Retrieved October 10, 2025.
            <br>
            <a href="https://www.smartgriddashboard.com/all/generation/" style="color:skyblue">https://www.smartgriddashboard.com/all/generation/</a>
        </li>
        <li>
            SONI and EirGrid. (2024). Ten-Year Generation Capacity Statement 2023–2032.
            <br>
            <a href="https://cms.soni.ltd.uk/sites/default/files/media/documents/SONI-Generation-Capacity-Statement-2023-2032.pdf" style="color:skyblue">https://cms.soni.ltd.uk/sites/default/files/media/documents/SONI-Generation-Capacity-Statement-2023-2032.pdf</a>
        </li>
        <li>
            EirGrid. (2023). Annual Report 2022.
            <br>
            <a href="https://cms.eirgrid.ie/sites/default/files/publications/EirGrid-Annual-Report-2022.pdf" style="color:skyblue">https://cms.eirgrid.ie/sites/default/files/publications/EirGrid-Annual-Report-2022.pdf</a>
        </li>
        <li>
            ESB. (n.d.). Generation and trading. Retrieved October 10, 2025.
            <br>
            <a href="https://esb.ie/what-we-do/generation-and-trading" style="color:skyblue">https://esb.ie/what-we-do/generation-and-trading</a>
        </li>
        <li>
            International Energy Agency (IEA). (2025, January 24). Ireland. Retrieved October 10, 2025.
            <br>
            <a href="https://www.iea.org/countries/ireland" style="color:skyblue">https://www.iea.org/countries/ireland</a>
        </li>
        <li>
            Wind Energy Ireland. (n.d.). Wind energy reports (Monthly dashboard). Retrieved October 10, 2025.
            <br>
            <a href="https://windenergyireland.com/about-wind/more-resources/monthly-dashboard" style="color:skyblue">https://windenergyireland.com/about-wind/more-resources/monthly-dashboard</a>
        </li>
        <li>
            Sustainable Energy Authority of Ireland (SEAI). (2022). Annual Report 2021.
            <br>
            <a href="https://www.seai.ie/sites/default/files/publications/SEAI-Annual-Report-2021.pdf" style="color:skyblue">https://www.seai.ie/sites/default/files/publications/SEAI-Annual-Report-2021.pdf</a>
        </li>
        <li>
            Sustainable Energy Authority of Ireland (SEAI). (2023). Energy in Ireland 2022.
            <br>
            <a href="https://www.seai.ie/sites/default/files/publications/Energy-in-Ireland-2022.pdf" style="color:skyblue">https://www.seai.ie/sites/default/files/publications/Energy-in-Ireland-2022.pdf</a>
        </li>
        <li>
            Sustainable Energy Authority of Ireland (SEAI). (2024). Energy in Ireland 2024.
            <br>
            <a href="https://www.seai.ie/sites/default/files/publications/energy-in-ireland-2024.pdf" style="color:skyblue">https://www.seai.ie/sites/default/files/publications/energy-in-ireland-2024.pdf</a>
        </li>
        <li>
            Sustainable Energy Authority of Ireland (SEAI). (2025, August). First Look: Ireland's Energy Supply and Security of Supply in 2024.
            <br>
            <a href="https://www.seai.ie/sites/default/files/publications/First-Look-Energy-Supply-and-Security-of-Supply.pdf" style="color:skyblue">https://www.seai.ie/sites/default/files/publications/First-Look-Energy-Supply-and-Security-of-Supply.pdf</a>
        </li>
        <li>
            Sustainable Energy Authority of Ireland (SEAI). (2024). Energy in Ireland 2024 data and charts (Appendix 10) [Data set].
            <br>
            <a href="https://www.seai.ie/sites/default/files/2025-02/Energy-in-Ireland-2024-data-and-charts.xlsx" style="color:skyblue">https://www.seai.ie/sites/default/files/2025-02/Energy-in-Ireland-2024-data-and-charts.xlsx</a>
        </li>
        <li>
            Sustainable Energy Authority of Ireland (SEAI). (n.d.). Sustainable biomass fuels in Ireland. Retrieved October 10, 2025.
            <br>
            <a href="https://www.seai.ie/renewable-energy/bioenergy/biomass-in-ireland" style="color:skyblue">https://www.seai.ie/renewable-energy/bioenergy/biomass-in-ireland</a>
        </li>
        <li>
            Sustainable Energy Authority of Ireland (SEAI). (2005). Tidal & current energy resources in Ireland.
            <br>
            <a href="https://www.seai.ie/sites/default/files/publications/Tidal_Current_Energy_Resources_in_Ireland_Report.pdf" style="color:skyblue">https://www.seai.ie/sites/default/files/publications/Tidal_Current_Energy_Resources_in_Ireland_Report.pdf</a>
        </li>
        <li>
            Sustainable Energy Authority of Ireland (SEAI). (n.d.). Geothermal energy. Retrieved June 4, 2024.
            <br>
            <a href="https://www.seai.ie/technologies/geothermal-energy/" style="color:skyblue">https://www.seai.ie/technologies/geothermal-energy/</a>
        </li>
        <li>
            Sustainable Energy Authority of Ireland (SEAI). (2025). SEAI annual report 2024: English and Irish.
            <br>
            <a href="https://www.seai.ie/sites/default/files/publications/SEAI-Annual-Report-2024-English-and-Irish.pdf" style="color:skyblue">https://www.seai.ie/sites/default/files/publications/SEAI-Annual-Report-2024-English-and-Irish.pdf</a>
        </li>
        <li>
            Sustainable Energy Authority of Ireland (SEAI). (n.d.). Offshore Renewable Energy. Retrieved October 10, 2025.
            <br>
            <a href="https://www.seai.ie/renewable-energy/Offshore-Renewable-Energy" style="color:skyblue">https://www.seai.ie/renewable-energy/Offshore-Renewable-Energy</a>
        </li>
        <li>
            Sustainable Energy Authority of Ireland (SEAI). (2011). Smart Grids Roadmap.
            <br>
            <a href="https://www.seai.ie/sites/default/files/publications/Smartgrid-Roadmap.pdf" style="color:skyblue">https://www.seai.ie/sites/default/files/publications/Smartgrid-Roadmap.pdf</a>
        </li>
    </ol>
</div>
</div>
<hr style="border: 0.5px solid skyblue;">
</div>


In [286]:
IE_fuel_technologies_match = {
    'AIR': [0.95, 0, 0, 0.95, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0.9, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0.9, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    'AMO': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    'BIO': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0.15, 0, 0, 0, 0, 0, 0, 0.9, 0, 0, 0.15, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    'ELE': [0.05, 0, 1, 0.05, 1, 1, 0, 0.65, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0.1, 0, 0, 0, 0, 1, 1, 1, 0, 1, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0],
    'GAS': [0, 0, 0, 0, 0, 0, 0, 0.35, 0.9, 0.8, 0, 0, 0, 0.95, 0.85, 0, 0, 0, 0, 0.8, 0.65, 0.7, 0, 0, 0, 0, 0, 0.2, 0.25, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0.95, 0, 0, 0.45, 0.5, 0, 0, 0, 0, 0, 0],
    'GEO': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    'HRD': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    'HYD': [0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0.05, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    'LIG': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    'NUC': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    'OIL': [0, 0, 0, 0, 0, 0, 0, 0, 0.1, 0.05, 0, 0, 0, 0.05, 0.15, 0, 0.1, 0, 0, 0.05, 0.35, 0.3, 0, 0, 0, 0, 0, 0.8, 0.75, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0.1, 0.15, 0, 0, 0, 0, 0, 0],
    'OTH': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    'PEA': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    'SUN': [0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    'THE': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    'WAT': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0],
    'WHT': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0],
    'WIN': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1],
    'WST': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0.45, 0.35, 0, 0, 0, 0, 0, 0]
}
index = ['ABHP','ALFC','ALKE','ASHP','BATS','BEVS','BSPG','CAES','COMC','COMCX','DMFC','GETH','GSHP','GTUR','GTURX','H2ST','HBBS','HDAM','HDAMC','HDLZ','HOBO','HOBOX','HPHS','HPHSC','HROR','HRORC','HYHP','ICEN','ICENX','MCFC','PAFC','P2BS','P2GS','P2HT','PEFC','PEME','PHOT','REFC','REHE','SCSP','SOFC','SOTH','SOXE','STUR','STURX','THMS','WAVE','WHEN','WTOF','WSHP','WTON']

# Create DataFrame
IE_fuel_technologies_match_df = pd.DataFrame(IE_fuel_technologies_match, index=index)

# Display the DataFrame
IE_fuel_technologies_match_df

,AIR,AMO,BIO,ELE,GAS,GEO,HRD,HYD,LIG,NUC,OIL,OTH,PEA,SUN,THE,WAT,WHT,WIN,WST
ABHP,0.95,0,0.00,0.05,0.00,0,0,0.00,0,0,0.00,0,0,0,0,0,0,0,0.00
ALFC,0.00,0,0.00,0.00,0.00,0,0,1.00,0,0,0.00,0,0,0,0,0,0,0,0.00
ALKE,0.00,0,0.00,1.00,0.00,0,0,0.00,0,0,0.00,0,0,0,0,0,0,0,0.00
ASHP,0.95,0,0.00,0.05,0.00,0,0,0.00,0,0,0.00,0,0,0,0,0,0,0,0.00
BATS,0.00,0,0.00,1.00,0.00,0,0,0.00,0,0,0.00,0,0,0,0,0,0,0,0.00
BEVS,0.00,0,0.00,1.00,0.00,0,0,0.00,0,0,0.00,0,0,0,0,0,0,0,0.00
BSPG,0.00,0,0.00,0.00,0.00,0,0,0.00,0,0,0.00,0,0,1,0,0,0,0,0.00
CAES,0.00,0,0.00,0.65,0.35,0,0,0.00,0,0,0.00,0,0,0,0,0,0,0,0.00
COMC,0.00,0,0.00,0.00,0.90,0,0,0.00,0,0,0.10,0,0,0,0,0,0,0,0.00
COMCX,0.00,0,0.15,0.00,0.80,0,0,0.00,0,0,0.05,0,0,0,0,0,0,0,0.00


<div style="background-color: black;">
<hr style="border: 0.5px solid skyblue;">
<div style="text-align: justify; margin-left: 0.0em; font-weight: normal; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
The Dispa-SET Technology - Fuel matching table for Kosovo.
<div style="text-align: justify; margin-left: 3.0em; font-weight: normal; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
Although the previous table can be used as support for all countries since it provides overall values from around the world, the following table specifically focuses on the Koso Energy System.
The same was built using the following additional sources:
</div>
<div style="text-align: justify; margin-left: 5em; font-weight: normal; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
<ol style="margin-top: 0; padding-left: 1.5em;">
<li>
xxxxxxxxxxxxxxxxxxxxxx. 
<br>
<a href="xxxxxxxxxxxxxxxxxxx" style="color:skyblue">xxxxxxxxxxxxxxxxxxxxxxi</a>
</li>
</ol>
</div>
</div>
<hr style="border: 0.5px solid skyblue;">
</div>

In [287]:
XK_fuel_technologies_match = {
    'AIR': [],
    'AMO': [],
    'BIO': [],
    'ELE': [],
    'GAS': [],
    'GEO': [],
    'HRD': [],
    'HYD': [],
    'LIG': [],
    'NUC': [],
    'OIL': [],
    'OTH': [],
    'PEA': [],
    'SUN': [],
    'THE': [],
    'WAT': [],
    'WHT': [],
    'WIN': [],
    'WST': []
}
index = ['ABHP','ALFC','ALKE','ASHP','BATS','BEVS','BSPG','CAES','COMC','COMCX','DMFC','GETH','GSHP','GTUR','GTURX','H2ST','HBBS','HDAM','HDAMC','HDLZ','HOBO','HOBOX','HPHS','HPHSC','HROR','HRORC','HYHP','ICEN','ICENX','MCFC','PAFC','P2BS','P2GS','P2HT','PEFC','PEME','PHOT','REFC','REHE','SCSP','SOFC','SOTH','SOXE','STUR','STURX','THMS','WAVE','WHEN','WTOF','WSHP','WTON']

# Create N/A DataFrame
XK_fuel_technologies_match_df = pd.DataFrame('N/A', index=index, columns=list(XK_fuel_technologies_match.keys()))

# Display the DataFrame
XK_fuel_technologies_match_df

,AIR,AMO,BIO,ELE,GAS,GEO,HRD,HYD,LIG,NUC,OIL,OTH,PEA,SUN,THE,WAT,WHT,WIN,WST
ABHP,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
ALFC,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
ALKE,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
ASHP,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
BATS,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
BEVS,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
BSPG,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
CAES,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
COMC,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
COMCX,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A


<div style="background-color: black;">
<hr style="border: 0.5px solid skyblue;">
<div style="text-align: justify; margin-left: 0.0em; font-weight: normal; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
The Dispa-SET Technology - Fuel matching table for Latvia.
<div style="text-align: justify; margin-left: 3.0em; font-weight: normal; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
Although the previous table can be used as support for all countries since it provides overall values from around the world, the following table specifically focuses on the Latv Energy System.
The same was built using the following additional sources:
</div>
<div style="text-align: justify; margin-left: 5em; font-weight: normal; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
<ol style="margin-top: 0; padding-left: 1.5em;">
<li>
xxxxxxxxxxxxxxxxxxxxxx. 
<br>
<a href="xxxxxxxxxxxxxxxxxxx" style="color:skyblue">xxxxxxxxxxxxxxxxxxxxxxi</a>
</li>
</ol>
</div>
</div>
<hr style="border: 0.5px solid skyblue;">
</div>

In [288]:
LV_fuel_technologies_match = {
    'AIR': [],
    'AMO': [],
    'BIO': [],
    'ELE': [],
    'GAS': [],
    'GEO': [],
    'HRD': [],
    'HYD': [],
    'LIG': [],
    'NUC': [],
    'OIL': [],
    'OTH': [],
    'PEA': [],
    'SUN': [],
    'THE': [],
    'WAT': [],
    'WHT': [],
    'WIN': [],
    'WST': []
}
index = ['ABHP','ALFC','ALKE','ASHP','BATS','BEVS','BSPG','CAES','COMC','COMCX','DMFC','GETH','GSHP','GTUR','GTURX','H2ST','HBBS','HDAM','HDAMC','HDLZ','HOBO','HOBOX','HPHS','HPHSC','HROR','HRORC','HYHP','ICEN','ICENX','MCFC','PAFC','P2BS','P2GS','P2HT','PEFC','PEME','PHOT','REFC','REHE','SCSP','SOFC','SOTH','SOXE','STUR','STURX','THMS','WAVE','WHEN','WTOF','WSHP','WTON']

# Create N/A DataFrame
LV_fuel_technologies_match_df = pd.DataFrame('N/A', index=index, columns=list(LV_fuel_technologies_match.keys()))

# Display the DataFrame
LV_fuel_technologies_match_df

,AIR,AMO,BIO,ELE,GAS,GEO,HRD,HYD,LIG,NUC,OIL,OTH,PEA,SUN,THE,WAT,WHT,WIN,WST
ABHP,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
ALFC,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
ALKE,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
ASHP,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
BATS,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
BEVS,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
BSPG,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
CAES,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
COMC,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
COMCX,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A


<div style="background-color: black;">
<hr style="border: 0.5px solid skyblue;">
<div style="text-align: justify; margin-left: 0.0em; font-weight: normal; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
The Dispa-SET Technology - Fuel matching table for Lithuania.
<div style="text-align: justify; margin-left: 3.0em; font-weight: normal; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
Although the previous table can be used as support for all countries since it provides overall values from around the world, the following table specifically focuses on the Lith Energy System.
The same was built using the following additional sources:
</div>
<div style="text-align: justify; margin-left: 5em; font-weight: normal; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
<ol style="margin-top: 0; padding-left: 1.5em;">
<li>
xxxxxxxxxxxxxxxxxxxxxx. 
<br>
<a href="xxxxxxxxxxxxxxxxxxx" style="color:skyblue">xxxxxxxxxxxxxxxxxxxxxxi</a>
</li>
</ol>
</div>
</div>
<hr style="border: 0.5px solid skyblue;">
</div>

In [289]:
LT_fuel_technologies_match = {
    'AIR': [],
    'AMO': [],
    'BIO': [],
    'ELE': [],
    'GAS': [],
    'GEO': [],
    'HRD': [],
    'HYD': [],
    'LIG': [],
    'NUC': [],
    'OIL': [],
    'OTH': [],
    'PEA': [],
    'SUN': [],
    'THE': [],
    'WAT': [],
    'WHT': [],
    'WIN': [],
    'WST': []
}
index = ['ABHP','ALFC','ALKE','ASHP','BATS','BEVS','BSPG','CAES','COMC','COMCX','DMFC','GETH','GSHP','GTUR','GTURX','H2ST','HBBS','HDAM','HDAMC','HDLZ','HOBO','HOBOX','HPHS','HPHSC','HROR','HRORC','HYHP','ICEN','ICENX','MCFC','PAFC','P2BS','P2GS','P2HT','PEFC','PEME','PHOT','REFC','REHE','SCSP','SOFC','SOTH','SOXE','STUR','STURX','THMS','WAVE','WHEN','WTOF','WSHP','WTON']

# Create N/A DataFrame
LT_fuel_technologies_match_df = pd.DataFrame('N/A', index=index, columns=list(LT_fuel_technologies_match.keys()))

# Display the DataFrame
LT_fuel_technologies_match_df

,AIR,AMO,BIO,ELE,GAS,GEO,HRD,HYD,LIG,NUC,OIL,OTH,PEA,SUN,THE,WAT,WHT,WIN,WST
ABHP,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
ALFC,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
ALKE,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
ASHP,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
BATS,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
BEVS,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
BSPG,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
CAES,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
COMC,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
COMCX,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A


<div style="background-color: black;">
<hr style="border: 0.5px solid skyblue;">
<div style="text-align: justify; margin-left: 0.0em; font-weight: normal; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
The Dispa-SET Technology - Fuel matching table for Malta.
<div style="text-align: justify; margin-left: 3.0em; font-weight: normal; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
Although the previous table can be used as support for all countries since it provides overall values from around the world, the following table specifically focuses on the Malt Energy System.
The same was built using the following additional sources:
</div>
<div style="text-align: justify; margin-left: 5em; font-weight: normal; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
<ol style="margin-top: 0; padding-left: 1.5em;">
<li>
xxxxxxxxxxxxxxxxxxxxxx. 
<br>
<a href="xxxxxxxxxxxxxxxxxxx" style="color:skyblue">xxxxxxxxxxxxxxxxxxxxxxi</a>
</li>
</ol>
</div>
</div>
<hr style="border: 0.5px solid skyblue;">
</div>

In [290]:
MT_fuel_technologies_match = {
    'AIR': [],
    'AMO': [],
    'BIO': [],
    'ELE': [],
    'GAS': [],
    'GEO': [],
    'HRD': [],
    'HYD': [],
    'LIG': [],
    'NUC': [],
    'OIL': [],
    'OTH': [],
    'PEA': [],
    'SUN': [],
    'THE': [],
    'WAT': [],
    'WHT': [],
    'WIN': [],
    'WST': []
}
index = ['ABHP','ALFC','ALKE','ASHP','BATS','BEVS','BSPG','CAES','COMC','COMCX','DMFC','GETH','GSHP','GTUR','GTURX','H2ST','HBBS','HDAM','HDAMC','HDLZ','HOBO','HOBOX','HPHS','HPHSC','HROR','HRORC','HYHP','ICEN','ICENX','MCFC','PAFC','P2BS','P2GS','P2HT','PEFC','PEME','PHOT','REFC','REHE','SCSP','SOFC','SOTH','SOXE','STUR','STURX','THMS','WAVE','WHEN','WTOF','WSHP','WTON']

# Create N/A DataFrame
MT_fuel_technologies_match_df = pd.DataFrame('N/A', index=index, columns=list(MT_fuel_technologies_match.keys()))

# Display the DataFrame
MT_fuel_technologies_match_df

,AIR,AMO,BIO,ELE,GAS,GEO,HRD,HYD,LIG,NUC,OIL,OTH,PEA,SUN,THE,WAT,WHT,WIN,WST
ABHP,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
ALFC,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
ALKE,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
ASHP,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
BATS,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
BEVS,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
BSPG,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
CAES,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
COMC,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
COMCX,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A


<div style="background-color: black;">
<hr style="border: 0.5px solid skyblue;">
<div style="text-align: justify; margin-left: 0.0em; font-weight: normal; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
The Dispa-SET Technology - Fuel matching table for Moldova.
<div style="text-align: justify; margin-left: 3.0em; font-weight: normal; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
Although the previous table can be used as support for all countries since it provides overall values from around the world, the following table specifically focuses on the Mold Energy System.
The same was built using the following additional sources:
</div>
<div style="text-align: justify; margin-left: 5em; font-weight: normal; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
<ol style="margin-top: 0; padding-left: 1.5em;">
<li>
xxxxxxxxxxxxxxxxxxxxxx. 
<br>
<a href="xxxxxxxxxxxxxxxxxxx" style="color:skyblue">xxxxxxxxxxxxxxxxxxxxxxi</a>
</li>
</ol>
</div>
</div>
<hr style="border: 0.5px solid skyblue;">
</div>

In [291]:
MD_fuel_technologies_match = {
    'AIR': [],
    'AMO': [],
    'BIO': [],
    'ELE': [],
    'GAS': [],
    'GEO': [],
    'HRD': [],
    'HYD': [],
    'LIG': [],
    'NUC': [],
    'OIL': [],
    'OTH': [],
    'PEA': [],
    'SUN': [],
    'THE': [],
    'WAT': [],
    'WHT': [],
    'WIN': [],
    'WST': []
}
index = ['ABHP','ALFC','ALKE','ASHP','BATS','BEVS','BSPG','CAES','COMC','COMCX','DMFC','GETH','GSHP','GTUR','GTURX','H2ST','HBBS','HDAM','HDAMC','HDLZ','HOBO','HOBOX','HPHS','HPHSC','HROR','HRORC','HYHP','ICEN','ICENX','MCFC','PAFC','P2BS','P2GS','P2HT','PEFC','PEME','PHOT','REFC','REHE','SCSP','SOFC','SOTH','SOXE','STUR','STURX','THMS','WAVE','WHEN','WTOF','WSHP','WTON']

# Create N/A DataFrame
MD_fuel_technologies_match_df = pd.DataFrame('N/A', index=index, columns=list(MD_fuel_technologies_match.keys()))

# Display the DataFrame
MD_fuel_technologies_match_df

,AIR,AMO,BIO,ELE,GAS,GEO,HRD,HYD,LIG,NUC,OIL,OTH,PEA,SUN,THE,WAT,WHT,WIN,WST
ABHP,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
ALFC,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
ALKE,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
ASHP,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
BATS,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
BEVS,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
BSPG,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
CAES,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
COMC,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
COMCX,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A


<div style="background-color: black;">
<hr style="border: 0.5px solid skyblue;">
<div style="text-align: justify; margin-left: 0.0em; font-weight: normal; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
The Dispa-SET Technology - Fuel matching table for Montenegro.
<div style="text-align: justify; margin-left: 3.0em; font-weight: normal; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
Although the previous table can be used as support for all countries since it provides overall values from around the world, the following table specifically focuses on the Mont Energy System.
The same was built using the following additional sources:
</div>
<div style="text-align: justify; margin-left: 5em; font-weight: normal; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
<ol style="margin-top: 0; padding-left: 1.5em;">
<li>
xxxxxxxxxxxxxxxxxxxxxx. 
<br>
<a href="xxxxxxxxxxxxxxxxxxx" style="color:skyblue">xxxxxxxxxxxxxxxxxxxxxxi</a>
</li>
</ol>
</div>
</div>
<hr style="border: 0.5px solid skyblue;">
</div>

In [292]:
ME_fuel_technologies_match = {
    'AIR': [],
    'AMO': [],
    'BIO': [],
    'ELE': [],
    'GAS': [],
    'GEO': [],
    'HRD': [],
    'HYD': [],
    'LIG': [],
    'NUC': [],
    'OIL': [],
    'OTH': [],
    'PEA': [],
    'SUN': [],
    'THE': [],
    'WAT': [],
    'WHT': [],
    'WIN': [],
    'WST': []
}
index = ['ABHP','ALFC','ALKE','ASHP','BATS','BEVS','BSPG','CAES','COMC','COMCX','DMFC','GETH','GSHP','GTUR','GTURX','H2ST','HBBS','HDAM','HDAMC','HDLZ','HOBO','HOBOX','HPHS','HPHSC','HROR','HRORC','HYHP','ICEN','ICENX','MCFC','PAFC','P2BS','P2GS','P2HT','PEFC','PEME','PHOT','REFC','REHE','SCSP','SOFC','SOTH','SOXE','STUR','STURX','THMS','WAVE','WHEN','WTOF','WSHP','WTON']

# Create N/A DataFrame
ME_fuel_technologies_match_df = pd.DataFrame('N/A', index=index, columns=list(ME_fuel_technologies_match.keys()))

# Display the DataFrame
ME_fuel_technologies_match_df

,AIR,AMO,BIO,ELE,GAS,GEO,HRD,HYD,LIG,NUC,OIL,OTH,PEA,SUN,THE,WAT,WHT,WIN,WST
ABHP,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
ALFC,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
ALKE,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
ASHP,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
BATS,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
BEVS,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
BSPG,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
CAES,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
COMC,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
COMCX,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A


<div style="background-color: black;">
<hr style="border: 0.5px solid skyblue;">
<div style="text-align: justify; margin-left: 0.0em; font-weight: normal; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
    The Dispa-SET Technology - Fuel matching table for Netherlands.
    <br>
<div style="text-align: justify; margin-left: 3.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
Although the previous table can be used as support for all countries since it provides overall values from around the world, the following table specifically focuses on the Netherland Energy System.
    The same was built using the following additional sources:
</div>
<div style="text-align: justify; margin-left: 5em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
    <ol style="margin-top: 0; padding-left: 1.5em;">
        <li>
            Centraal Bureau voor de Statistiek (CBS). (2023). Energiebalans; aanbod, omzetting en verbruik [Energy balance; supply, transformation and consumption].
            <br>
            <a href="https://www.cbs.nl/en-gb/figures/detail/83109ENG" style="color:skyblue">https://www.cbs.nl/en-gb/figures/detail/83109ENG</a>
        </li>
        <li>
            Netherlands Enterprise Agency (RVO). (2023). Energy and climate: Figures and reports.
            <br>
            <a href="https://english.rvo.nl/topics/energy-agreement/energy-and-climate-reports" style="color:skyblue">https://english.rvo.nl/topics/energy-agreement/energy-and-climate-reports</a>
        </li>
        <li>
            International Energy Agency. (2023). Netherlands 2023: Energy Policy Review. IEA.
            <br>
            <a href="https://www.iea.org/reports/the-netherlands-2024" style="color:skyblue">https://www.iea.org/reports/the-netherlands-2024</a>
        </li>
        <li>
            Eurostat. (2024). Energy database. European Commission.
            <br>
            <a href="https://ec.europa.eu/eurostat/web/energy/" style="color:skyblue">https://ec.europa.eu/eurostat/web/energy/</a>
        </li>
    </ol>
</div>
</div>
<hr style="border: 0.5px solid skyblue;">
</div>

In [293]:
NL_fuel_technologies_match = {
    'AIR': [0.05, 0, 0, 0.95, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    'AMO': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0.2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    'BIO': [0, 0, 0, 0, 0, 0, 0, 0, 0.1, 0.2, 0, 0, 0, 0.05, 0.1, 0, 1, 0, 0, 0, 0.05, 0.15, 0, 0, 0, 0, 0.1, 0.05, 0.1, 0.1, 0.1, 0, 0, 0, 0.1, 0, 0, 0, 0, 0, 0.1, 0, 0, 0.05, 0.05, 0, 0, 0, 0, 0, 0],
    'ELE': [0, 1, 1, 0.05, 1, 1, 0, 1, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0.9, 0, 0, 0, 0, 0.8, 1, 1, 0.05, 1, 0, 1, 1, 0, 0.05, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0],
    'GAS': [0, 0, 0, 0, 0, 0, 0, 0, 0.85, 0.75, 0, 0, 0, 0.85, 0.75, 0, 0, 0, 0, 0, 0.85, 0.65, 0, 0, 0, 0, 0, 0.5, 0.6, 0.85, 0.85, 0, 0, 0, 0.8, 0, 0, 0, 0, 0, 0.8, 0, 0, 0.85, 0.85, 0, 0, 0, 0, 0, 0],
    'GEO': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    'HRD': [0, 0, 0, 0, 0, 0, 0, 0, 0.05, 0.05, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0.05, 0.05, 0, 0, 0, 0, 0, 0.1, 0.1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0.05, 0.05, 0, 0, 0, 0, 0, 0],
    'HYD': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    'LIG': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0.05, 0.05, 0, 0, 0, 0, 0, 0],
    'NUC': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    'OIL': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0.1, 0.15, 0, 0, 0, 0, 0, 0.05, 0.1, 0, 0, 0, 0, 0, 0.35, 0.2, 0.05, 0.05, 0, 0, 0, 0.05, 0, 0, 0, 0, 0, 0.05, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    'OTH': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    'PEA': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    'SUN': [0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    'THE': [0.95, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0],
    'WAT': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 0, 0, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0],
    'WHT': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0],
    'WIN': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1],
    'WST': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0.05, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
}

index = ['ABHP','ALFC','ALKE','ASHP','BATS','BEVS','BSPG','CAES','COMC','COMCX','DMFC','GETH','GSHP','GTUR','GTURX','H2ST','HBBS','HDAM','HDAMC','HDLZ','HOBO','HOBOX','HPHS','HPHSC','HROR','HRORC','HYHP','ICEN','ICENX','MCFC','PAFC','P2BS','P2GS','P2HT','PEFC','PEME','PHOT','REFC','REHE','SCSP','SOFC','SOTH','SOXE','STUR','STURX','THMS','WAVE','WHEN','WTOF','WSHP','WTON']

# Create DataFrame
NL_fuel_technologies_match_df = pd.DataFrame(NL_fuel_technologies_match, index=index)

# Display the DataFrame
NL_fuel_technologies_match_df

,AIR,AMO,BIO,ELE,GAS,GEO,HRD,HYD,LIG,NUC,OIL,OTH,PEA,SUN,THE,WAT,WHT,WIN,WST
ABHP,0.05,0.0,0.00,0.00,0.00,0,0.00,0,0.00,0,0.00,0,0,0,0.95,0,0,0,0.00
ALFC,0.00,0.0,0.00,1.00,0.00,0,0.00,0,0.00,0,0.00,0,0,0,0.00,0,0,0,0.00
ALKE,0.00,0.0,0.00,1.00,0.00,0,0.00,0,0.00,0,0.00,0,0,0,0.00,0,0,0,0.00
ASHP,0.95,0.0,0.00,0.05,0.00,0,0.00,0,0.00,0,0.00,0,0,0,0.00,0,0,0,0.00
BATS,0.00,0.0,0.00,1.00,0.00,0,0.00,0,0.00,0,0.00,0,0,0,0.00,0,0,0,0.00
BEVS,0.00,0.0,0.00,1.00,0.00,0,0.00,0,0.00,0,0.00,0,0,0,0.00,0,0,0,0.00
BSPG,0.00,0.0,0.00,0.00,0.00,0,0.00,0,0.00,0,0.00,0,0,1,0.00,0,0,0,0.00
CAES,0.00,0.0,0.00,1.00,0.00,0,0.00,0,0.00,0,0.00,0,0,0,0.00,0,0,0,0.00
COMC,0.00,0.0,0.10,0.00,0.85,0,0.05,0,0.00,0,0.00,0,0,0,0.00,0,0,0,0.00
COMCX,0.00,0.0,0.20,0.00,0.75,0,0.05,0,0.00,0,0.00,0,0,0,0.00,0,0,0,0.00


<div style="background-color: black;">
<hr style="border: 0.5px solid skyblue;">
<div style="text-align: justify; margin-left: 0.0em; font-weight: normal; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
The Dispa-SET Technology - Fuel matching table for North Macedonia.
<div style="text-align: justify; margin-left: 3.0em; font-weight: normal; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
Although the previous table can be used as support for all countries since it provides overall values from around the world, the following table specifically focuses on the Maced Energy System.
The same was built using the following additional sources:
</div>
<div style="text-align: justify; margin-left: 5em; font-weight: normal; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
<ol style="margin-top: 0; padding-left: 1.5em;">
<li>
xxxxxxxxxxxxxxxxxxxxxx. 
<br>
<a href="xxxxxxxxxxxxxxxxxxx" style="color:skyblue">xxxxxxxxxxxxxxxxxxxxxxi</a>
</li>
</ol>
</div>
</div>
<hr style="border: 0.5px solid skyblue;">
</div>

In [294]:
MK_fuel_technologies_match = {
    'AIR': [],
    'AMO': [],
    'BIO': [],
    'ELE': [],
    'GAS': [],
    'GEO': [],
    'HRD': [],
    'HYD': [],
    'LIG': [],
    'NUC': [],
    'OIL': [],
    'OTH': [],
    'PEA': [],
    'SUN': [],
    'THE': [],
    'WAT': [],
    'WHT': [],
    'WIN': [],
    'WST': []
}
index = ['ABHP','ALFC','ALKE','ASHP','BATS','BEVS','BSPG','CAES','COMC','COMCX','DMFC','GETH','GSHP','GTUR','GTURX','H2ST','HBBS','HDAM','HDAMC','HDLZ','HOBO','HOBOX','HPHS','HPHSC','HROR','HRORC','HYHP','ICEN','ICENX','MCFC','PAFC','P2BS','P2GS','P2HT','PEFC','PEME','PHOT','REFC','REHE','SCSP','SOFC','SOTH','SOXE','STUR','STURX','THMS','WAVE','WHEN','WTOF','WSHP','WTON']

# Create N/A DataFrame
MK_fuel_technologies_match_df = pd.DataFrame('N/A', index=index, columns=list(MK_fuel_technologies_match.keys()))

# Display the DataFrame
MK_fuel_technologies_match_df

,AIR,AMO,BIO,ELE,GAS,GEO,HRD,HYD,LIG,NUC,OIL,OTH,PEA,SUN,THE,WAT,WHT,WIN,WST
ABHP,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
ALFC,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
ALKE,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
ASHP,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
BATS,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
BEVS,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
BSPG,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
CAES,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
COMC,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
COMCX,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A


<div style="background-color: black;">
<hr style="border: 0.5px solid skyblue;">
<div style="text-align: justify; margin-left: 0.0em; font-weight: normal; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
    The Dispa-SET Technology - Fuel matching table for Norway.
    <br>
<div style="text-align: justify; margin-left: 3.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
Although the previous table can be used as support for all countries since it provides overall values from around the world, the following table specifically focuses on the Norwegian Energy System.
    The same was built using the following additional sources:
</div>
<div style="text-align: justify; margin-left: 5em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
    <ol style="margin-top: 0; padding-left: 1.5em;">
        <li>
            Statistics Norway. (2023). Energy balance and electricity.
            <br>
            <a href="https://www.ssb.no/en/energi-og-industri/statistikker/energibalanse" style="color:skyblue">https://www.ssb.no/en/energi-og-industri/statistikker/energibalanse</a>
        </li>
        <li>
            Norges vassdrags- og energidirektorat. (2023). Energy data and statistics.
            <br>
            <a href="https://www.nve.no/search/?term=energy%20data%20and%20statistics/" style="color:skyblue">https://www.nve.no/search/?term=energy%20data%20and%20statistics//</a>
        </li>
        <li>
            International Energy Agency. (2023). Norway 2023: Energy Policy Review. IEA.
            <br>
            <a href="https://www.iea.org/reports/norway-2022" style="color:skyblue">https://www.iea.org/reports/norway-2022</a>
        </li>
        <li>
            The Norwegian Ministry of Petroleum and Energy. (2021). Long-term perspectives for the Norwegian energy sector.
            <br>
            <a href="https://www.regjeringen.no/en/search/id86008/?isfilteropen=True&term=long-term-perspectives-for-the-norwegian-energy-sector" style="color:skyblue">https://www.regjeringen.no/en/search/id86008/?isfilteropen=True&term=long-term-perspectives-for-the-norwegian-energy-sector/</a>
        </li>
    </ol>
</div>
</div>
<hr style="border: 0.5px solid skyblue;">
</div>

In [295]:
NO_fuel_technologies_match = {
    'AIR': [0.95, 0, 0, 0.95, 0, 0, 0, 0, 0, 0, 0, 0, 0.95, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0.9, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0.95, 0],
    'AMO': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    'BIO': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0.4, 0, 0, 0, 0, 0.35, 0, 1, 0, 0, 0, 0.6, 0.5, 0, 0, 0, 0, 0, 0.25, 0.35, 0, 0, 0.35, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0.2, 0.4, 0, 0, 0, 0, 0, 0],
    'ELE': [0.05, 0, 1, 0.05, 1, 1, 0, 0.65, 0, 0, 0, 0, 0.05, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0.1, 0, 0, 0, 0, 0.5, 1, 1, 0, 1, 0, 0, 1, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0.05, 0],
    'GAS': [0, 0, 0, 0, 0, 0, 0, 0.35, 0.9, 0.5, 0, 0, 0, 0.85, 0.55, 0, 0, 0, 0, 0, 0.25, 0.4, 0, 0, 0, 0, 0, 0.2, 0.45, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0.6, 0.5, 0, 0, 0, 0, 0, 0],
    'GEO': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    'HRD': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    'HYD': [0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0.15, 0.1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0.15, 0.2, 0, 0, 0.15, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0.1, 0, 0, 0, 0, 0, 0],
    'LIG': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    'NUC': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    'OIL': [0, 0, 0, 0, 0, 0, 0, 0, 0.1, 0.1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0.15, 0.1, 0, 0, 0, 0, 0, 0.4, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0.2, 0, 0, 0, 0, 0, 0, 0],
    'OTH': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    'PEA': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    'SUN': [0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    'THE': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0],
    'WAT': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0],
    'WHT': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    'WIN': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1],
    'WST': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
}
index = ['ABHP','ALFC','ALKE','ASHP','BATS','BEVS','BSPG','CAES','COMC','COMCX','DMFC','GETH','GSHP','GTUR','GTURX','H2ST','HBBS','HDAM','HDAMC','HDLZ','HOBO','HOBOX','HPHS','HPHSC','HROR','HRORC','HYHP','ICEN','ICENX','MCFC','PAFC','P2BS','P2GS','P2HT','PEFC','PEME','PHOT','REFC','REHE','SCSP','SOFC','SOTH','SOXE','STUR','STURX','THMS','WAVE','WHEN','WTOF','WSHP','WTON']

# Create N/A DataFrame
NO_fuel_technologies_match_df = pd.DataFrame(NO_fuel_technologies_match, index=index)

# Display the DataFrame
NO_fuel_technologies_match_df

,AIR,AMO,BIO,ELE,GAS,GEO,HRD,HYD,LIG,NUC,OIL,OTH,PEA,SUN,THE,WAT,WHT,WIN,WST
ABHP,0.95,0,0.00,0.05,0.00,0,0,0.00,0,0,0.00,0,0,0,0,0,0,0,0
ALFC,0.00,0,0.00,0.00,0.00,0,0,1.00,0,0,0.00,0,0,0,0,0,0,0,0
ALKE,0.00,0,0.00,1.00,0.00,0,0,0.00,0,0,0.00,0,0,0,0,0,0,0,0
ASHP,0.95,0,0.00,0.05,0.00,0,0,0.00,0,0,0.00,0,0,0,0,0,0,0,0
BATS,0.00,0,0.00,1.00,0.00,0,0,0.00,0,0,0.00,0,0,0,0,0,0,0,0
BEVS,0.00,0,0.00,1.00,0.00,0,0,0.00,0,0,0.00,0,0,0,0,0,0,0,0
BSPG,0.00,0,0.00,0.00,0.00,0,0,0.00,0,0,0.00,0,0,1,0,0,0,0,0
CAES,0.00,0,0.00,0.65,0.35,0,0,0.00,0,0,0.00,0,0,0,0,0,0,0,0
COMC,0.00,0,0.00,0.00,0.90,0,0,0.00,0,0,0.10,0,0,0,0,0,0,0,0
COMCX,0.00,0,0.40,0.00,0.50,0,0,0.00,0,0,0.10,0,0,0,0,0,0,0,0


<div style="background-color: black;">
<hr style="border: 0.5px solid skyblue;">
<div style="text-align: justify; margin-left: 0.0em; font-weight: normal; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
The Dispa-SET Technology - Fuel matching table for Portugal.
<div style="text-align: justify; margin-left: 3.0em; font-weight: normal; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
Although the previous table can be used as support for all countries since it provides overall values from around the world, the following table specifically focuses on the Portu Energy System.
The same was built using the following additional sources:
</div>
<div style="text-align: justify; margin-left: 5em; font-weight: normal; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
<ol style="margin-top: 0; padding-left: 1.5em;">
<li>
xxxxxxxxxxxxxxxxxxxxxx. 
<br>
<a href="xxxxxxxxxxxxxxxxxxx" style="color:skyblue">xxxxxxxxxxxxxxxxxxxxxxi</a>
</li>
</ol>
</div>
</div>
<hr style="border: 0.5px solid skyblue;">
</div>

In [296]:
PT_fuel_technologies_match = {
    'AIR': [],
    'AMO': [],
    'BIO': [],
    'ELE': [],
    'GAS': [],
    'GEO': [],
    'HRD': [],
    'HYD': [],
    'LIG': [],
    'NUC': [],
    'OIL': [],
    'OTH': [],
    'PEA': [],
    'SUN': [],
    'THE': [],
    'WAT': [],
    'WHT': [],
    'WIN': [],
    'WST': []
}
index = ['ABHP','ALFC','ALKE','ASHP','BATS','BEVS','BSPG','CAES','COMC','COMCX','DMFC','GETH','GSHP','GTUR','GTURX','H2ST','HBBS','HDAM','HDAMC','HDLZ','HOBO','HOBOX','HPHS','HPHSC','HROR','HRORC','HYHP','ICEN','ICENX','MCFC','PAFC','P2BS','P2GS','P2HT','PEFC','PEME','PHOT','REFC','REHE','SCSP','SOFC','SOTH','SOXE','STUR','STURX','THMS','WAVE','WHEN','WTOF','WSHP','WTON']

# Create N/A DataFrame
PT_fuel_technologies_match_df = pd.DataFrame('N/A', index=index, columns=list(PT_fuel_technologies_match.keys()))

# Display the DataFrame
PT_fuel_technologies_match_df

,AIR,AMO,BIO,ELE,GAS,GEO,HRD,HYD,LIG,NUC,OIL,OTH,PEA,SUN,THE,WAT,WHT,WIN,WST
ABHP,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
ALFC,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
ALKE,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
ASHP,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
BATS,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
BEVS,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
BSPG,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
CAES,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
COMC,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
COMCX,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A


<div style="background-color: black;">
<hr style="border: 0.5px solid skyblue;">
<div style="text-align: justify; margin-left: 0.0em; font-weight: normal; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
The Dispa-SET Technology - Fuel matching table for Romania.
<div style="text-align: justify; margin-left: 3.0em; font-weight: normal; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
Although the previous table can be used as support for all countries since it provides overall values from around the world, the following table specifically focuses on the Roma Energy System.
The same was built using the following additional sources:
</div>
<div style="text-align: justify; margin-left: 5em; font-weight: normal; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
<ol style="margin-top: 0; padding-left: 1.5em;">
<li>
xxxxxxxxxxxxxxxxxxxxxx. 
<br>
<a href="xxxxxxxxxxxxxxxxxxx" style="color:skyblue">xxxxxxxxxxxxxxxxxxxxxxi</a>
</li>
</ol>
</div>
</div>
<hr style="border: 0.5px solid skyblue;">
</div>

In [297]:
RO_fuel_technologies_match = {
    'AIR': [],
    'AMO': [],
    'BIO': [],
    'ELE': [],
    'GAS': [],
    'GEO': [],
    'HRD': [],
    'HYD': [],
    'LIG': [],
    'NUC': [],
    'OIL': [],
    'OTH': [],
    'PEA': [],
    'SUN': [],
    'THE': [],
    'WAT': [],
    'WHT': [],
    'WIN': [],
    'WST': []
}
index = ['ABHP','ALFC','ALKE','ASHP','BATS','BEVS','BSPG','CAES','COMC','COMCX','DMFC','GETH','GSHP','GTUR','GTURX','H2ST','HBBS','HDAM','HDAMC','HDLZ','HOBO','HOBOX','HPHS','HPHSC','HROR','HRORC','HYHP','ICEN','ICENX','MCFC','PAFC','P2BS','P2GS','P2HT','PEFC','PEME','PHOT','REFC','REHE','SCSP','SOFC','SOTH','SOXE','STUR','STURX','THMS','WAVE','WHEN','WTOF','WSHP','WTON']

# Create N/A DataFrame
RO_fuel_technologies_match_df = pd.DataFrame('N/A', index=index, columns=list(RO_fuel_technologies_match.keys()))

# Display the DataFrame
RO_fuel_technologies_match_df

,AIR,AMO,BIO,ELE,GAS,GEO,HRD,HYD,LIG,NUC,OIL,OTH,PEA,SUN,THE,WAT,WHT,WIN,WST
ABHP,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
ALFC,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
ALKE,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
ASHP,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
BATS,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
BEVS,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
BSPG,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
CAES,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
COMC,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
COMCX,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A


<div style="background-color: black;">
<hr style="border: 0.5px solid skyblue;">
<div style="text-align: justify; margin-left: 0.0em; font-weight: normal; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
The Dispa-SET Technology - Fuel matching table for Russian Federation.
<div style="text-align: justify; margin-left: 3.0em; font-weight: normal; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
Although the previous table can be used as support for all countries since it provides overall values from around the world, the following table specifically focuses on the Russ Energy System.
The same was built using the following additional sources:
</div>
<div style="text-align: justify; margin-left: 5em; font-weight: normal; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
<ol style="margin-top: 0; padding-left: 1.5em;">
<li>
xxxxxxxxxxxxxxxxxxxxxx. 
<br>
<a href="xxxxxxxxxxxxxxxxxxx" style="color:skyblue">xxxxxxxxxxxxxxxxxxxxxxi</a>
</li>
</ol>
</div>
</div>
<hr style="border: 0.5px solid skyblue;">
</div>

In [298]:
RU_fuel_technologies_match = {
    'AIR': [],
    'AMO': [],
    'BIO': [],
    'ELE': [],
    'GAS': [],
    'GEO': [],
    'HRD': [],
    'HYD': [],
    'LIG': [],
    'NUC': [],
    'OIL': [],
    'OTH': [],
    'PEA': [],
    'SUN': [],
    'THE': [],
    'WAT': [],
    'WHT': [],
    'WIN': [],
    'WST': []
}
index = ['ABHP','ALFC','ALKE','ASHP','BATS','BEVS','BSPG','CAES','COMC','COMCX','DMFC','GETH','GSHP','GTUR','GTURX','H2ST','HBBS','HDAM','HDAMC','HDLZ','HOBO','HOBOX','HPHS','HPHSC','HROR','HRORC','HYHP','ICEN','ICENX','MCFC','PAFC','P2BS','P2GS','P2HT','PEFC','PEME','PHOT','REFC','REHE','SCSP','SOFC','SOTH','SOXE','STUR','STURX','THMS','WAVE','WHEN','WTOF','WSHP','WTON']

# Create N/A DataFrame
RU_fuel_technologies_match_df = pd.DataFrame('N/A', index=index, columns=list(RU_fuel_technologies_match.keys()))

# Display the DataFrame
RU_fuel_technologies_match_df

,AIR,AMO,BIO,ELE,GAS,GEO,HRD,HYD,LIG,NUC,OIL,OTH,PEA,SUN,THE,WAT,WHT,WIN,WST
ABHP,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
ALFC,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
ALKE,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
ASHP,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
BATS,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
BEVS,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
BSPG,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
CAES,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
COMC,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
COMCX,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A


<div style="background-color: black;">
<hr style="border: 0.5px solid skyblue;">
<div style="text-align: justify; margin-left: 0.0em; font-weight: normal; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
The Dispa-SET Technology - Fuel matching table for Serbia.
<div style="text-align: justify; margin-left: 3.0em; font-weight: normal; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
Although the previous table can be used as support for all countries since it provides overall values from around the world, the following table specifically focuses on the Serb Energy System.
The same was built using the following additional sources:
</div>
<div style="text-align: justify; margin-left: 5em; font-weight: normal; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
<ol style="margin-top: 0; padding-left: 1.5em;">
<li>
xxxxxxxxxxxxxxxxxxxxxx. 
<br>
<a href="xxxxxxxxxxxxxxxxxxx" style="color:skyblue">xxxxxxxxxxxxxxxxxxxxxxi</a>
</li>
</ol>
</div>
</div>
<hr style="border: 0.5px solid skyblue;">
</div>

In [299]:
RS_fuel_technologies_match = {
    'AIR': [],
    'AMO': [],
    'BIO': [],
    'ELE': [],
    'GAS': [],
    'GEO': [],
    'HRD': [],
    'HYD': [],
    'LIG': [],
    'NUC': [],
    'OIL': [],
    'OTH': [],
    'PEA': [],
    'SUN': [],
    'THE': [],
    'WAT': [],
    'WHT': [],
    'WIN': [],
    'WST': []
}
index = ['ABHP','ALFC','ALKE','ASHP','BATS','BEVS','BSPG','CAES','COMC','COMCX','DMFC','GETH','GSHP','GTUR','GTURX','H2ST','HBBS','HDAM','HDAMC','HDLZ','HOBO','HOBOX','HPHS','HPHSC','HROR','HRORC','HYHP','ICEN','ICENX','MCFC','PAFC','P2BS','P2GS','P2HT','PEFC','PEME','PHOT','REFC','REHE','SCSP','SOFC','SOTH','SOXE','STUR','STURX','THMS','WAVE','WHEN','WTOF','WSHP','WTON']

# Create N/A DataFrame
RS_fuel_technologies_match_df = pd.DataFrame('N/A', index=index, columns=list(RS_fuel_technologies_match.keys()))

# Display the DataFrame
RS_fuel_technologies_match_df

,AIR,AMO,BIO,ELE,GAS,GEO,HRD,HYD,LIG,NUC,OIL,OTH,PEA,SUN,THE,WAT,WHT,WIN,WST
ABHP,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
ALFC,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
ALKE,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
ASHP,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
BATS,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
BEVS,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
BSPG,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
CAES,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
COMC,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
COMCX,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A


<div style="background-color: black;">
<hr style="border: 0.5px solid skyblue;">
<div style="text-align: justify; margin-left: 0.0em; font-weight: normal; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
The Dispa-SET Technology - Fuel matching table for Slovakia.
<div style="text-align: justify; margin-left: 3.0em; font-weight: normal; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
Although the previous table can be used as support for all countries since it provides overall values from around the world, the following table specifically focuses on the Slova Energy System.
The same was built using the following additional sources:
</div>
<div style="text-align: justify; margin-left: 5em; font-weight: normal; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
<ol style="margin-top: 0; padding-left: 1.5em;">
<li>
xxxxxxxxxxxxxxxxxxxxxx. 
<br>
<a href="xxxxxxxxxxxxxxxxxxx" style="color:skyblue">xxxxxxxxxxxxxxxxxxxxxxi</a>
</li>
</ol>
</div>
</div>
<hr style="border: 0.5px solid skyblue;">
</div>

In [300]:
SK_fuel_technologies_match = {
    'AIR': [],
    'AMO': [],
    'BIO': [],
    'ELE': [],
    'GAS': [],
    'GEO': [],
    'HRD': [],
    'HYD': [],
    'LIG': [],
    'NUC': [],
    'OIL': [],
    'OTH': [],
    'PEA': [],
    'SUN': [],
    'THE': [],
    'WAT': [],
    'WHT': [],
    'WIN': [],
    'WST': []
}
index = ['ABHP','ALFC','ALKE','ASHP','BATS','BEVS','BSPG','CAES','COMC','COMCX','DMFC','GETH','GSHP','GTUR','GTURX','H2ST','HBBS','HDAM','HDAMC','HDLZ','HOBO','HOBOX','HPHS','HPHSC','HROR','HRORC','HYHP','ICEN','ICENX','MCFC','PAFC','P2BS','P2GS','P2HT','PEFC','PEME','PHOT','REFC','REHE','SCSP','SOFC','SOTH','SOXE','STUR','STURX','THMS','WAVE','WHEN','WTOF','WSHP','WTON']

# Create N/A DataFrame
SK_fuel_technologies_match_df = pd.DataFrame('N/A', index=index, columns=list(SK_fuel_technologies_match.keys()))

# Display the DataFrame
SK_fuel_technologies_match_df

,AIR,AMO,BIO,ELE,GAS,GEO,HRD,HYD,LIG,NUC,OIL,OTH,PEA,SUN,THE,WAT,WHT,WIN,WST
ABHP,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
ALFC,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
ALKE,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
ASHP,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
BATS,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
BEVS,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
BSPG,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
CAES,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
COMC,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
COMCX,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A


<div style="background-color: black;">
<hr style="border: 0.5px solid skyblue;">
<div style="text-align: justify; margin-left: 0.0em; font-weight: normal; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
The Dispa-SET Technology - Fuel matching table for Slovenia.
<div style="text-align: justify; margin-left: 3.0em; font-weight: normal; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
Although the previous table can be used as support for all countries since it provides overall values from around the world, the following table specifically focuses on the Slove Energy System.
The same was built using the following additional sources:
</div>
<div style="text-align: justify; margin-left: 5em; font-weight: normal; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
<ol style="margin-top: 0; padding-left: 1.5em;">
<li>
xxxxxxxxxxxxxxxxxxxxxx. 
<br>
<a href="xxxxxxxxxxxxxxxxxxx" style="color:skyblue">xxxxxxxxxxxxxxxxxxxxxxi</a>
</li>
</ol>
</div>
</div>
<hr style="border: 0.5px solid skyblue;">
</div>

In [301]:
SI_fuel_technologies_match = {
    'AIR': [],
    'AMO': [],
    'BIO': [],
    'ELE': [],
    'GAS': [],
    'GEO': [],
    'HRD': [],
    'HYD': [],
    'LIG': [],
    'NUC': [],
    'OIL': [],
    'OTH': [],
    'PEA': [],
    'SUN': [],
    'THE': [],
    'WAT': [],
    'WHT': [],
    'WIN': [],
    'WST': []
}
index = ['ABHP','ALFC','ALKE','ASHP','BATS','BEVS','BSPG','CAES','COMC','COMCX','DMFC','GETH','GSHP','GTUR','GTURX','H2ST','HBBS','HDAM','HDAMC','HDLZ','HOBO','HOBOX','HPHS','HPHSC','HROR','HRORC','HYHP','ICEN','ICENX','MCFC','PAFC','P2BS','P2GS','P2HT','PEFC','PEME','PHOT','REFC','REHE','SCSP','SOFC','SOTH','SOXE','STUR','STURX','THMS','WAVE','WHEN','WTOF','WSHP','WTON']

# Create N/A DataFrame
SI_fuel_technologies_match_df = pd.DataFrame('N/A', index=index, columns=list(SI_fuel_technologies_match.keys()))

# Display the DataFrame
SI_fuel_technologies_match_df

,AIR,AMO,BIO,ELE,GAS,GEO,HRD,HYD,LIG,NUC,OIL,OTH,PEA,SUN,THE,WAT,WHT,WIN,WST
ABHP,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
ALFC,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
ALKE,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
ASHP,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
BATS,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
BEVS,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
BSPG,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
CAES,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
COMC,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
COMCX,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A


<div style="background-color: black;">
<hr style="border: 0.5px solid skyblue;">
<div style="text-align: justify; margin-left: 0.0em; font-weight: normal; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
    The Dispa-SET Technology - Fuel matching table for United Kingdom.
    <br>
<div style="text-align: justify; margin-left: 3.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
Although the previous table can be used as support for all countries since it provides overall values from around the world, the following table specifically focuses on the United Kingdom Energy System.
    The same was built using the following additional sources:
</div>
<div style="text-align: justify; margin-left: 5em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
    <a href="https://www.gov.uk/government/collections/digest-of-uk-energy-statistics-dukes" style="color:skyblue">Department for Energy Security and Net Zero - Digest of UK Energy Statistics (DUKES) (Accessed: 2025-09-30)</a>
    <br>
    <a href="https://www.thecrownestate.co.uk/our-business/offshore-wind/" style="color:skyblue">The Crown Estate - Offshore wind (Accessed: 2025-09-30)</a>
    <br>
</div>
</div>
<hr style="border: 0.5px solid skyblue;">
</div>

In [302]:
UK_fuel_technologies_match = {
    'AIR': [0.05, 0, 0, 0.95, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    'AMO': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0.15, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    'BIO': [0, 0, 0, 0, 0, 0, 0, 0, 0.08, 0.2, 0, 0, 0, 0.05, 0.15, 0, 1, 0, 0, 0, 0.05, 0.15, 0, 0, 0, 0, 0.1, 0.05, 0.1, 0.1, 0.1, 0, 0, 0, 0.1, 0, 0, 0, 0, 0, 0.1, 0, 0, 0.05, 0.05, 0, 0, 0, 0, 0, 0],
    'ELE': [0, 1, 1, 0.05, 1, 1, 0, 1, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0.95, 0.95, 0, 0, 0.9, 0, 0, 0, 0, 0.85, 1, 1, 0.05, 1, 0, 1, 1, 0, 0.05, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0],
    'GAS': [0, 0, 0, 0, 0, 0, 0, 0, 0.85, 0.7, 0, 0, 0, 0.85, 0.7, 0, 0, 0, 0, 0, 0.85, 0.65, 0, 0, 0, 0, 0, 0.4, 0.5, 0.8, 0.8, 0, 0, 0, 0.8, 0, 0, 0, 0, 0, 0.8, 0, 0, 0.1, 0.15, 0, 0, 0, 0, 0, 0],
    'GEO': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0.000001, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    'HRD': [0, 0, 0, 0, 0, 0, 0, 0, 0.02, 0.05, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0.05, 0, 0, 0, 0, 0, 0.1, 0.1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0.02, 0.05, 0, 0, 0, 0, 0, 0],
    'HYD': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    'LIG': [0, 0, 0, 0, 0, 0, 0, 0, 0.05, 0.05, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    'NUC': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0.8, 0.75, 0, 0, 0, 0, 0, 0],
    'OIL': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0.1, 0.15, 0, 0, 0, 0, 0, 0.1, 0.1, 0, 0, 0, 0, 0, 0.45, 0.3, 0.1, 0.1, 0, 0, 0, 0.05, 0, 0, 0, 0, 0, 0.05, 0, 0, 0.03, 0, 0, 0, 0, 0, 0, 0],
    'OTH': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    'PEA': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    'SUN': [0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0.000001, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    'THE': [0.95, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0],
    'WAT': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 0, 0, 0.05, 0.05, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    'WHT': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0],
    'WIN': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1],
    'WST': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0.05, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
}

index = ['ABHP','ALFC','ALKE','ASHP','BATS','BEVS','BSPG','CAES','COMC','COMCX','DMFC','GETH','GSHP','GTUR','GTURX','H2ST','HBBS','HDAM','HDAMC','HDLZ','HOBO','HOBOX','HPHS','HPHSC','HROR','HRORC','HYHP','ICEN','ICENX','MCFC','PAFC','P2BS','P2GS','P2HT','PEFC','PEME','PHOT','REFC','REHE','SCSP','SOFC','SOTH','SOXE','STUR','STURX','THMS','WAVE','WHEN','WTOF','WSHP','WTON']

# Create DataFrame
UK_fuel_technologies_match_df = pd.DataFrame(UK_fuel_technologies_match, index=index)

# Display the DataFrame
UK_fuel_technologies_match_df

,AIR,AMO,BIO,ELE,GAS,GEO,HRD,HYD,LIG,NUC,OIL,OTH,PEA,SUN,THE,WAT,WHT,WIN,WST
ABHP,0.05,0.00,0.00,0.00,0.00,0.000000,0.00,0,0.00,0.00,0.00,0,0,0.000000,0.95,0.00,0,0,0.00
ALFC,0.00,0.00,0.00,1.00,0.00,0.000000,0.00,0,0.00,0.00,0.00,0,0,0.000000,0.00,0.00,0,0,0.00
ALKE,0.00,0.00,0.00,1.00,0.00,0.000000,0.00,0,0.00,0.00,0.00,0,0,0.000000,0.00,0.00,0,0,0.00
ASHP,0.95,0.00,0.00,0.05,0.00,0.000000,0.00,0,0.00,0.00,0.00,0,0,0.000000,0.00,0.00,0,0,0.00
BATS,0.00,0.00,0.00,1.00,0.00,0.000000,0.00,0,0.00,0.00,0.00,0,0,0.000000,0.00,0.00,0,0,0.00
BEVS,0.00,0.00,0.00,1.00,0.00,0.000000,0.00,0,0.00,0.00,0.00,0,0,0.000000,0.00,0.00,0,0,0.00
BSPG,0.00,0.00,0.00,0.00,0.00,0.000000,0.00,0,0.00,0.00,0.00,0,0,1.000000,0.00,0.00,0,0,0.00
CAES,0.00,0.00,0.00,1.00,0.00,0.000000,0.00,0,0.00,0.00,0.00,0,0,0.000000,0.00,0.00,0,0,0.00
COMC,0.00,0.00,0.08,0.00,0.85,0.000000,0.02,0,0.05,0.00,0.00,0,0,0.000000,0.00,0.00,0,0,0.00
COMCX,0.00,0.00,0.20,0.00,0.70,0.000000,0.05,0,0.05,0.00,0.00,0,0,0.000000,0.00,0.00,0,0,0.00


<div style="background-color: black;">
<hr style="border: 0.5px solid skyblue;">
<div style="text-align: justify; margin-left: 0.0em; font-weight: normal; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
    The Dispa-SET Technology - Fuel matching table for Spain.
    <br>
<div style="text-align: justify; margin-left: 3.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
Although the previous table can be used as support for all countries since it provides overall values from around the world, the following table specifically focuses on the Spanish Energy System.
    The same was built using the following additional sources:
</div>
<div style="text-align: justify; margin-left: 5em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
    <ol style="margin-top: 0; padding-left: 1.5em;">
        <li>
            International Energy Agency - Spain 2021: Energy Policy Review (2021).
            <br>
            <a href="https://www.iea.org/reports/spain-2021" style="color:skyblue">https://www.iea.org/reports/spain-2021</a>
        </li>
        <li>
            Red Eléctrica de España - El sistema eléctrico español 2022 (2023).
            <br>
            <a href="https://www.ree.es/es/datos/publicaciones" style="color:skyblue">https://www.ree.es/es/datos/publicaciones</a>
        </li>
        <li>
            Red Eléctrica de España - Series estadísticas nacionales (Accessed: 2025-09-30).
            <br>
            <a href="https://www.ree.es/es/datos/aldia" style="color:skyblue">https://www.ree.es/es/datos/aldia</a>
        </li>
        <li>
            Spanish Ministry for the Ecological Transition and the Demographic Challenge - Plan Nacional Integrado de Energía y Clima (PNIEC) 2021-2030 (2020).
            <br>
            <a href="https://www.miteco.gob.es/es/prensa/pniec.aspx" style="color:skyblue">https://www.miteco.gob.es/es/prensa/pniec.aspx</a>
        </li>
    </ol>
</div>
</div>
<hr style="border: 0.5px solid skyblue;">
</div>

In [303]:
ES_fuel_technologies_match = {
    'AIR': [0.05, 0, 0, 0.95, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    'AMO': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0.15, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    'BIO': [0, 0, 0, 0, 0, 0, 0, 0, 0.08, 0.2, 0, 0, 0, 0.05, 0.15, 0, 1, 0, 0, 0, 0.05, 0.15, 0, 0, 0, 0, 0.1, 0.05, 0.1, 0.1, 0.1, 0, 0, 0, 0.1, 0, 0, 0, 0, 0, 0.1, 0, 0, 0.05, 0.05, 0, 0, 0, 0, 0, 0],
    'ELE': [0, 1, 1, 0.05, 1, 1, 0, 1, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0.9, 0, 0, 0, 0, 0.85, 1, 1, 0.05, 1, 0, 1, 1, 0, 0.05, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0],
    'GAS': [0, 0, 0, 0, 0, 0, 0, 0, 0.85, 0.7, 0, 0, 0, 0.85, 0.7, 0, 0, 0, 0, 0, 0.85, 0.65, 0, 0, 0, 0, 0, 0.4, 0.5, 0.8, 0.8, 0, 0, 0, 0.8, 0, 0, 0, 0, 0, 0.8, 0, 0, 0.1, 0.15, 0, 0, 0, 0, 0, 0],
    'GEO': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    'HRD': [0, 0, 0, 0, 0, 0, 0, 0, 0.02, 0.05, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0.05, 0, 0, 0, 0, 0, 0.1, 0.1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0.02, 0.05, 0, 0, 0, 0, 0, 0],
    'HYD': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    'LIG': [0, 0, 0, 0, 0, 0, 0, 0, 0.05, 0.05, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    'NUC': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0.8, 0.75, 0, 0, 0, 0, 0, 0],
    'OIL': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0.1, 0.15, 0, 0, 0, 0, 0, 0.1, 0.1, 0, 0, 0, 0, 0, 0.45, 0.3, 0.1, 0.1, 0, 0, 0, 0.05, 0, 0, 0, 0, 0, 0.05, 0, 0, 0.03, 0, 0, 0, 0, 0, 0, 0],
    'OTH': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    'PEA': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    'SUN': [0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    'THE': [0.95, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0],
    'WAT': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 0, 0, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0],
    'WHT': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0],
    'WIN': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1],
    'WST': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0.05, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
}

index = ['ABHP','ALFC','ALKE','ASHP','BATS','BEVS','BSPG','CAES','COMC','COMCX','DMFC','GETH','GSHP','GTUR','GTURX','H2ST','HBBS','HDAM','HDAMC','HDLZ','HOBO','HOBOX','HPHS','HPHSC','HROR','HRORC','HYHP','ICEN','ICENX','MCFC','PAFC','P2BS','P2GS','P2HT','PEFC','PEME','PHOT','REFC','REHE','SCSP','SOFC','SOTH','SOXE','STUR','STURX','THMS','WAVE','WHEN','WTOF','WSHP','WTON']

# Create DataFrame
ES_fuel_technologies_match_df = pd.DataFrame(ES_fuel_technologies_match, index=index)

# Display the DataFrame
ES_fuel_technologies_match_df

,AIR,AMO,BIO,ELE,GAS,GEO,HRD,HYD,LIG,NUC,OIL,OTH,PEA,SUN,THE,WAT,WHT,WIN,WST
ABHP,0.05,0.00,0.00,0.00,0.00,0,0.00,0,0.00,0.00,0.00,0,0,0,0.95,0,0,0,0.00
ALFC,0.00,0.00,0.00,1.00,0.00,0,0.00,0,0.00,0.00,0.00,0,0,0,0.00,0,0,0,0.00
ALKE,0.00,0.00,0.00,1.00,0.00,0,0.00,0,0.00,0.00,0.00,0,0,0,0.00,0,0,0,0.00
ASHP,0.95,0.00,0.00,0.05,0.00,0,0.00,0,0.00,0.00,0.00,0,0,0,0.00,0,0,0,0.00
BATS,0.00,0.00,0.00,1.00,0.00,0,0.00,0,0.00,0.00,0.00,0,0,0,0.00,0,0,0,0.00
BEVS,0.00,0.00,0.00,1.00,0.00,0,0.00,0,0.00,0.00,0.00,0,0,0,0.00,0,0,0,0.00
BSPG,0.00,0.00,0.00,0.00,0.00,0,0.00,0,0.00,0.00,0.00,0,0,1,0.00,0,0,0,0.00
CAES,0.00,0.00,0.00,1.00,0.00,0,0.00,0,0.00,0.00,0.00,0,0,0,0.00,0,0,0,0.00
COMC,0.00,0.00,0.08,0.00,0.85,0,0.02,0,0.05,0.00,0.00,0,0,0,0.00,0,0,0,0.00
COMCX,0.00,0.00,0.20,0.00,0.70,0,0.05,0,0.05,0.00,0.00,0,0,0,0.00,0,0,0,0.00


<div style="background-color: black;">
<hr style="border: 0.5px solid skyblue;">
<div style="text-align: justify; margin-left: 0.0em; font-weight: normal; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
    The Dispa-SET Technology - Fuel matching table for Luxembourg.
    <br>
<div style="text-align: justify; margin-left: 3.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
Although the previous table can be used as support for all countries since it provides overall values from around the world, the following table specifically focuses on the Luxembourg Energy System.
    The same was built using the following additional sources:
</div>
<div style="text-align: justify; margin-left: 5em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
    <ol style="margin-top: 0; padding-left: 1.5em;">
        <li>
            International Energy Agency - Luxembourg 2020: Energy Policy Review (2020).
            <br>
            <a href="https://www.iea.org/reports/luxembourg-2020" style="color:skyblue">https://www.iea.org/reports/luxembourg-2020</a>
        </li>
        <li>
            STATEC - Energy (Accessed: 2025-09-30).
            <br>
            <a href="https://statistiques.public.lu/en.html" style="color:skyblue">https://statistiques.public.lu/en.html</a>
        </li>
    </ol>
</div>
</div>
<hr style="border: 0.5px solid skyblue;">
</div>

In [304]:
LU_fuel_technologies_match = {
    'AIR': [0.95, 0, 0, 0.95, 0, 0, 0, 0, 0, 0, 0, 0, 0.95, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0.9, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0.95, 0],
    'AMO': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    'BIO': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0.2, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0.2, 0.25, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0.066892464, 0.25, 0, 0, 0, 0, 0, 0],
    'ELE': [0.05, 0, 1, 0.05, 1, 1, 0, 1, 0, 0, 0, 0, 0.05, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0.1, 0, 0, 0, 0, 1, 1, 1, 0, 1, 0, 0, 1, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0.05, 0],
    'GAS': [0, 0, 0, 0, 0, 0, 0, 0, 1, 0.75, 0, 0, 0, 1, 0.95, 0, 0, 0, 0, 0, 0.75, 0.65, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0.082133785, 0.7, 0, 0, 0, 0, 0, 0],
    'GEO': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0.008467401, 0, 0, 0, 0, 0, 0, 0],
    'HRD': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0.32938188, 0, 0, 0, 0, 0, 0, 0],
    'HYD': [0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0.000084674, 0, 0, 0, 0, 0, 0, 0],
    'LIG': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0.210838273, 0, 0, 0, 0, 0, 0, 0],
    'NUC': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0.116850127, 0, 0, 0, 0, 0, 0, 0],
    'OIL': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0.05, 0, 0, 0, 0, 0.05, 0, 0, 0, 0, 0, 0.05, 0.1, 0, 0, 0, 0, 0, 0.7, 0.8, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0.123624047, 0.05, 0, 0, 0, 0, 0, 0],
    'OTH': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0.00084674, 0, 0, 0, 0, 0, 0, 0],
    'PEA': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0.014394581, 0, 0, 0, 0, 0, 0, 0],
    'SUN': [0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 1, 0, 0.00592718, 0, 0, 0, 0, 0, 0, 0],
    'THE': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0],
    'WAT': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 0, 0, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0.00084674, 0, 0, 1, 0, 0, 0, 0],
    'WHT': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    'WIN': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1],
    'WST': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0.3, 0.2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0.038950042, 0, 0, 0, 0, 0, 0, 0]
}

index = ['ABHP','ALFC','ALKE','ASHP','BATS','BEVS','BSPG','CAES','COMC','COMCX','DMFC','GETH','GSHP','GTUR','GTURX','H2ST','HBBS','HDAM','HDAMC','HDLZ','HOBO','HOBOX','HPHS','HPHSC','HROR','HRORC','HYHP','ICEN','ICENX','MCFC','PAFC','P2BS','P2GS','P2HT','PEFC','PEME','PHOT','REFC','REHE','SCSP','SOFC','SOTH','SOXE','STUR','STURX','THMS','WAVE','WHEN','WTOF','WSHP','WTON']

# Create DataFrame
LU_fuel_technologies_match_df = pd.DataFrame(LU_fuel_technologies_match, index=index)

# Display the DataFrame
LU_fuel_technologies_match_df

,AIR,AMO,BIO,ELE,GAS,GEO,HRD,HYD,LIG,NUC,OIL,OTH,PEA,SUN,THE,WAT,WHT,WIN,WST
ABHP,0.95,0,0.000000,0.05,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0,0.000000,0,0,0.00000
ALFC,0.00,0,0.000000,0.00,0.000000,0.000000,0.000000,1.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0,0.000000,0,0,0.00000
ALKE,0.00,0,0.000000,1.00,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0,0.000000,0,0,0.00000
ASHP,0.95,0,0.000000,0.05,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0,0.000000,0,0,0.00000
BATS,0.00,0,0.000000,1.00,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0,0.000000,0,0,0.00000
BEVS,0.00,0,0.000000,1.00,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0,0.000000,0,0,0.00000
BSPG,0.00,0,0.000000,0.00,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,1.000000,0,0.000000,0,0,0.00000
CAES,0.00,0,0.000000,1.00,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0,0.000000,0,0,0.00000
COMC,0.00,0,0.000000,0.00,1.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0,0.000000,0,0,0.00000
COMCX,0.00,0,0.200000,0.00,0.750000,0.000000,0.000000,0.000000,0.000000,0.00000,0.050000,0.000000,0.000000,0.000000,0,0.000000,0,0,0.00000


<div style="background-color: black;">
<hr style="border: 0.5px solid skyblue;">
<div style="text-align: justify; margin-left: 0.0em; font-weight: normal; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
The Dispa-SET Technology - Fuel matching table for Italy.
<div style="text-align: justify; margin-left: 3.0em; font-weight: normal; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
Although the previous table can be used as support for all countries since it provides overall values from around the world, the following table specifically focuses on the Italian Energy System.
The same was built using the following additional sources:
</div>
<div style="text-align: justify; margin-left: 5em; font-weight: normal; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
    <ol style="margin-top: 0; padding-left: 1.5em;">
        <li>
            Gestore dei Servizi Energetici (GSE). (2023). Rapporto Statistico 2022: Energia da Fonti Rinnovabili in Italia.
            <br>
            <a href="https://www.gse.it/dati-e-scenari/rapporti" style="color:skyblue">https://www.gse.it/dati-e-scenari/rapporti</a>
        </li>
        <li>
            International Energy Agency. (2023). Italy 2023: Energy Policy Review. IEA.
            <br>
            <a href="https://www.iea.org/reports/italy-2023" style="color:skyblue">https://www.iea.org/reports/italy-2023</a>
        </li>
        <li>
            Terna S.p.A. (2023). Dati statistici sull'energia elettrica in Italia.
            <br>
            <a href="https://www.terna.it/it/sistema-elettrico/statistiche" style="color:skyblue">https://www.terna.it/it/sistema-elettrico/statistiche</a>
        </li>
        <li>
            Italian Ministry of the Environment and Energy Security. (2023). Piano Nazionale Integrato per l'Energia e il Clima (PNIEC).
            <br>
            <a href="https://www.mase.gov.it/portale/home" style="color:skyblue">https://www.mase.gov.it/portale/home</a>
        </li>
    </ol>
</div>
</div>
<hr style="border: 0.5px solid skyblue;">
</div>

In [305]:
IT_fuel_technologies_match = {
    'AIR': [0.95, 0, 0, 0.95, 0, 0, 0, 0, 0, 0, 0, 0, 0.95, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0.9, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0.95, 0],
    'AMO': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    'BIO': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0.2, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0.1, 0.15, 0, 0, 0, 0, 0, 0.2, 0.25, 0, 0, 0.15, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0.15, 0.2, 0, 0, 0, 0, 0, 0],
    'ELE': [0.05, 0, 1, 0.05, 1, 1, 0, 1, 0, 0, 0, 0, 0.05, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0.1, 0, 0, 0, 0, 0.7, 1, 1, 0, 1, 0, 0, 1, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0.05, 0],
    'GAS': [0, 0, 0, 0, 0, 0, 0, 0, 0.9, 0.75, 0, 0, 0, 0.85, 0.8, 0, 0, 0, 0, 0, 0.75, 0.7, 0, 0, 0, 0, 0, 0.4, 0.5, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0.4, 0.65, 0, 0, 0, 0, 0, 0],
    'GEO': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0.1, 0, 0, 0, 0, 0, 0, 0],
    'HRD': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    'HYD': [0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0.15, 0.2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0.1, 0.15, 0, 0, 0.15, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0.15, 0, 0, 0, 0, 0, 0],
    'LIG': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    'NUC': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    'OIL': [0, 0, 0, 0, 0, 0, 0, 0, 0.1, 0.05, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0.15, 0.15, 0, 0, 0, 0, 0, 0.3, 0.1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0.35, 0, 0, 0, 0, 0, 0, 0],
    'OTH': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    'PEA': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    'SUN': [0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    'THE': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0],
    'WAT': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 0, 0, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0],
    'WHT': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    'WIN': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1],
    'WST': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
}

# The index list (51 elements) for the DataFrame rows
index = ['ABHP','ALFC','ALKE','ASHP','BATS','BEVS','BSPG','CAES','COMC','COMCX','DMFC','GETH','GSHP','GTUR','GTURX','H2ST','HBBS','HDAM','HDAMC','HDLZ','HOBO','HOBOX','HPHS','HPHSC','HROR','HRORC','HYHP','ICEN','ICENX','MCFC','PAFC','P2BS','P2GS','P2HT','PEFC','PEME','PHOT','REFC','REHE','SCSP','SOFC','SOTH','SOXE','STUR','STURX','THMS','WAVE','WHEN','WTOF','WSHP','WTON']

# Create DataFrame
IT_fuel_technologies_match_df = pd.DataFrame(IT_fuel_technologies_match, index=index)

# Display the DataFrame
IT_fuel_technologies_match_df

,AIR,AMO,BIO,ELE,GAS,GEO,HRD,HYD,LIG,NUC,OIL,OTH,PEA,SUN,THE,WAT,WHT,WIN,WST
ABHP,0.95,0,0.00,0.05,0.00,0.0,0,0.00,0,0,0.00,0,0,0,0,0,0,0,0
ALFC,0.00,0,0.00,0.00,0.00,0.0,0,1.00,0,0,0.00,0,0,0,0,0,0,0,0
ALKE,0.00,0,0.00,1.00,0.00,0.0,0,0.00,0,0,0.00,0,0,0,0,0,0,0,0
ASHP,0.95,0,0.00,0.05,0.00,0.0,0,0.00,0,0,0.00,0,0,0,0,0,0,0,0
BATS,0.00,0,0.00,1.00,0.00,0.0,0,0.00,0,0,0.00,0,0,0,0,0,0,0,0
BEVS,0.00,0,0.00,1.00,0.00,0.0,0,0.00,0,0,0.00,0,0,0,0,0,0,0,0
BSPG,0.00,0,0.00,0.00,0.00,0.0,0,0.00,0,0,0.00,0,0,1,0,0,0,0,0
CAES,0.00,0,0.00,1.00,0.00,0.0,0,0.00,0,0,0.00,0,0,0,0,0,0,0,0
COMC,0.00,0,0.00,0.00,0.90,0.0,0,0.00,0,0,0.10,0,0,0,0,0,0,0,0
COMCX,0.00,0,0.20,0.00,0.75,0.0,0,0.00,0,0,0.05,0,0,0,0,0,0,0,0


<div style="background-color: black;">
<hr style="border: 0.5px solid skyblue;">
<div style="text-align: justify; margin-left: 0.0em; font-weight: normal; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
The Dispa-SET Technology - Fuel matching table for Sweden.
<div style="text-align: justify; margin-left: 3.0em; font-weight: normal; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
Although the previous table can be used as support for all countries since it provides overall values from around the world, the following table specifically focuses on the Sweden Energy System.
The same was built using the following additional sources:
</div>
<div style="text-align: justify; margin-left: 5em; font-weight: normal; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
<ol style="margin-top: 0; padding-left: 1.5em;">
    <li>
        Energy in Sweden - Facts and Figures 2023 
        <br>
        <a href="https://www.energimyndigheten.se/en/news/2023/energy-in-sweden---facts-and-figures-2023/" style="color:skyblue">https://www.energimyndigheten.se/en/news/2023/energy-in-sweden---facts-and-figures-2023/</a>
    </li>
    <li>
        [PDF] Energy in Sweden 2022 – An overview - Energimyndigheten 
        <br>
        <a href="https://energimyndigheten.a-w2m.se/Arkitektkopia/GetTemplateResource/121?id=5cec65f476d74e0e98a0eb814d1daf1a&res=5dc13196322d4cb3b08dafacec06d733&lr=False&fn=ET%202022_04_webb.pdf&elp=portal&elt=t&eloid=5cec65f476d74e0e98a0eb814d1daf1a" style="color:skyblue">https://energimyndigheten.a-w2m.se/...</a>
    </li>
    <li>
        [XLS] Innehåll - Energimyndigheten 
        <br>
        <a href="http://www.energimyndigheten.se/48d848/globalassets/statistik/energilaget/energy-in-sweden-facts-and-figures-20242.xlsx" style="color:skyblue">http://www.energimyndigheten.se/48d848/globalassets/statistik/...</a>
    </li>
    <li>
        Sweden 2024 – Analysis - IEA 
        <br>
        <a href="https://www.iea.org/reports/sweden-2024" style="color:skyblue">https://www.iea.org/reports/sweden-2024</a>
    </li>
    <li>
        Ny rapport: Nordic Grid Development Perspective 2023 
        <br>
        <a href="https://www.svk.se/press-och-nyheter/nyheter/allmanna-nyheter/2023/ny-rapport-nordic-grid-development-perspective-2023/" style="color:skyblue">https://www.svk.se/press-och-nyheter/nyheter/...</a>
    </li>
    <li>
        Shedding light on energy in Europe – 2024 edition - EC Europa 
        <br>
        <a href="https://ec.europa.eu/eurostat/web/interactive-publications/energy-2024" style="color:skyblue">https://ec.europa.eu/eurostat/web/interactive-publications/energy-2024</a>
    </li>
    <li>
        EU Energy in Figures: statistical data on EU energy sector 
        <br>
        <a href="https://energy.ec.europa.eu/news/eu-energy-figures-statistical-data-eu-energy-sector-2024-10-14_en" style="color:skyblue">https://energy.ec.europa.eu/news/eu-energy-figures-statistical-data...</a>
    </li>
    <li>
        Database - Energy - Eurostat - EC Europa 
        <br>
        <a href="https://ec.europa.eu/eurostat/web/energy/database" style="color:skyblue">https://ec.europa.eu/eurostat/web/energy/database</a>
    </li>
    <li>
        Renewable energy statistics - Statistics Explained - Eurostat 
        <br>
        <a href="https://ec.europa.eu/eurostat/statistics-explained/index.php/Renewable_energy_statistics" style="color:skyblue">https://ec.europa.eu/eurostat/statistics-explained/index.php/Renewable_energy_statistics</a>
    </li>
    <li>
        Energy statistics - an overview - EC Europa - European Union 
        <br>
        <a href="https://ec.europa.eu/eurostat/statistics-explained/index.php/Energy_statistics_-_an_overview" style="color:skyblue">https://ec.europa.eu/eurostat/statistics-explained/index.php/Energy_statistics_-_an_overview</a>
    </li>
    <li>
        [PDF] Sweden's draft integrated national energy and climate plan 
        <br>
        <a href="https://www.government.se/contentassets/e731726022cd4e0b8ffa0f8229893115/swedens-draft-integrated-national-energy-and-climate-plan/" style="color:skyblue">https://www.government.se/contentassets/...</a>
    </li>
    <li>
        [PDF] Draft updated National Energy and Climate Plan (NECP) for Sweden 
        <br>
        <a href="https://commission.europa.eu/system/files/2023-07/EN_SWEDEN%20DRAFT%20UPDATED%20NECP.pdf" style="color:skyblue">https://commission.europa.eu/system/files/2023-07/EN_SWEDEN%20DRAFT%20UPDATED%20NECP.pdf</a>
    </li>
    <li>
        The Swedish offer to support ambitious climate plans - Government.se 
        <br>
        <a href="https://www.government.se/articles/2025/05/the-swedish-offer-to-support-ambitious-climate-plans/" style="color:skyblue">https://www.government.se/articles/2025/05/the-swedish-offer-to-support-ambitious-climate-plans/</a>
    </li>
</ol>
</div>
</div>
<hr style="border: 0.5px solid skyblue;">
</div>

In [306]:
SE_fuel_technologies_match = {
    'AIR': [0.95, 0, 0, 0.95, 0, 0, 0, 0, 0, 0, 0, 0, 0.95, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0.9, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0.95, 0],
    'AMO': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    'BIO': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0.6, 0, 0, 0, 0, 0.5, 0, 1, 0, 0, 0, 0.7, 0.6, 0, 0, 0, 0, 0, 0.3, 0.5, 0, 0, 0.4, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0.2, 0.5, 0, 0, 0, 0, 0, 0],
    'ELE': [0.05, 0, 1, 0.05, 1, 1, 0, 0.65, 0, 0, 0, 0, 0.05, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0.1, 0, 0, 0, 0, 0.5, 1, 1, 0, 1, 0, 0, 1, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0.05, 0],
    'GAS': [0, 0, 0, 0, 0, 0, 0, 0.35, 0.9, 0.3, 0, 0, 0, 0.8, 0.4, 0, 0, 0, 0, 0, 0.2, 0.3, 0, 0, 0, 0, 0, 0.2, 0.3, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0.3, 0, 0, 0, 0, 0, 0],
    'GEO': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    'HRD': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    'HYD': [0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0.2, 0.1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0.1, 0.2, 0, 0, 0.1, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0.2, 0, 0, 0, 0, 0, 0],
    'LIG': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    'NUC': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0.8, 0, 0, 0, 0, 0, 0, 0],
    'OIL': [0, 0, 0, 0, 0, 0, 0, 0, 0.1, 0.1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0.1, 0.1, 0, 0, 0, 0, 0, 0.4, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    'OTH': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    'PEA': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    'SUN': [0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    'THE': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0],
    'WAT': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0],
    'WHT': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    'WIN': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1],
    'WST': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
}

# The index list (51 elements) for the DataFrame rows
index = ['ABHP','ALFC','ALKE','ASHP','BATS','BEVS','BSPG','CAES','COMC','COMCX','DMFC','GETH','GSHP','GTUR','GTURX','H2ST','HBBS','HDAM','HDAMC','HDLZ','HOBO','HOBOX','HPHS','HPHSC','HROR','HRORC','HYHP','ICEN','ICENX','MCFC','PAFC','P2BS','P2GS','P2HT','PEFC','PEME','PHOT','REFC','REHE','SCSP','SOFC','SOTH','SOXE','STUR','STURX','THMS','WAVE','WHEN','WTOF','WSHP','WTON']

# Create DataFrame
SE_fuel_technologies_match_df = pd.DataFrame(SE_fuel_technologies_match, index=index)

# Display the DataFrame
SE_fuel_technologies_match_df

,AIR,AMO,BIO,ELE,GAS,GEO,HRD,HYD,LIG,NUC,OIL,OTH,PEA,SUN,THE,WAT,WHT,WIN,WST
ABHP,0.95,0,0.0,0.05,0.00,0,0,0.0,0,0.0,0.0,0,0,0,0,0,0,0,0
ALFC,0.00,0,0.0,0.00,0.00,0,0,1.0,0,0.0,0.0,0,0,0,0,0,0,0,0
ALKE,0.00,0,0.0,1.00,0.00,0,0,0.0,0,0.0,0.0,0,0,0,0,0,0,0,0
ASHP,0.95,0,0.0,0.05,0.00,0,0,0.0,0,0.0,0.0,0,0,0,0,0,0,0,0
BATS,0.00,0,0.0,1.00,0.00,0,0,0.0,0,0.0,0.0,0,0,0,0,0,0,0,0
BEVS,0.00,0,0.0,1.00,0.00,0,0,0.0,0,0.0,0.0,0,0,0,0,0,0,0,0
BSPG,0.00,0,0.0,0.00,0.00,0,0,0.0,0,0.0,0.0,0,0,1,0,0,0,0,0
CAES,0.00,0,0.0,0.65,0.35,0,0,0.0,0,0.0,0.0,0,0,0,0,0,0,0,0
COMC,0.00,0,0.0,0.00,0.90,0,0,0.0,0,0.0,0.1,0,0,0,0,0,0,0,0
COMCX,0.00,0,0.6,0.00,0.30,0,0,0.0,0,0.0,0.1,0,0,0,0,0,0,0,0


<div style="background-color: black;">
<hr style="border: 0.5px solid skyblue;">
<div style="text-align: justify; margin-left: 0.0em; font-weight: normal; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
The Dispa-SET Technology - Fuel matching table for Switzerland.
<div style="text-align: justify; margin-left: 3.0em; font-weight: normal; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
Although the previous table can be used as support for all countries since it provides overall values from around the world, the following table specifically focuses on the Swiss Energy System.
The same was built using the following additional sources:
</div>
<div style="text-align: justify; margin-left: 5em; font-weight: normal; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
<ol style="margin-top: 0; padding-left: 1.5em;">
    <li>
        Bundesamt für Energie (BFE). (2023). Schweizerische Gesamtenergiestatistik 2022.
        <br>
        <a href="https://www.bfe.admin.ch/bfe/en/home/supply/statistics-and-geodata/energy-statistics/overall-energy-statistics.html" style="color:skyblue">https://www.bfe.admin.ch/bfe/en/home/supply/statistics-and-geodata/energy-statistics/overall-energy-statistics.html</a>
    </li>
    <li>
        Swiss Federal Office of Energy (SFOE). (2023). Swiss Energy Strategy 2050.
        <br>
        <a href="https://www.bfe.admin.ch/bfe/en/home/policy/energy-strategy-2050.html" style="color:skyblue">https://www.bfe.admin.ch/bfe/en/home/policy/energy-strategy-2050.html</a>
    </li>
    <li>
        International Energy Agency (IEA). (2022). Switzerland 2022: Energy Policy Review.
        <br>
        <a href="https://www.iea.org/reports/switzerland-2023" style="color:skyblue">https://www.iea.org/reports/switzerland-2023</a>
    </li>
    <li>
        Swissgrid. (2023). Grid and infrastructure data.
        <br>
        <a href="https://www.swissgrid.ch/en/home/newsroom/newsfeed/20240416-01.html" style="color:skyblue">https://www.swissgrid.ch/en/home/newsroom/newsfeed/20240416-01.html</a>
    </li>
    <li>
        Eurostat. (2024). Energy database. European Commission.
        <br>
        <a href="https://ec.europa.eu/eurostat/web/energy/database" style="color:skyblue">https://ec.europa.eu/eurostat/web/energy/database</a>
    </li>
</ol>
</div>
</div>
<hr style="border: 0.5px solid skyblue;">
</div>

In [307]:
CH_fuel_technologies_match = {
    'AIR': [0.95, 0, 0, 0.95, 0, 0, 0, 0, 0, 0, 0, 0, 0.95, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0.9, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0.95, 0],
    'AMO': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    'BIO': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0.25, 0, 0, 0, 0, 0.2, 0, 1, 0, 0, 0, 0.15, 0.25, 0, 0, 0, 0, 0, 0.25, 0.35, 0, 0, 0.2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0.15, 0.25, 0, 0, 0, 0, 0, 0],
    'ELE': [0.05, 0, 1, 0.05, 1, 1, 0, 0.65, 0, 0, 0, 0, 0.05, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0.1, 0, 0, 0, 0, 0.6, 1, 1, 0, 1, 0, 0, 1, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0.05, 0],
    'GAS': [0, 0, 0, 0, 0, 0, 0, 0.35, 1, 0.7, 0, 0, 0, 0.9, 0.75, 0, 0, 0, 0, 0, 0.7, 0.65, 0, 0, 0, 0, 0, 0.4, 0.5, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0.1, 0.6, 0, 0, 0, 0, 0, 0],
    'GEO': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    'HRD': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    'HYD': [0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0.1, 0.05, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0.1, 0.15, 0, 0, 0.2, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0.15, 0, 0, 0, 0, 0, 0],
    'LIG': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    'NUC': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0.75, 0, 0, 0, 0, 0, 0, 0],
    'OIL': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0.05, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0.15, 0.1, 0, 0, 0, 0, 0, 0.25, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    'OTH': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    'PEA': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    'SUN': [0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    'THE': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0],
    'WAT': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0],
    'WHT': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    'WIN': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1],
    'WST': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
}

# The index list (51 elements) for the DataFrame rows
index = ['ABHP','ALFC','ALKE','ASHP','BATS','BEVS','BSPG','CAES','COMC','COMCX','DMFC','GETH','GSHP','GTUR','GTURX','H2ST','HBBS','HDAM','HDAMC','HDLZ','HOBO','HOBOX','HPHS','HPHSC','HROR','HRORC','HYHP','ICEN','ICENX','MCFC','PAFC','P2BS','P2GS','P2HT','PEFC','PEME','PHOT','REFC','REHE','SCSP','SOFC','SOTH','SOXE','STUR','STURX','THMS','WAVE','WHEN','WTOF','WSHP','WTON']

# Create DataFrame
CH_fuel_technologies_match_df = pd.DataFrame(CH_fuel_technologies_match, index=index)

# Display the DataFrame
CH_fuel_technologies_match_df

,AIR,AMO,BIO,ELE,GAS,GEO,HRD,HYD,LIG,NUC,OIL,OTH,PEA,SUN,THE,WAT,WHT,WIN,WST
ABHP,0.95,0,0.00,0.05,0.00,0,0,0.00,0,0.00,0.00,0,0,0,0,0,0,0,0
ALFC,0.00,0,0.00,0.00,0.00,0,0,1.00,0,0.00,0.00,0,0,0,0,0,0,0,0
ALKE,0.00,0,0.00,1.00,0.00,0,0,0.00,0,0.00,0.00,0,0,0,0,0,0,0,0
ASHP,0.95,0,0.00,0.05,0.00,0,0,0.00,0,0.00,0.00,0,0,0,0,0,0,0,0
BATS,0.00,0,0.00,1.00,0.00,0,0,0.00,0,0.00,0.00,0,0,0,0,0,0,0,0
BEVS,0.00,0,0.00,1.00,0.00,0,0,0.00,0,0.00,0.00,0,0,0,0,0,0,0,0
BSPG,0.00,0,0.00,0.00,0.00,0,0,0.00,0,0.00,0.00,0,0,1,0,0,0,0,0
CAES,0.00,0,0.00,0.65,0.35,0,0,0.00,0,0.00,0.00,0,0,0,0,0,0,0,0
COMC,0.00,0,0.00,0.00,1.00,0,0,0.00,0,0.00,0.00,0,0,0,0,0,0,0,0
COMCX,0.00,0,0.25,0.00,0.70,0,0,0.00,0,0.00,0.05,0,0,0,0,0,0,0,0


<div style="background-color: black;">
<hr style="border: 0.5px solid skyblue;">
<div style="text-align: justify; margin-left: 0.0em; font-weight: normal; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
The Dispa-SET Technology - Fuel matching table for Austria.
<div style="text-align: justify; margin-left: 3.0em; font-weight: normal; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
Although the previous table can be used as support for all countries since it provides overall values from around the world, the following table specifically focuses on the Austrian Energy System.
The same was built using the following additional sources:
</div>
<div style="text-align: justify; margin-left: 5em; font-weight: normal; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
<ol style="margin-top: 0; padding-left: 1.5em;">
    <li>
        Bundesministerium für Klimaschutz, Umwelt, Energie, Mobilität, Innovation und Technologie (BMK). (2023). Integrated National Energy and Climate Plan for Austria 2021-2030.
        <br>
        <a href="https://www.bmk.gv.at/en.html" style="color:skyblue">https://www.bmk.gv.at/en.html</a>
    </li>
    <li>
        E-Control Austria. (2023). Energy statistics and data.
        <br>
        <a href="https://www.e-control.at/en/" style="color:skyblue">https://www.e-control.at/en/</a>
    </li>
    <li>
        International Energy Agency. (2020). Austria 2020: Energy Policy Review.
        <br>
        <a href="https://www.iea.org/reports/austria-2020" style="color:skyblue">https://www.iea.org/reports/austria-2020</a>
    </li>
    <li>
        Eurostat. (2024). Energy database. European Commission.
        <br>
        <a href="https://ec.europa.eu/eurostat/web/energy/database" style="color:skyblue">https://ec.europa.eu/eurostat/web/energy/database</a>
    </li>
    <li>
        Statistics Austria. (2023). Energy statistics.
        <br>
        <a href="https://www.statistik.at/en/statistics/energy-and-environment/energy" style="color:skyblue">https://www.statistik.at/en/statistics/energy-and-environment/energy</a>
    </li>
</ol>
</div>
</div>
<hr style="border: 0.5px solid skyblue;">
</div>

In [308]:
AT_fuel_technologies_match = {
    'AIR': [0.95, 0, 0, 0.95, 0, 0, 0, 0, 0, 0, 0, 0, 0.95, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0.9, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0.95, 0],
    'AMO': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    'BIO': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0.45, 0, 0, 0, 0, 0.4, 0, 1, 0, 0, 0, 0.6, 0.5, 0, 0, 0, 0, 0, 0.35, 0.45, 0, 0, 0.4, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0.25, 0.45, 0, 0, 0, 0, 0, 0],
    'ELE': [0.05, 0, 1, 0.05, 1, 1, 0, 0.65, 0, 0, 0, 0, 0.05, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0.1, 0, 0, 0, 0, 0.45, 1, 1, 0, 1, 0, 0, 1, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0.05, 0],
    'GAS': [0, 0, 0, 0, 0, 0, 0, 0.35, 0.85, 0.5, 0, 0, 0, 0.8, 0.55, 0, 0, 0, 0, 0, 0.3, 0.4, 0, 0, 0, 0, 0, 0.25, 0.4, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0.4, 0.5, 0, 0, 0, 0, 0, 0],
    'GEO': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    'HRD': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    'HYD': [0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0.2, 0.05, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0.15, 0.15, 0, 0, 0.15, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0.05, 0, 0, 0, 0, 0, 0],
    'LIG': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    'NUC': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    'OIL': [0, 0, 0, 0, 0, 0, 0, 0, 0.15, 0.05, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0.1, 0.1, 0, 0, 0, 0, 0, 0.25, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0.35, 0, 0, 0, 0, 0, 0, 0],
    'OTH': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    'PEA': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    'SUN': [0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    'THE': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0],
    'WAT': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0],
    'WHT': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    'WIN': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1],
    'WST': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
}

# The index list (51 elements) for the DataFrame rows
index = ['ABHP','ALFC','ALKE','ASHP','BATS','BEVS','BSPG','CAES','COMC','COMCX','DMFC','GETH','GSHP','GTUR','GTURX','H2ST','HBBS','HDAM','HDAMC','HDLZ','HOBO','HOBOX','HPHS','HPHSC','HROR','HRORC','HYHP','ICEN','ICENX','MCFC','PAFC','P2BS','P2GS','P2HT','PEFC','PEME','PHOT','REFC','REHE','SCSP','SOFC','SOTH','SOXE','STUR','STURX','THMS','WAVE','WHEN','WTOF','WSHP','WTON']

# Create DataFrame
AT_fuel_technologies_match_df = pd.DataFrame(AT_fuel_technologies_match, index=index)

# Display the DataFrame
AT_fuel_technologies_match_df

,AIR,AMO,BIO,ELE,GAS,GEO,HRD,HYD,LIG,NUC,OIL,OTH,PEA,SUN,THE,WAT,WHT,WIN,WST
ABHP,0.95,0,0.00,0.05,0.00,0,0,0.00,0,0,0.00,0,0,0,0,0,0,0,0
ALFC,0.00,0,0.00,0.00,0.00,0,0,1.00,0,0,0.00,0,0,0,0,0,0,0,0
ALKE,0.00,0,0.00,1.00,0.00,0,0,0.00,0,0,0.00,0,0,0,0,0,0,0,0
ASHP,0.95,0,0.00,0.05,0.00,0,0,0.00,0,0,0.00,0,0,0,0,0,0,0,0
BATS,0.00,0,0.00,1.00,0.00,0,0,0.00,0,0,0.00,0,0,0,0,0,0,0,0
BEVS,0.00,0,0.00,1.00,0.00,0,0,0.00,0,0,0.00,0,0,0,0,0,0,0,0
BSPG,0.00,0,0.00,0.00,0.00,0,0,0.00,0,0,0.00,0,0,1,0,0,0,0,0
CAES,0.00,0,0.00,0.65,0.35,0,0,0.00,0,0,0.00,0,0,0,0,0,0,0,0
COMC,0.00,0,0.00,0.00,0.85,0,0,0.00,0,0,0.15,0,0,0,0,0,0,0,0
COMCX,0.00,0,0.45,0.00,0.50,0,0,0.00,0,0,0.05,0,0,0,0,0,0,0,0


<div style="background-color: black;">
<hr style="border: 0.5px solid skyblue;">
<div style="text-align: justify; margin-left: 0.0em; font-weight: normal; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
The Dispa-SET Technology - Fuel matching table for Poland.
<div style="text-align: justify; margin-left: 3.0em; font-weight: normal; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
Although the previous table can be used as support for all countries since it provides overall values from around the world, the following table specifically focuses on the PolaK Energy System.
The same was built using the following additional sources:
</div>
<div style="text-align: justify; margin-left: 5em; font-weight: normal; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
<ol style="margin-top: 0; padding-left: 1.5em;">
<li>
Ministerstwo Klimatu i Środowiska. (2021). Energy policy of Poland until 2040 (PEP2040). 
<br>
<a href="https://www.gov.pl/web/klimat/polityka-energetyczna-polski" style="color:skyblue">https://www.gov.pl/web/klimat/polityka-energetyczna-polski</a>
</li>
<li>
Polskie Sieci Elektroenergetyczne (PSE). (2023). 
Dane statystyczne [Statistical data]. 
<br>
<a href="https://www.pse.pl/dane-systemowe" style="color:skyblue">https://www.pse.pl/dane-systemowe</a>
</li>
<li>
Główny Urząd Statystyczny (Statistics Poland). (2023). 
Energy statistics. 
<br>
<a href="https://stat.gov.pl/en/topics/environment-energy/energy/" style="color:skyblue">https://stat.gov.pl/en/topics/environment-energy/energy/</a>
</li>
<li>
International Energy Agency. (2022). 
Poland 2022: Energy Policy Review. 
<br>
<a href="https://www.iea.org/reports/poland-2022" style="color:skyblue">https://www.iea.org/reports/poland-2022</a>
</li>
<li>
Eurostat. (2024). 
Energy database. European Commission.
<br>
<a href="https://ec.europa.eu/eurostat/web/energy/database" style="color:skyblue">https://ec.europa.eu/eurostat/web/energy/database</a>
</li>
</ol>
</div>
</div>
<hr style="border: 0.5px solid skyblue;">
</div>

In [309]:
PL_fuel_technologies_match = {
    'AIR': [0.95, 0, 0, 0.95, 0, 0, 0, 0, 0, 0, 0, 0, 0.95, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0.9, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0.95, 0],
    'AMO': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    'BIO': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0.15, 0, 0, 0, 0, 0.2, 0, 1, 0, 0, 0, 0.2, 0.25, 0, 0, 0, 0, 0, 0.15, 0.2, 0, 0, 0.25, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0.15, 0.2, 0, 0, 0, 0, 0, 0],
    'ELE': [0.05, 0, 1, 0.05, 1, 1, 0, 0, 0, 0, 0, 0, 0.05, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0.1, 0, 0, 0, 0, 0.6, 1, 1, 0, 1, 0, 0, 1, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0.05, 0],
    'GAS': [0, 0, 0, 0, 0, 0, 0, 0, 0.6, 0.7, 0, 0, 0, 0.7, 0.65, 0, 0, 0, 0, 0, 0.25, 0.4, 0, 0, 0, 0, 0, 0.2, 0.35, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0.2, 0.45, 0, 0, 0, 0, 0, 0],
    'GEO': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    'HRD': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0.25, 0.1, 0, 0, 0, 0, 0, 0.4, 0.25, 0, 0, 0, 0, 0, 0.45, 0.35, 0, 0, 0.15, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0.55, 0.3, 0, 0, 0, 0, 0, 0],
    'HYD': [0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0.05, 0.05, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0.05, 0.1, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0.05, 0, 0, 0, 0, 0, 0],
    'LIG': [0, 0, 0, 0, 0, 0, 0, 0, 0.4, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0.15, 0.1, 0, 0, 0, 0, 0, 0.15, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0.1, 0, 0, 0, 0, 0, 0, 0],
    'NUC': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    'OIL': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0.15, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    'OTH': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    'PEA': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    'SUN': [0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    'THE': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0],
    'WAT': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0],
    'WHT': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    'WIN': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1],
    'WST': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
}

# The index list (51 elements) for the DataFrame rows
index = ['ABHP','ALFC','ALKE','ASHP','BATS','BEVS','BSPG','CAES','COMC','COMCX','DMFC','GETH','GSHP','GTUR','GTURX','H2ST','HBBS','HDAM','HDAMC','HDLZ','HOBO','HOBOX','HPHS','HPHSC','HROR','HRORC','HYHP','ICEN','ICENX','MCFC','PAFC','P2BS','P2GS','P2HT','PEFC','PEME','PHOT','REFC','REHE','SCSP','SOFC','SOTH','SOXE','STUR','STURX','THMS','WAVE','WHEN','WTOF','WSHP','WTON']

# Create DataFrame
PL_fuel_technologies_match_df = pd.DataFrame(PL_fuel_technologies_match, index=index)

# Display the DataFrame
PL_fuel_technologies_match_df

,AIR,AMO,BIO,ELE,GAS,GEO,HRD,HYD,LIG,NUC,OIL,OTH,PEA,SUN,THE,WAT,WHT,WIN,WST
ABHP,0.95,0,0.00,0.05,0.00,0,0.00,0.00,0.00,0,0.00,0,0,0,0,0,0,0,0
ALFC,0.00,0,0.00,0.00,0.00,0,0.00,1.00,0.00,0,0.00,0,0,0,0,0,0,0,0
ALKE,0.00,0,0.00,1.00,0.00,0,0.00,0.00,0.00,0,0.00,0,0,0,0,0,0,0,0
ASHP,0.95,0,0.00,0.05,0.00,0,0.00,0.00,0.00,0,0.00,0,0,0,0,0,0,0,0
BATS,0.00,0,0.00,1.00,0.00,0,0.00,0.00,0.00,0,0.00,0,0,0,0,0,0,0,0
BEVS,0.00,0,0.00,1.00,0.00,0,0.00,0.00,0.00,0,0.00,0,0,0,0,0,0,0,0
BSPG,0.00,0,0.00,0.00,0.00,0,0.00,0.00,0.00,0,0.00,0,0,1,0,0,0,0,0
CAES,0.00,0,0.00,0.00,0.00,0,0.00,0.00,0.00,0,0.00,0,0,0,0,0,0,0,0
COMC,0.00,0,0.00,0.00,0.60,0,0.00,0.00,0.40,0,0.00,0,0,0,0,0,0,0,0
COMCX,0.00,0,0.15,0.00,0.70,0,0.00,0.00,0.00,0,0.15,0,0,0,0,0,0,0,0


<div style="background-color: black;">
<hr style="border: 0.5px solid skyblue;">
<div style="text-align: justify; margin-left: 0.0em; font-weight: normal; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
The Dispa-SET Technology - Fuel matching table for Slovenia.
<div style="text-align: justify; margin-left: 3.0em; font-weight: normal; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
Although the previous table can be used as support for all countries since it provides overall values from around the world, the following table specifically focuses on the Slove Energy System.
The same was built using the following additional sources:
</div>
<div style="text-align: justify; margin-left: 5em; font-weight: normal; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
<ol style="margin-top: 0; padding-left: 1.5em;">
<li>
xxxxxxxxxxxxxxxxxxxxxx. 
<br>
<a href="xxxxxxxxxxxxxxxxxxx" style="color:skyblue">xxxxxxxxxxxxxxxxxxxxxxi</a>
</li>
</ol>
</div>
</div>
<hr style="border: 0.5px solid skyblue;">
</div>

In [310]:
SI_fuel_technologies_match = {
    'AIR': [],
    'AMO': [],
    'BIO': [],
    'ELE': [],
    'GAS': [],
    'GEO': [],
    'HRD': [],
    'HYD': [],
    'LIG': [],
    'NUC': [],
    'OIL': [],
    'OTH': [],
    'PEA': [],
    'SUN': [],
    'THE': [],
    'WAT': [],
    'WHT': [],
    'WIN': [],
    'WST': []
}
index = ['ABHP','ALFC','ALKE','ASHP','BATS','BEVS','BSPG','CAES','COMC','COMCX','DMFC','GETH','GSHP','GTUR','GTURX','H2ST','HBBS','HDAM','HDAMC','HDLZ','HOBO','HOBOX','HPHS','HPHSC','HROR','HRORC','HYHP','ICEN','ICENX','MCFC','PAFC','P2BS','P2GS','P2HT','PEFC','PEME','PHOT','REFC','REHE','SCSP','SOFC','SOTH','SOXE','STUR','STURX','THMS','WAVE','WHEN','WTOF','WSHP','WTON']

# Create N/A DataFrame
SI_fuel_technologies_match_df = pd.DataFrame('N/A', index=index, columns=list(SI_fuel_technologies_match.keys()))

# Display the DataFrame
SI_fuel_technologies_match_df

,AIR,AMO,BIO,ELE,GAS,GEO,HRD,HYD,LIG,NUC,OIL,OTH,PEA,SUN,THE,WAT,WHT,WIN,WST
ABHP,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
ALFC,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
ALKE,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
ASHP,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
BATS,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
BEVS,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
BSPG,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
CAES,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
COMC,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
COMCX,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A


<div style="background-color: black;">
<hr style="border: 0.5px solid skyblue;">
<div style="text-align: justify; margin-left: 0.0em; font-weight: normal; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
The Dispa-SET Technology - Fuel matching table for Turkey.
<div style="text-align: justify; margin-left: 3.0em; font-weight: normal; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
Although the previous table can be used as support for all countries since it provides overall values from around the world, the following table specifically focuses on the Turk Energy System.
The same was built using the following additional sources:
</div>
<div style="text-align: justify; margin-left: 5em; font-weight: normal; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
<ol style="margin-top: 0; padding-left: 1.5em;">
<li>
International Energy Agency (IEA) - Comprehensive Analysis
<br>
International Energy Agency. (2021). Turkey 2021: Energy Policy Review. IEA. <a href="https://www.iea.org/reports/turkey-2021" style="color:skyblue">https://www.iea.org/reports/turkey-2021</a>
<br>
Provides detailed analysis of Turkey's entire energy system, policies, and future outlook.
</li>
<li>
Turkish Statistical Institute (TurkStat) - Official National Statistics
<br>
Turkish Statistical Institute. (2024). Energy statistics. <a href="https://data.tuik.gov.tr/Kategori/GetKategori?p=cevre-ve-enerji-116" style="color:skyblue">https://data.tuik.gov.tr/Kategori/GetKategori?p=cevre-ve-enerji-116</a>
<br>
Official source for comprehensive energy balance, consumption, and import/export data.
</li>
<li>
Ministry of Energy and Natural Resources - Policy Context
<br>
Ministry of Energy and Natural Resources. (2024). Energy policies and strategies. <a href="https://enerji.gov.tr/bilgi-merkezi-enerji-politikalari" style="color:skyblue">https://enerji.gov.tr/bilgi-merkezi-enerji-politikalari</a>
<br>
Provides the policy framework and strategic direction for Turkey's energy sector.
</li>
<li>
Eurostat - Comparative European Data
<br>
Eurostat. (2024). Energy database. European Commission. <a href="https://ec.europa.eu/eurostat/web/energy/data/database" style="color:skyblue">https://ec.europa.eu/eurostat/web/energy/data/database</a>
</li>
</ol>
</div>
</div>
<hr style="border: 0.5px solid skyblue;">
</div>

In [311]:
TR_fuel_technologies_match = {
    'AIR': [0.95, 0, 0, 0.95, 0, 0, 0, 0, 0, 0, 0, 0, 0.95, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0.9, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0.95, 0],
    'AMO': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    'BIO': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0.15, 0, 0, 0, 0, 0.2, 0, 1, 0, 0, 0, 0.25, 0.3, 0, 0, 0, 0, 0, 0.2, 0.25, 0, 0, 0.25, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0.15, 0.25, 0, 0, 0, 0, 0, 0],
    'ELE': [0.05, 0, 1, 0.05, 1, 1, 0, 0.65, 0, 0, 0, 0, 0.05, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0.1, 0, 0, 0, 0, 0.6, 1, 1, 0, 1, 0, 0, 1, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0.05, 0],
    'GAS': [0, 0, 0, 0, 0, 0, 0, 0.35, 0.85, 0.75, 0, 0, 0, 0.8, 0.65, 0, 0, 0, 0, 0, 0.6, 0.55, 0, 0, 0, 0, 0, 0.35, 0.5, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0.45, 0.6, 0, 0, 0, 0, 0, 0],
    'GEO': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    'HRD': [0, 0, 0, 0, 0, 0, 0, 0, 0.15, 0, 0, 0, 0, 0.15, 0.1, 0, 0, 0, 0, 0, 0.1, 0.1, 0, 0, 0, 0, 0, 0.25, 0.2, 0, 0, 0.15, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0.3, 0.1, 0, 0, 0, 0, 0, 0],
    'HYD': [0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0.05, 0.05, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0.05, 0.05, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0.05, 0, 0, 0, 0, 0, 0],
    'LIG': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0.05, 0.05, 0, 0, 0, 0, 0, 0.15, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0.1, 0, 0, 0, 0, 0, 0, 0],
    'NUC': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    'OIL': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0.1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    'OTH': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    'PEA': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    'SUN': [0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    'THE': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0],
    'WAT': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0],
    'WHT': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    'WIN': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1],
    'WST': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
}
index = ['ABHP','ALFC','ALKE','ASHP','BATS','BEVS','BSPG','CAES','COMC','COMCX','DMFC','GETH','GSHP','GTUR','GTURX','H2ST','HBBS','HDAM','HDAMC','HDLZ','HOBO','HOBOX','HPHS','HPHSC','HROR','HRORC','HYHP','ICEN','ICENX','MCFC','PAFC','P2BS','P2GS','P2HT','PEFC','PEME','PHOT','REFC','REHE','SCSP','SOFC','SOTH','SOXE','STUR','STURX','THMS','WAVE','WHEN','WTOF','WSHP','WTON']

# Create DataFrame
TR_fuel_technologies_match_df = pd.DataFrame(TR_fuel_technologies_match, index=index)

# Display the DataFrame
TR_fuel_technologies_match_df

,AIR,AMO,BIO,ELE,GAS,GEO,HRD,HYD,LIG,NUC,OIL,OTH,PEA,SUN,THE,WAT,WHT,WIN,WST
ABHP,0.95,0,0.00,0.05,0.00,0,0.00,0.00,0.00,0,0.0,0,0,0,0,0,0,0,0
ALFC,0.00,0,0.00,0.00,0.00,0,0.00,1.00,0.00,0,0.0,0,0,0,0,0,0,0,0
ALKE,0.00,0,0.00,1.00,0.00,0,0.00,0.00,0.00,0,0.0,0,0,0,0,0,0,0,0
ASHP,0.95,0,0.00,0.05,0.00,0,0.00,0.00,0.00,0,0.0,0,0,0,0,0,0,0,0
BATS,0.00,0,0.00,1.00,0.00,0,0.00,0.00,0.00,0,0.0,0,0,0,0,0,0,0,0
BEVS,0.00,0,0.00,1.00,0.00,0,0.00,0.00,0.00,0,0.0,0,0,0,0,0,0,0,0
BSPG,0.00,0,0.00,0.00,0.00,0,0.00,0.00,0.00,0,0.0,0,0,1,0,0,0,0,0
CAES,0.00,0,0.00,0.65,0.35,0,0.00,0.00,0.00,0,0.0,0,0,0,0,0,0,0,0
COMC,0.00,0,0.00,0.00,0.85,0,0.15,0.00,0.00,0,0.0,0,0,0,0,0,0,0,0
COMCX,0.00,0,0.15,0.00,0.75,0,0.00,0.00,0.00,0,0.1,0,0,0,0,0,0,0,0


<div style="background-color: black;">
<hr style="border: 0.5px solid skyblue;">
<div style="text-align: justify; margin-left: 0.0em; font-weight: normal; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
The Dispa-SET Technology - Fuel matching table for Ukraine.
<div style="text-align: justify; margin-left: 3.0em; font-weight: normal; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
Although the previous table can be used as support for all countries since it provides overall values from around the world, the following table specifically focuses on the Ukra Energy System.
The same was built using the following additional sources:
</div>
<div style="text-align: justify; margin-left: 5em; font-weight: normal; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
<ol style="margin-top: 0; padding-left: 1.5em;">
<li>
xxxxxxxxxxxxxxxxxxxxxx. 
<br>
<a href="xxxxxxxxxxxxxxxxxxx" style="color:skyblue">xxxxxxxxxxxxxxxxxxxxxxi</a>
</li>
</ol>
</div>
</div>
<hr style="border: 0.5px solid skyblue;">
</div>

In [312]:
UA_fuel_technologies_match = {
    'AIR': [],
    'AMO': [],
    'BIO': [],
    'ELE': [],
    'GAS': [],
    'GEO': [],
    'HRD': [],
    'HYD': [],
    'LIG': [],
    'NUC': [],
    'OIL': [],
    'OTH': [],
    'PEA': [],
    'SUN': [],
    'THE': [],
    'WAT': [],
    'WHT': [],
    'WIN': [],
    'WST': []
}
index = ['ABHP','ALFC','ALKE','ASHP','BATS','BEVS','BSPG','CAES','COMC','COMCX','DMFC','GETH','GSHP','GTUR','GTURX','H2ST','HBBS','HDAM','HDAMC','HDLZ','HOBO','HOBOX','HPHS','HPHSC','HROR','HRORC','HYHP','ICEN','ICENX','MCFC','PAFC','P2BS','P2GS','P2HT','PEFC','PEME','PHOT','REFC','REHE','SCSP','SOFC','SOTH','SOXE','STUR','STURX','THMS','WAVE','WHEN','WTOF','WSHP','WTON']

# Create N/A DataFrame
UA_fuel_technologies_match_df = pd.DataFrame('N/A', index=index, columns=list(UA_fuel_technologies_match.keys()))

# Display the DataFrame
UA_fuel_technologies_match_df

,AIR,AMO,BIO,ELE,GAS,GEO,HRD,HYD,LIG,NUC,OIL,OTH,PEA,SUN,THE,WAT,WHT,WIN,WST
ABHP,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
ALFC,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
ALKE,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
ASHP,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
BATS,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
BEVS,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
BSPG,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
CAES,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
COMC,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
COMCX,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A


<div style="background-color: black;">
<hr style="border: 0.5px solid skyblue;">
<div style="text-align: justify; margin-left: 3.0em; font-weight: bold; font-size: 17px; font-family: 'Times New Roman', serif; color: skyblue;">
6.5. Technology Match by Country Dictionary
</div>
<div style="text-align: justify; margin-left: 0.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
All the data frame of Fuel type proportions by Technology for each modeled counties is joining into a single dictionary.
</div>
<hr style="border: 0.5px solid skyblue;">
</div>

In [313]:
fuel_technologies_match_dict = {
    zone: globals()[f"{zone}_fuel_technologies_match_df"]
    for zone in zone_names
}

fuel_technologies_match_dict

{'BE':         AIR  AMO   BIO   ELE   GAS  GEO  HRD  HYD  LIG  NUC   OIL  OTH  PEA  \
 ABHP   0.05  0.0  0.00  0.00  0.00    0    0    0    0  0.0  0.00    0    0   
 ALFC   0.00  0.0  0.00  1.00  0.00    0    0    0    0  0.0  0.00    0    0   
 ALKE   0.00  0.0  0.00  1.00  0.00    0    0    0    0  0.0  0.00    0    0   
 ASHP   0.95  0.0  0.00  0.05  0.00    0    0    0    0  0.0  0.00    0    0   
 BATS   0.00  0.0  0.00  1.00  0.00    0    0    0    0  0.0  0.00    0    0   
 BEVS   0.00  0.0  0.00  1.00  0.00    0    0    0    0  0.0  0.00    0    0   
 BSPG   0.00  0.0  0.00  0.00  0.00    0    0    0    0  0.0  0.00    0    0   
 CAES   0.00  0.0  0.00  1.00  0.00    0    0    0    0  0.0  0.00    0    0   
 COMC   0.00  0.0  0.05  0.00  0.95    0    0    0    0  0.0  0.00    0    0   
 COMCX  0.00  0.0  0.15  0.00  0.80    0    0    0    0  0.0  0.00    0    0   
 DMFC   0.00  0.0  0.00  0.00  0.00    0    0    0    0  0.0  0.00    1    0   
 GETH   0.00  0.0  0.00  0.00  0.0

<div style="background-color: black;">
<hr style="border: 0.5px solid skyblue;">
<div style="text-align: right; margin-left: 3.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman;color:skyblue">
    Tracking Variables.
</div>
    <div style="text-align: right; margin-left: 1.50em; font-weight: unbold; font-size: 13px; font-family: TimesNewRoman;color:skyblue">
    This cells are just to confirm all the file names, file paths and other information related to the data being processed.
    <br>
  Also are used to ensure the inputs for next cells in order to avoid to re-enter the same information each time.
</div>
<hr style="border: 0.5px solid skyblue;">
</div>

In [314]:
print (f"Name of the DispaSET Unleash folder:                       {dispaSET_unleash_folder_name}\n")
print (f"Path to the DispaSET Unleash folder:                       {dispaSET_unleash_folder_path}\n")
print (f"Name of the Power Plants Base data folder:                 {power_plants_base_data_folder_name}\n")
print (f"Path to the Power Plants Base data folder:                 {power_plants_base_data_folder_path}\n")
print (f"Name of Power Plants_Pypsa Raw data folder:                {power_plants_pypsa_raw_data_folder_name}\n")
print (f"Path to the Power Plants_Pypsa Raw data folder:            {power_plants_pypsa_raw_data_folder_path}\n")
print (f"Name of the Power Plants_Pypsa Formated data folder:       {power_plants_pypsa_formated_data_folder_name}\n")
print (f"Path to the Power Plants_Pypsa Formated data folder:       {power_plants_pypsa_formated_data_folder_path}\n")
print (f"Name of the zones:                                         {zone_names}\n")
print (f"Target year:                                               {data_target_year}\n")
print (f"Name of the Power Plant DataFrames (dictionary):           {list(power_plants_dfs_dict.keys())}\n")
print (f"Name of the Technologies Equivalences Dictionary:          tech_equivalences_dict\n")
print (f"Name of the Overall Fuel Technology Match DataFrame:       overal_fuel_technologies_match_df\n")
print (f"Name of the Fuel Technology Match Dictionary:              fuel_technologies_match_dict\n")

Name of the DispaSET Unleash folder:                       Dispa-SET_Unleash

Path to the DispaSET Unleash folder:                       /home/ray/Dispa-SET_Unleash

Name of the Power Plants Base data folder:                 PowerPlants

Path to the Power Plants Base data folder:                 /home/ray/Dispa-SET_Unleash/Database/PowerPlants

Name of Power Plants_Pypsa Raw data folder:                PowerPlants

Path to the Power Plants_Pypsa Raw data folder:            /home/ray/Dispa-SET_Unleash/RawData_PyPSA/Reference_Scenario/PowerPlants

Name of the Power Plants_Pypsa Formated data folder:       PowerPlants

Path to the Power Plants_Pypsa Formated data folder:       /home/ray/Dispa-SET_Unleash/Database_PyPSA/Reference_Scenario/PowerPlants

Name of the zones:                                         ['BE', 'FR', 'DE', 'NL', 'UK']

Target year:                                               2030

Name of the Power Plant DataFrames (dictionary):           ['BE_2030', 'FR_2030', 'DE_

<div style="background-color: black;">
    <hr style="border: 2px solid skyblue;">
    <div style="text-align: justify; margin-left: 3.0em; font-weight: bold; font-size: 18px; font-family: TimesNewRoman; color:skyblue">
    7. Raw Data Uploading
    </div>
    <div style="text-align: justify; margin-left: 0.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
The nodal capacities PyPSA outputs are converted into data frame for its management.
    </div>
<hr style="border: 0.5px solid skyblue;">
</div>

In [315]:
file_path = os.path.join(power_plants_pypsa_raw_data_folder_path, "nodal_capacities.csv")

nodal_capacities_df = pd.read_csv(file_path)

nodal_capacities_df

,cluster,Unnamed: 1,Unnamed: 2,6,6.1,6.2
0,ll,NaN,NaN,vopt,vopt,vopt
1,opt,NaN,NaN,1H-T-H-B-I-A-dist1,1H-T-H-B-I-A-dist1,1H-T-H-B-I-A-dist1
2,planning_horizon,NaN,NaN,2030,2040,2050
3,generators,NaN,coal,33028.7100662858,20.157946290498,0.00439350191459953
4,generators,NaN,lignite,11143.9976777995,19.0295991828038,0.00367573147029184
...,...,...,...,...,...,...
652,stores,NL1 0,residential urban decentral water tanks,0.343476644057879,1.37948036679268,1.4466688695137
653,stores,NL1 0,services rural water tanks,1.7106336080301,4.92764035982957,4.9974419298418
654,stores,NL1 0,services urban decentral water tanks,0.409615202103847,0.786612903715892,0.852507027902251
655,stores,NL1 0,solid biomass,17227013.3536155,19944981.651406,22871222.6749819


<div style="background-color: black;">
<hr style="border: 0.5px solid skyblue;">
    <div style="text-align: justify; margin-left: 0.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
Setting the target year row as columns index.
    </div>
<hr style="border: 0.5px solid skyblue;">
</div>

In [316]:
target_int = int(data_target_year)

# --- Step 1: Convert row to numeric where possible, check for match ---
def row_contains_target_year(row):
    # convert row to numeric where possible
    numeric_row = pd.to_numeric(row, errors='coerce')
    if (numeric_row == target_int).any():
        return True
    # fallback: check string equality
    str_row = row.astype(str).str.strip()
    if (str_row == data_target_year).any():
        return True
    return False

# Find the first matching row
first_year_row_index = nodal_capacities_df.apply(row_contains_target_year, axis=1).idxmax()

# --- Step 2: Use that row as new headers ---
new_columns = nodal_capacities_df.loc[first_year_row_index].astype(str).tolist()

# --- Step 3: Drop the row and set new headers ---
df_with_new_headers = nodal_capacities_df.drop(first_year_row_index).copy()
df_with_new_headers.columns = new_columns
nodal_capacities_df = df_with_new_headers.reset_index(drop=True)

nodal_capacities_df

,planning_horizon,nan,nan,2030,2040,2050
0,ll,NaN,NaN,vopt,vopt,vopt
1,opt,NaN,NaN,1H-T-H-B-I-A-dist1,1H-T-H-B-I-A-dist1,1H-T-H-B-I-A-dist1
2,generators,NaN,coal,33028.7100662858,20.157946290498,0.00439350191459953
3,generators,NaN,lignite,11143.9976777995,19.0295991828038,0.00367573147029184
4,generators,NaN,load,1000000,1000000,1000000
...,...,...,...,...,...,...
651,stores,NL1 0,residential urban decentral water tanks,0.343476644057879,1.37948036679268,1.4466688695137
652,stores,NL1 0,services rural water tanks,1.7106336080301,4.92764035982957,4.9974419298418
653,stores,NL1 0,services urban decentral water tanks,0.409615202103847,0.786612903715892,0.852507027902251
654,stores,NL1 0,solid biomass,17227013.3536155,19944981.651406,22871222.6749819


<div style="background-color: black;">
<hr style="border: 0.5px solid skyblue;">
    <div style="text-align: justify; margin-left: 0.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
Naming undefined columns.
    </div>
<hr style="border: 0.5px solid skyblue;">
</div>

In [317]:
def is_unnamed_label(label):
    """Return True for NaN/empty/'nan'/'Unnamed...' column labels."""
    if pd.isna(label):
        return True
    s = str(label).strip()
    if s == '' or s.lower() == 'nan':
        return True
    if re.match(r'Unnamed', s):
        return True
    return False

# Assuming nodal_capacities_df is the DataFrame and zone_names is defined

cols = list(nodal_capacities_df.columns)
new_cols = []
zone_count = 0
tech_count = 0

for i, col_label in enumerate(cols):
    if is_unnamed_label(col_label):
        # read the column by position to avoid issues with NaN labels
        col_series = nodal_capacities_df.iloc[:, i].dropna().astype(str)

        # find any zone substring (case-sensitive)
        found_zone = False
        for val in col_series:
            # skip obvious 'nan' strings
            if val.strip().lower() == 'nan':
                continue
            for zone in zone_names:
                if zone in val:      # case-sensitive substring match
                    found_zone = True
                    break
            if found_zone:
                break

        if found_zone:
            zone_count += 1
            new_name = 'zone' if zone_count == 1 else f'zone_{zone_count}'
        else:
            tech_count += 1
            new_name = 'tech' if tech_count == 1 else f'tech_{tech_count}'

        new_cols.append(new_name)
    else:
        new_cols.append(col_label)

# apply the new column names determined above
nodal_capacities_df.columns = new_cols

# Change the header of the first column (index 0) to 'Element'
nodal_capacities_df.columns.values[0] = 'Element'
# --- NEW CODE ADDITION END ---

# quick check
print(list(zip(cols, nodal_capacities_df.columns)))

nodal_capacities_df

[('planning_horizon', 'Element'), ('nan', 'zone'), ('nan', 'tech'), ('2030', '2030'), ('2040', '2040'), ('2050', '2050')]


,Element,zone,tech,2030,2040,2050
0,ll,NaN,NaN,vopt,vopt,vopt
1,opt,NaN,NaN,1H-T-H-B-I-A-dist1,1H-T-H-B-I-A-dist1,1H-T-H-B-I-A-dist1
2,generators,NaN,coal,33028.7100662858,20.157946290498,0.00439350191459953
3,generators,NaN,lignite,11143.9976777995,19.0295991828038,0.00367573147029184
4,generators,NaN,load,1000000,1000000,1000000
...,...,...,...,...,...,...
651,stores,NL1 0,residential urban decentral water tanks,0.343476644057879,1.37948036679268,1.4466688695137
652,stores,NL1 0,services rural water tanks,1.7106336080301,4.92764035982957,4.9974419298418
653,stores,NL1 0,services urban decentral water tanks,0.409615202103847,0.786612903715892,0.852507027902251
654,stores,NL1 0,solid biomass,17227013.3536155,19944981.651406,22871222.6749819


<div style="background-color: black;">
<hr style="border: 0.5px solid skyblue;">
    <div style="text-align: justify; margin-left: 0.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
Filtering all the rows not related with the selected countries.
    </div>
<hr style="border: 0.5px solid skyblue;">
</div>

In [318]:
# Function to check if a row contains any zone substring
def row_contains_zone(row, zones):
    for val in row.astype(str):
        for zone in zones:
            if zone in val:  # case-sensitive substring match
                return True
    return False

# Filter rows
mask = nodal_capacities_df.apply(lambda row: row_contains_zone(row, zone_names), axis=1)
nodal_capacities_df = nodal_capacities_df[mask].reset_index(drop=True)

nodal_capacities_df

,Element,zone,tech,2030,2040,2050
0,generators,BE1 0,gas,39563.0509463557,39563.0509463557,39563.0509463557
1,generators,BE1 0,load,31000000,30000000,28000000
2,generators,BE1 0,offwind,2261.8,2261.8,1549.8
3,generators,BE1 0,offwind-ac,1738.19937359281,1738.19937359281,1738.19937359281
4,generators,BE1 0,offwind-dc,1999.99944646756,4000.00050712006,4712.00052697313
...,...,...,...,...,...,...
626,stores,NL1 0,residential urban decentral water tanks,0.343476644057879,1.37948036679268,1.4466688695137
627,stores,NL1 0,services rural water tanks,1.7106336080301,4.92764035982957,4.9974419298418
628,stores,NL1 0,services urban decentral water tanks,0.409615202103847,0.786612903715892,0.852507027902251
629,stores,NL1 0,solid biomass,17227013.3536155,19944981.651406,22871222.6749819


<div style="background-color: black;">
<hr style="border: 0.5px solid skyblue;">
    <div style="text-align: justify; margin-left: 0.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
Combining the columnes 'Element' and 'tech' to adequate with the Technologies Dictionary nomenclature.
    </div>
<hr style="border: 0.5px solid skyblue;">
</div>

In [319]:
# Join 'Element' and 'tech' with a space
nodal_capacities_df['tech'] = nodal_capacities_df['Element'] + ' ' + nodal_capacities_df['tech']

# Drop the 'Element' column
nodal_capacities_df = nodal_capacities_df.drop(columns=['Element'])

nodal_capacities_df

,zone,tech,2030,2040,2050
0,BE1 0,generators gas,39563.0509463557,39563.0509463557,39563.0509463557
1,BE1 0,generators load,31000000,30000000,28000000
2,BE1 0,generators offwind,2261.8,2261.8,1549.8
3,BE1 0,generators offwind-ac,1738.19937359281,1738.19937359281,1738.19937359281
4,BE1 0,generators offwind-dc,1999.99944646756,4000.00050712006,4712.00052697313
...,...,...,...,...,...
626,NL1 0,stores residential urban decentral water tanks,0.343476644057879,1.37948036679268,1.4466688695137
627,NL1 0,stores services rural water tanks,1.7106336080301,4.92764035982957,4.9974419298418
628,NL1 0,stores services urban decentral water tanks,0.409615202103847,0.786612903715892,0.852507027902251
629,NL1 0,stores solid biomass,17227013.3536155,19944981.651406,22871222.6749819


<div style="background-color: black;">
<hr style="border: 0.5px solid skyblue;">
    <div style="text-align: justify; margin-left: 0.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
Filtering the target year.
    </div>
<hr style="border: 0.5px solid skyblue;">
</div>

In [320]:
# list of columns to keep
cols_to_keep = [col for col in nodal_capacities_df.columns 
                if str(col).startswith("zone") 
                or str(col).startswith("tech") 
                or str(col) == str(data_target_year)]

# filter dataframe
nodal_capacities_df = nodal_capacities_df[cols_to_keep].copy()

nodal_capacities_df

,zone,tech,2030
0,BE1 0,generators gas,39563.0509463557
1,BE1 0,generators load,31000000
2,BE1 0,generators offwind,2261.8
3,BE1 0,generators offwind-ac,1738.19937359281
4,BE1 0,generators offwind-dc,1999.99944646756
...,...,...,...
626,NL1 0,stores residential urban decentral water tanks,0.343476644057879
627,NL1 0,stores services rural water tanks,1.7106336080301
628,NL1 0,stores services urban decentral water tanks,0.409615202103847
629,NL1 0,stores solid biomass,17227013.3536155


<div style="background-color: black;">
<hr style="border: 0.5px solid skyblue;">
    <div style="text-align: justify; margin-left:0.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
Cleaning the zone names.
    </div>
<hr style="border: 0.5px solid skyblue;">
</div>

In [321]:
def replace_with_zone(cell, zones):
    cell_str = str(cell)  # ensure string
    for zone in zones:
        if zone in cell_str:  # case-sensitive substring match
            return zone       # replace whole cell with the matched zone
    return cell  # keep original if no match

# Apply only to the 'zone' column
nodal_capacities_df['zone'] = nodal_capacities_df['zone'].apply(lambda x: replace_with_zone(x, zone_names))

nodal_capacities_df

,zone,tech,2030
0,BE,generators gas,39563.0509463557
1,BE,generators load,31000000
2,BE,generators offwind,2261.8
3,BE,generators offwind-ac,1738.19937359281
4,BE,generators offwind-dc,1999.99944646756
...,...,...,...
626,NL,stores residential urban decentral water tanks,0.343476644057879
627,NL,stores services rural water tanks,1.7106336080301
628,NL,stores services urban decentral water tanks,0.409615202103847
629,NL,stores solid biomass,17227013.3536155


<div style="background-color: black;">
<hr style="border: 0.5px solid skyblue;">
    <div style="text-align: justify; margin-left: 0.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
Dividing the capacities file into the selected countries.
    </div>
<hr style="border: 0.5px solid skyblue;">
</div>

In [322]:
# Create dictionary to store the derived DataFrames
nodal_capacities_dfs_dict = {}

for zone in zone_names:
    # Filter rows where 'zone' matches the current zone
    df_zone = nodal_capacities_df[nodal_capacities_df['zone'] == zone].copy()
    
    # Save into dictionary with a descriptive key
    nodal_capacities_dfs_dict[f"{zone}_nodal_capacities_df"] = df_zone

nodal_capacities_dfs_dict

{'BE_nodal_capacities_df':     zone                                            tech               2030
 0     BE                                  generators gas   39563.0509463557
 1     BE                                 generators load           31000000
 2     BE                              generators offwind             2261.8
 3     BE                           generators offwind-ac   1738.19937359281
 4     BE                           generators offwind-dc   1999.99944646756
 ..   ...                                             ...                ...
 561   BE  stores residential urban decentral water tanks  0.620639625874326
 562   BE               stores services rural water tanks   1.00436547503266
 563   BE     stores services urban decentral water tanks  0.680430055032749
 564   BE                            stores solid biomass   20944781.4561726
 565   BE                stores urban central water tanks   714893.437595446
 
 [106 rows x 3 columns],
 'FR_nodal_capacities_d

<div style="background-color: black;">
<hr style="border: 0.5px solid skyblue;">
    <div style="text-align: justify; margin-left: 0.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
Aggregating repeated tech items.
    </div>
<hr style="border: 0.5px solid skyblue;">
</div>

In [323]:
# Convert the integer year to a string to match the column name
data_target_year_str = str(data_target_year)

# Create an empty DataFrame to store the rows that will be erased
erased_rows_df = pd.DataFrame()

for df_name in nodal_capacities_dfs_dict:
    df = nodal_capacities_dfs_dict[df_name]

    # Convert the target year column to numeric, ignoring the header
    df[data_target_year_str] = pd.to_numeric(df[data_target_year_str], errors='coerce')

    # Identify duplicate rows based on 'zone' and 'tech', keeping the first occurrence
    duplicates = df.duplicated(subset=['zone', 'tech'], keep='first')

    # Select the rows that are duplicates (i.e., those to be erased)
    erased_part = df[duplicates].copy()

    # Append these erased rows to the new DataFrame
    erased_rows_df = pd.concat([erased_rows_df, erased_part], ignore_index=True)

    # Group by 'zone' and 'tech', then sum the target year column
    grouped_df = df.groupby(['zone', 'tech'], as_index=False)[data_target_year_str].sum()

    # Update the dictionary with the processed DataFrame
    nodal_capacities_dfs_dict[df_name] = grouped_df

# Verify the result
print("\n--- Processed DataFrames ---")
for df_name, df in nodal_capacities_dfs_dict.items():
    print(f"\nDataFrame: {df_name}")
    print(df)

print("\n--- Erased Rows DataFrame ---")
print(erased_rows_df)


--- Processed DataFrames ---

DataFrame: BE_nodal_capacities_df
    zone                                            tech          2030
0     BE                                  generators gas  3.956305e+04
1     BE                                 generators load  3.100000e+07
2     BE                              generators offwind  2.261800e+03
3     BE                           generators offwind-ac  1.738199e+03
4     BE                           generators offwind-dc  1.999999e+03
..   ...                                             ...           ...
101   BE  stores residential urban decentral water tanks  6.206396e-01
102   BE               stores services rural water tanks  1.004365e+00
103   BE     stores services urban decentral water tanks  6.804301e-01
104   BE                            stores solid biomass  2.094478e+07
105   BE                stores urban central water tanks  7.148934e+05

[106 rows x 3 columns]

DataFrame: FR_nodal_capacities_df
    zone                

<div style="background-color: black;">
<hr style="border: 0.5px solid skyblue;">
<div style="text-align: right; margin-left: 3.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman;color:skyblue">
    Tracking Variables.
</div>
    <div style="text-align: right; margin-left: 1.50em; font-weight: unbold; font-size: 13px; font-family: TimesNewRoman;color:skyblue">
    This cells are just to confirm all the file names, file paths and other information related to the data being processed.
    <br>
  Also are used to ensure the inputs for next cells in order to avoid to re-enter the same information each time.
</div>
<hr style="border: 0.5px solid skyblue;">
</div>

In [324]:
print (f"Name of the DispaSET Unleash folder:                       {dispaSET_unleash_folder_name}\n")
print (f"Path to the DispaSET Unleash folder:                       {dispaSET_unleash_folder_path}\n")
print (f"Name of the Power Plants Base data folder:                 {power_plants_base_data_folder_name}\n")
print (f"Path to the Power Plants Base data folder:                 {power_plants_base_data_folder_path}\n")
print (f"Name of Power Plants_Pypsa Raw data folder:                {power_plants_pypsa_raw_data_folder_name}\n")
print (f"Path to the Power Plants_Pypsa Raw data folder:            {power_plants_pypsa_raw_data_folder_path}\n")
print (f"Name of the Power Plants_Pypsa Formated data folder:       {power_plants_pypsa_formated_data_folder_name}\n")
print (f"Path to the Power Plants_Pypsa Formated data folder:       {power_plants_pypsa_formated_data_folder_path}\n")
print (f"Name of the zones:                                         {zone_names}\n")
print (f"Target year:                                               {data_target_year}\n")
print (f"Name of the Power Plant DataFrames (dictionary):           {list(power_plants_dfs_dict.keys())}\n")
print (f"Name of the Technologies Equivalences Dictionary:          tech_equivalences_dict\n")
print (f"Name of the Overall Fuel Technology Match DataFrame:       overal_fuel_technologies_match_df\n")
print (f"Name of the Fuel Technology Match Dictionary:              fuel_technologies_match_dict\n")
print (f"Name of the PyPSA Nodal Capacities DataFrames (dictionary):{list(nodal_capacities_dfs_dict.keys())}\n")

Name of the DispaSET Unleash folder:                       Dispa-SET_Unleash

Path to the DispaSET Unleash folder:                       /home/ray/Dispa-SET_Unleash

Name of the Power Plants Base data folder:                 PowerPlants

Path to the Power Plants Base data folder:                 /home/ray/Dispa-SET_Unleash/Database/PowerPlants

Name of Power Plants_Pypsa Raw data folder:                PowerPlants

Path to the Power Plants_Pypsa Raw data folder:            /home/ray/Dispa-SET_Unleash/RawData_PyPSA/Reference_Scenario/PowerPlants

Name of the Power Plants_Pypsa Formated data folder:       PowerPlants

Path to the Power Plants_Pypsa Formated data folder:       /home/ray/Dispa-SET_Unleash/Database_PyPSA/Reference_Scenario/PowerPlants

Name of the zones:                                         ['BE', 'FR', 'DE', 'NL', 'UK']

Target year:                                               2030

Name of the Power Plant DataFrames (dictionary):           ['BE_2030', 'FR_2030', 'DE_

<div style="background-color: black;">
    <hr style="border: 2px solid skyblue;">
    <div style="text-align: justify; margin-left: 3.0em; font-weight: bold; font-size: 18px; font-family: TimesNewRoman; color:skyblue">
    8. PyPSA to Dispa-SET Power Plants Data Formatting
    </div>
    <div style="text-align: justify; margin-left: 0.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
The power plants dataframe start to be formatting.
    </div>
    <hr style="border: 1px dashed skyblue;">
    <div style="text-align: justify; margin-left: 3.0em; font-weight: bold; font-size: 17px; font-family: TimesNewRoman; color:skyblue">
    8.1. Unit, PowerCapacity, Nunits and Zone Data Clasification
    </div>
    <div style="text-align: justify; margin-left: 0.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
The columns Unit, PowerCapacity, Nunits, Zone, Technology and Fuel of the final Power Plants inputs are added to the final Power Plant sheet.
    <br>
    Just the Unit, PowerCapacity, Nunits, Zone are correctly filtered at this stage.
    </div>
<hr style="border: 0.5px solid skyblue;">
</div>

In [325]:
for zone in zone_names:
    # Construct keys for both dictionaries
    nodal_key = f"{zone}_nodal_capacities_df"
    power_key = f"{zone}_{data_target_year}"

    # Get the corresponding dataframes
    nodal_df = nodal_capacities_dfs_dict.get(nodal_key)
    power_df = power_plants_dfs_dict.get(power_key)

    if nodal_df is not None and power_df is not None:
        new_rows = []

        for _, row in nodal_df.iterrows():
            tech = row['tech']
            # Check if tech is in the equivalence dictionary
            if tech in tech_equivalences_dict:
                # Safely access the column using .loc
                power_capacity = row.loc[str(data_target_year)]

                # Create a new row with the required values
                new_row = {
                    'Unit': tech,
                    'PowerCapacity': power_capacity,
                    'Nunits': 1,
                    'Zone': row['zone'],
                    # MODIFIED LINES BELOW:
                    'Technology': ' '.join(tech_equivalences_dict[tech]['tech']) if isinstance(tech_equivalences_dict[tech]['tech'], list) else tech_equivalences_dict[tech]['tech'],
                    'Fuel': ' '.join(tech_equivalences_dict[tech]['fuel']) if isinstance(tech_equivalences_dict[tech]['fuel'], list) else tech_equivalences_dict[tech]['fuel']
                }

                new_rows.append(new_row)

        # Concatenate new rows to the existing dataframe
        if new_rows:
            power_df = pd.concat([power_df, pd.DataFrame(new_rows)], ignore_index=True)

        # Update the dictionary with the modified dataframe
        power_plants_dfs_dict[power_key] = power_df

power_plants_dfs_dict

/tmp/ipykernel_2590693/3897183125.py:35: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  power_df = pd.concat([power_df, pd.DataFrame(new_rows)], ignore_index=True)
/tmp/ipykernel_2590693/3897183125.py:35: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  power_df = pd.concat([power_df, pd.DataFrame(new_rows)], ignore_index=True)
/tmp/ipykernel_2590693/3897183125.py:35: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no 

{'BE_2030':    Unnamed: 0                                      Unit  PowerCapacity Nunits  \
 0         NaN                        generators offwind    2261.800000      1   
 1         NaN                     generators offwind-ac    1738.199374      1   
 2         NaN                     generators offwind-dc    1999.999446      1   
 3         NaN                         generators onwind    5999.998849      1   
 4         NaN                            generators ror      59.015340      1   
 5         NaN                          generators solar   11999.999412      1   
 6         NaN                  generators solar rooftop    1999.996368      1   
 7         NaN                                links CCGT    6279.540144      1   
 8         NaN                                 links DAC       0.007254      1   
 9         NaN                     links H2 Electrolysis     150.000000      1   
 10        NaN                        links H2 Fuel Cell     126.632530      1   
 11  

<div style="background-color: black;">
    <hr style="border: 1px solid skyblue;">
    <div style="text-align: justify; margin-left: 3.0em; font-weight: bold; font-size: 17px; font-family: TimesNewRoman; color:skyblue">
    8.2. Technology Data Clasification
    </div>
    <div style="text-align: justify; margin-left: 0.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
    The column technology is separating for all those units that can be matched with more than one technology.
    <br>
    The desegregation of the power capacity will be done according the following rule / proportion.
    </div>
    <div style="text-align: center; font-weight: bold; font-size: 20px; font-family: TimesNewRoman; color:skyblue; margin-top: 10px; margin-bottom: 10px;">
    $E_k = \frac{2^{N-k}}{2^N - 1}$
    </div>
    <div style="text-align: justify; margin-left: 0.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
    Where $k$ is the position of the element / Technology, $k=1$ is the first, $k=N$ is the last
    </div>
<hr style="border: 0.5px solid skyblue;">
</div>

In [326]:
# Function to compute proportional weights
def compute_proportions(n):
    weights = [2**(n - k - 1) for k in range(n)]
    total = sum(weights)
    return [w / total for w in weights]

# Process each DataFrame in the dictionary
for zone in zone_names:
    key = f"{zone}_{data_target_year}"
    df = power_plants_dfs_dict.get(key)

    if df is not None:
        new_rows = []

        for _, row in df.iterrows():
            techs = str(row['Technology']).split()
            if len(techs) > 1:
                proportions = compute_proportions(len(techs))
                for idx, (tech, prop) in enumerate(zip(techs, proportions), start=1):
                    new_row = row.copy()
                    new_row['Technology'] = tech
                    new_row['Unit'] = f"{row['Unit']}_{idx}"
                    new_row['PowerCapacity'] = row['PowerCapacity'] * prop
                    new_rows.append(new_row)
            else:
                new_rows.append(row)

        # Replace the original DataFrame with the expanded one
        power_plants_dfs_dict[key] = pd.DataFrame(new_rows)

power_plants_dfs_dict

{'BE_2030':     Unnamed: 0                                      Unit  PowerCapacity  \
 0          NaN                        generators offwind    2261.800000   
 1          NaN                     generators offwind-ac    1738.199374   
 2          NaN                     generators offwind-dc    1999.999446   
 3          NaN                         generators onwind    5999.998849   
 4          NaN                            generators ror      59.015340   
 5          NaN                          generators solar   11999.999412   
 6          NaN                  generators solar rooftop    1999.996368   
 7          NaN                                links CCGT    6279.540144   
 8          NaN                                 links DAC       0.007254   
 9          NaN                   links H2 Electrolysis_1      85.714286   
 9          NaN                   links H2 Electrolysis_2      42.857143   
 9          NaN                   links H2 Electrolysis_3      21.428571   
 

<div style="background-color: black;">
    <hr style="border: 1px solid skyblue;">
    <div style="text-align: justify; margin-left: 3.0em; font-weight: bold; font-size: 17px; font-family: TimesNewRoman; color:skyblue">
    8.3. Fuel Data Clasification
    </div>
    <div style="text-align: justify; margin-left: 0.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
    The column Feul is separating for all those units that can be matched with more than one fuel.
    <br>
    The desegregation of the power capacity will be done according the proportional technology - fuel tables for the corresponding country .
    </div>
<hr style="border: 0.5px solid skyblue;">
</div>

In [327]:
processed_power_plants_dfs_dict = {}

for zone in zone_names:
    power_key = f"{zone}_{data_target_year}"
    fuel_key = zone
    power_df = power_plants_dfs_dict.get(power_key)
    fuel_df = fuel_technologies_match_dict.get(fuel_key, overal_fuel_technologies_match_df)

    if power_df is None:
        print(f"⚠️ No power plant DataFrame found for {power_key}")
        continue

    df = power_df.copy()
    new_rows = []
    rows_to_drop = []

    for idx, row in df.iterrows():
        tech = str(row["Technology"]).strip()
        fuel_str = str(row["Fuel"]).strip()
        fuels = fuel_str.split()

        if len(fuels) <= 1:
            continue

        if tech not in fuel_df.index:
            rows_to_drop.append(idx)
            continue

        original_capacity = row["PowerCapacity"]
        new_capacities = []
        new_rows_for_this_row = []

        for i, fuel in enumerate(fuels, start=1):
            if fuel not in fuel_df.columns:
                continue
            multiplier = fuel_df.loc[tech, fuel]
            if pd.isna(multiplier):
                continue
            new_row = row.copy()
            new_row["Unit"] = f"{row['Unit']}_{i}"
            new_row["Fuel"] = fuel
            new_row["PowerCapacity"] = original_capacity * multiplier
            new_rows_for_this_row.append(new_row)
            new_capacities.append(new_row["PowerCapacity"])

        if not new_rows_for_this_row:
            rows_to_drop.append(idx)
            continue

        # Sum the new capacities
        sum_new_capacities = sum(new_capacities)
        # If the sum is less than the original, add the difference to the largest new capacity
        if sum_new_capacities < original_capacity:
            max_capacity_idx = max(range(len(new_capacities)), key=lambda i: new_capacities[i])
            new_rows_for_this_row[max_capacity_idx]["PowerCapacity"] += (original_capacity - sum_new_capacities)

        new_rows.extend(new_rows_for_this_row)
        rows_to_drop.append(idx)

    df = df.drop(rows_to_drop, errors="ignore")
    if new_rows:
        df = pd.concat([df, pd.DataFrame(new_rows)], ignore_index=True)

    processed_power_plants_dfs_dict[power_key] = df

print("✅ Processing complete. Updated DataFrames stored in 'processed_power_plants_dfs_dict'.")

processed_power_plants_dfs_dict

✅ Processing complete. Updated DataFrames stored in 'processed_power_plants_dfs_dict'.


{'BE_2030':     Unnamed: 0                                      Unit  PowerCapacity  \
 0          NaN                        generators offwind    2261.800000   
 1          NaN                     generators offwind-ac    1738.199374   
 2          NaN                     generators offwind-dc    1999.999446   
 3          NaN                         generators onwind    5999.998849   
 4          NaN                            generators ror      59.015340   
 5          NaN                          generators solar   11999.999412   
 6          NaN                  generators solar rooftop    1999.996368   
 7          NaN                                 links DAC       0.007254   
 8          NaN                   links H2 Electrolysis_1      85.714286   
 9          NaN                   links H2 Electrolysis_2      42.857143   
 10         NaN                   links H2 Electrolysis_3      21.428571   
 11         NaN                      links H2 Fuel Cell_1      63.814818   
 

<div style="background-color: black;">
<hr style="border: 0.5px solid skyblue;">
<div style="text-align: right; margin-left: 3.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman;color:skyblue">
    Tracking Variables.
</div>
    <div style="text-align: right; margin-left: 1.50em; font-weight: unbold; font-size: 13px; font-family: TimesNewRoman;color:skyblue">
    This cells are just to confirm all the file names, file paths and other information related to the data being processed.
    <br>
  Also are used to ensure the inputs for next cells in order to avoid to re-enter the same information each time.
</div>
<hr style="border: 0.5px solid skyblue;">
</div>

In [328]:
print (f"Name of the DispaSET Unleash folder:                       {dispaSET_unleash_folder_name}\n")
print (f"Path to the DispaSET Unleash folder:                       {dispaSET_unleash_folder_path}\n")
print (f"Name of the Power Plants Base data folder:                 {power_plants_base_data_folder_name}\n")
print (f"Path to the Power Plants Base data folder:                 {power_plants_base_data_folder_path}\n")
print (f"Name of Power Plants_Pypsa Raw data folder:                {power_plants_pypsa_raw_data_folder_name}\n")
print (f"Path to the Power Plants_Pypsa Raw data folder:            {power_plants_pypsa_raw_data_folder_path}\n")
print (f"Name of the Power Plants_Pypsa Formated data folder:       {power_plants_pypsa_formated_data_folder_name}\n")
print (f"Path to the Power Plants_Pypsa Formated data folder:       {power_plants_pypsa_formated_data_folder_path}\n")
print (f"Name of the zones:                                         {zone_names}\n")
print (f"Target year:                                               {data_target_year}\n")
print (f"Name of the Power Plant DataFrames (dictionary):           {list(power_plants_dfs_dict.keys())}\n")
print (f"Name of the Technologies Equivalences Dictionary:          tech_equivalences_dict\n")
print (f"Name of the Overall Fuel Technology Match DataFrame:       overal_fuel_technologies_match_df\n")
print (f"Name of the Fuel Technology Match Dictionary:              fuel_technologies_match_dict\n")
print (f"Name of the PyPSA Nodal Capacities DataFrames (dictionary):{list(nodal_capacities_dfs_dict.keys())}\n")
print (f"Name of the Processed Power Plant DataFrames (dictionary): {list(processed_power_plants_dfs_dict.keys())}\n")

Name of the DispaSET Unleash folder:                       Dispa-SET_Unleash

Path to the DispaSET Unleash folder:                       /home/ray/Dispa-SET_Unleash

Name of the Power Plants Base data folder:                 PowerPlants

Path to the Power Plants Base data folder:                 /home/ray/Dispa-SET_Unleash/Database/PowerPlants

Name of Power Plants_Pypsa Raw data folder:                PowerPlants

Path to the Power Plants_Pypsa Raw data folder:            /home/ray/Dispa-SET_Unleash/RawData_PyPSA/Reference_Scenario/PowerPlants

Name of the Power Plants_Pypsa Formated data folder:       PowerPlants

Path to the Power Plants_Pypsa Formated data folder:       /home/ray/Dispa-SET_Unleash/Database_PyPSA/Reference_Scenario/PowerPlants

Name of the zones:                                         ['BE', 'FR', 'DE', 'NL', 'UK']

Target year:                                               2030

Name of the Power Plant DataFrames (dictionary):           ['BE_2030', 'FR_2030', 'DE_

<div style="background-color: black;">
<hr style="border: 0.5px solid skyblue;">
    <div style="text-align: justify; margin-left: 0.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
Erasing rows whit nule Power Capacity.
    </div>
<hr style="border: 0.5px solid skyblue;">
</div>

In [329]:
# Iterate over each key in the dictionary
for key in list(processed_power_plants_dfs_dict.keys()):
    # Filter out rows where 'PowerCapacity' is 0
    processed_power_plants_dfs_dict[key] = processed_power_plants_dfs_dict[key][
        processed_power_plants_dfs_dict[key]['PowerCapacity'] != 0
    ]

processed_power_plants_dfs_dict

{'BE_2030':     Unnamed: 0                                      Unit  PowerCapacity  \
 0          NaN                        generators offwind    2261.800000   
 1          NaN                     generators offwind-ac    1738.199374   
 2          NaN                     generators offwind-dc    1999.999446   
 3          NaN                         generators onwind    5999.998849   
 4          NaN                            generators ror      59.015340   
 5          NaN                          generators solar   11999.999412   
 6          NaN                  generators solar rooftop    1999.996368   
 7          NaN                                 links DAC       0.007254   
 8          NaN                   links H2 Electrolysis_1      85.714286   
 9          NaN                   links H2 Electrolysis_2      42.857143   
 10         NaN                   links H2 Electrolysis_3      21.428571   
 11         NaN                      links H2 Fuel Cell_1      63.814818   
 

<div style="background-color: black;">
  <hr style="border: 1px solid skyblue;">
  <div style="text-align: justify; margin-left: 3.0em; font-weight: bold; font-size: 17px; font-family: 'Times New Roman'; color: skyblue;">
    8.4. CHP Units Classification
  </div>
  <div style="text-align: justify; margin-left: 0.0em; font-weight: normal; font-size: 14px; font-family: 'Times New Roman'; color: skyblue;">
    The CHP units in Dispaset have the following features:
  </div>
    
  <div style="text-align: justify; margin-left: 0.0em; font-weight: normal; font-size: 11px; font-family: 'Times New Roman'; color: skyblue;">
    <span style="display: inline-block; width: 12em; font-weight: bold; font-family: 'Times New Roman'; color: skyblue">CHPType</span>
    : This defines the operational characteristic of the CHP plant. It determines how the plant can vary its electricity and heat output.
    <br>
    <span style="display: inline-block; width: 12em; font-weight: bold; font-family: 'Times New Roman'; color: skyblue;">CHPPowerToHeat</span>
    : Also known as the Power-to-Heat Ratio; σ. It's the ratio of electrical power output to useful heat output.
$$ \sigma = \frac{\text{Power Output}}{\text{Heat Output}} $$
    <span style="display: inline-block; width: 12em; font-weight: bold; font-family: 'Times New Roman'; color: skyblue;">CHPPowerLossFactor</span>
    : It quantifies the amount of electrical power lost per unit of increased heat extraction. It represents the trade-off, i.e., producing more heat—by extracting steam—results in a loss of potential electricity generation.
    <br>
    <span style="display: inline-block; width: 12em; font-weight: bold; font-family: 'Times New Roman'; color: skyblue;">CHPMaxHeat</span>
    : The highest amount of useful thermal energy—in MWth—that the unit can produce, determined by the size of its heat exchangers and district heating connections.
  </div>

<hr style="border: 2px dashed skyblue;">
    <div style="text-align: justify; margin-left: 3.0em; font-weight: bold; font-size: 16px; font-family: TimesNewRoman; color:skyblue">
    8.4.1. CHP Units Nomenclature - CHP Technology List
    </div>
    <div style="text-align: justify; margin-left: 0.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
    The following technologies can be set as CHP units:
    <br>
<hr style="border: 0.5px solid skyblue;">
</div>

In [330]:
# Define all the Dispa-SET CHP technology lists from the common.py script of Dispa-SET core scripts
chp_technologies_set = ['COMC', 'COMCX', 'GTUR', 'GTURX', 'ICEN', 'ICENX', 'STUR', 'STURX', 'MCFC', 'PAFC', 'PEFC', 'SOFC']

# Convert the set back to a sorted list
chp_technologies_list = sorted(list(chp_technologies_set))

# Create a DataFrame with the single column
dispaSET_chp_tech_list = pd.DataFrame(chp_technologies_list, columns=['Dispa-SET CHP Technologies'])

# Print the final DataFrame
dispaSET_chp_tech_list

,Dispa-SET CHP Technologies
0,COMC
1,COMCX
2,GTUR
3,GTURX
4,ICEN
5,ICENX
6,MCFC
7,PAFC
8,PEFC
9,SOFC


<div style="background-color: black;">
  <hr style="border: 1px solid skyblue;">
    <div style="text-align: justify; margin-left: 3.0em; font-weight: bold; font-size: 16px; font-family: TimesNewRoman; color:skyblue">
    8.4.2. CHP Units Nomenclature - CHPType Dictionary
    </div>
    <div style="text-align: justify; margin-left: 0.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
    The dispa-SET CHP technologies nomenclature are loaded from: 
    <br>
    <span style="display: inline-block; margin-left: 2.0em; font-size: 11px">
    "<a href="https://github.com/energy-modelling-toolkit/Dispa-SET/blob/master/dispaset/common.py" style="color:skyblue">https://github.com/energy-modelling-toolkit/Dispa-SET/blob/master/dispaset/common.py</a>"
    </span>
    <br>
    The same is homogenized by the dictionary built in the next cell:   
    <div style="text-align: justify; margin-left:2em; font-weight: unbold; font-size: 11px; font-family: TimesNewRoman; color:skyblue">
    <span style="display: inline-block; width: 6em; font-weight: bold; font-family: 'Times New Roman'; color: skyblue">* Notes</span>
    Keep values of the first column identical to the 'List of CHP types', since it depends of the Dipsa-SET core code.
    <br>
    <span style="display: inline-block; width: 6em; font-weight: bold; font-family: 'Times New Roman'; color: skyblue"></span>
    If the list in the <code>commons.py</code> script changes, the 'chp_type_dictionary' has to be updated accordingly.
    <br>
    <span style="display: inline-block; width: 6em; font-weight: bold; font-family: 'Times New Roman'; color: skyblue"></span>
    This equivalence dictionary is currently limited and requires updating with additional values.
</div>
<hr style="border: 0.5px solid skyblue;">
</div>

In [331]:
# Dictionary mapping CHP types to descriptions and equivalent technologies
chp_type_dict = {
    
"extraction"    :   {"Description": "Extraction Condensing Turbine; Can vary power/heat output flexibly",     "Equivalent_CHPType":  ["-"                                         ]},
    
"back-pressure" :   {"Description": "Back-Pressure Turbine; Produces power and heat in a fixed ratio"   ,     "Equivalent_CHPType":  ["links urban central gas CHP"             ,
                                                                                                                                      "links urban central gas CHP CC"          , 
                                                                                                                                      "links urban central solid biomass CHP"   ,
                                                                                                                                      "links urban central solid biomass CHP CC"  ]},
    
"p2h"           :   {"Description": "Power-to-Heat Unit; Converts electrical energy into heat"          ,     "Equivalent_CHPType":  ["-"                                         ]}

}

chp_type_dict

{'extraction': {'Description': 'Extraction Condensing Turbine; Can vary power/heat output flexibly',
  'Equivalent_CHPType': ['-']},
 'back-pressure': {'Description': 'Back-Pressure Turbine; Produces power and heat in a fixed ratio',
  'Equivalent_CHPType': ['links urban central gas CHP',
   'links urban central gas CHP CC',
   'links urban central solid biomass CHP',
   'links urban central solid biomass CHP CC']},
 'p2h': {'Description': 'Power-to-Heat Unit; Converts electrical energy into heat',
  'Equivalent_CHPType': ['-']}}

<div style="background-color: black;">
<hr style="border: 0.5px solid skyblue;">
    <div style="text-align: justify; margin-left: 0.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
Classifying the CHP unit type.
    </div>
<hr style="border: 0.5px solid skyblue;">
</div>

In [332]:
# Iterate through each DataFrame in the dictionary
for key, df in processed_power_plants_dfs_dict.items():
    for index, row in df.iterrows():
        # Check if the 'Technology' is in dispaSET_chp_tech_list
        if row['Technology'] in dispaSET_chp_tech_list['Dispa-SET CHP Technologies'].values:
            unit = row['Unit']
            # Iterate through chp_type_dictionary to find a match in 'Equivalent_CHPType'
            for chp_key, chp_value in chp_type_dict.items():
                if unit in chp_value['Equivalent_CHPType']:
                    # Update the 'CHPType' column with the matching key from chp_type_dictionary
                    df.at[index, 'CHPType'] = chp_key
                    break

# Print the updated column from the DataFrame
for key, df in processed_power_plants_dfs_dict.items():
    print(f"CHPType column for {key}:")
    print(df[['CHPType']])

CHPType column for BE_2030:
          CHPType
0             NaN
1             NaN
2             NaN
3             NaN
4             NaN
5             NaN
6             NaN
7             NaN
8             NaN
9             NaN
10            NaN
11            NaN
12            NaN
13            NaN
14            NaN
15            NaN
16            NaN
17            NaN
18            NaN
19            NaN
20            NaN
21            NaN
22            NaN
23            NaN
24            NaN
25            NaN
26  back-pressure
27  back-pressure
28  back-pressure
29  back-pressure
30            NaN
31            NaN
32            NaN
34            NaN
35            NaN
37            NaN
39            NaN
CHPType column for FR_2030:
          CHPType
0             NaN
1             NaN
2             NaN
3             NaN
4             NaN
5             NaN
6             NaN
7             NaN
8             NaN
9             NaN
10            NaN
11            NaN
12            NaN
13      

/tmp/ipykernel_2590693/4057367236.py:11: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'back-pressure' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.at[index, 'CHPType'] = chp_key
/tmp/ipykernel_2590693/4057367236.py:11: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'back-pressure' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.at[index, 'CHPType'] = chp_key
/tmp/ipykernel_2590693/4057367236.py:11: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'back-pressure' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.at[index, 'CHPType'] = chp_key
/tmp/ipykernel_2590693/4057367236.py:11: FutureWarning: Setting an ite

<div style="background-color: black;">
  <hr style="border: 1px solid skyblue;">
    <div style="text-align: justify; margin-left: 3.0em; font-weight: bold; font-size: 16px; font-family: TimesNewRoman; color:skyblue">
    8.4.3. CHP Units Nomenclature - CHP Features Dictionary
    </div>
    <div style="text-align: justify; margin-left: 0.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
    The CHP features values—CHPPowerToHeat; CHPPowerLossFactor; CHPMaxHeat; COP—are updated into the dictionary from the following sources:
    <br>
    <span style="display: inline-block; margin-left: 2.0em; font-size: 11px">
    1. UROSEVIC, D. et al., 2013: "<a href="https://www.sciencedirect.com/science/article/pii/S0360544213005975" style="color:skyblue">https://www.sciencedirect.com/science/article/pii/S0360544213005975</a>"
    </span>
    <br>
    <span style="display: inline-block; margin-left: 2.0em; font-size: 11px">
    2. LECOMTE, T. et al., 2017: "<a href="https://publications.jrc.ec.europa.eu/repository/handle/JRC107769" style="color:skyblue">https://publications.jrc.ec.europa.eu/repository/handle/JRC107769</a>"
    </span>
    <br>
    <span style="display: inline-block; margin-left: 2.0em; font-size: 11px">
    3. EPA Catalog of CHP Technologies, 2017: "<a href="https://www.epa.gov/sites/default/files/2015-07/documents/catalog_of_chp_technologies.pdf" style="color:skyblue">https://www.epa.gov/sites/default/files/2015-07/documents/catalog_of_chp_technologies.pdf</a>"
    </span>
<hr style="border: 0.5px solid skyblue;">
</div>

In [333]:
# Dictionary mapping CHP technologies features 
chp_parameters_dict = pd.DataFrame({
    
    'Technology'         :   [  'COMC'                          , 'GTUR'                          , 'STUR'                          , 'ICEN'                          ,
                                'COMC'                          , 'GTUR'                          , 'STUR'                          , 'ICEN'                          , 
                                'ASHP'                          , 'GSHP'                          , 'WSHP'                                                               ],
    
    'CHPType'           :   [  'extraction'                    , 'extraction'                    , 'extraction'                    , 'extraction'                    ,
                                'back-pressure'                 , 'back-pressure'                 , 'back-pressure'                 , 'back-pressure'                 ,
                                'p2h'                           , 'p2h'                           , 'p2h'                                                                ],
    
    'CHPPowerToHeat'     :   [  0.55                            , 0.45                            , 0.4                             , 0.5                             ,
                                0.65                            , 0.55                            , 0.6                             , 0.45                            ,
                                2.8                             , 3.5                             ,  4.0                                                                 ],
    
    'CHPPowerLossFactor' :   [  0.25                            , 0.35                            , 0.3                             , 0.2                             ,
                                None                            , None                            , None                            , None                            ,
                                None                            , None                            , None                                                                 ],
    
    'CHPMaxHeat'         :   [  'PowerCapacity * 1.2'           , 'PowerCapacity * 1.2'           , 'PowerCapacity * 1.2'           , 'PowerCapacity * 1.2'           ,
                                'PowerCapacity / CHPPowerToHeat', 'PowerCapacity / CHPPowerToHeat', 'PowerCapacity / CHPPowerToHeat', 'PowerCapacity / CHPPowerToHeat',
                                'PowerCapacity * COP'           , 'PowerCapacity * COP'           , 'PowerCapacity * COP'           ,                                    ],
                                                                  
    'COP'                :   [  None                            , None                            , None                            , None                            ,
                                None                            , None                            , None                            , None                            ,
                                2.8                             , 3.5                             , 4.0                                                                  ]
})

chp_parameters_dict

,Technology,CHPType,CHPPowerToHeat,CHPPowerLossFactor,CHPMaxHeat,COP
0,COMC,extraction,0.55,0.25,PowerCapacity * 1.2,NaN
1,GTUR,extraction,0.45,0.35,PowerCapacity * 1.2,NaN
2,STUR,extraction,0.40,0.30,PowerCapacity * 1.2,NaN
3,ICEN,extraction,0.50,0.20,PowerCapacity * 1.2,NaN
4,COMC,back-pressure,0.65,NaN,PowerCapacity / CHPPowerToHeat,NaN
5,GTUR,back-pressure,0.55,NaN,PowerCapacity / CHPPowerToHeat,NaN
6,STUR,back-pressure,0.60,NaN,PowerCapacity / CHPPowerToHeat,NaN
7,ICEN,back-pressure,0.45,NaN,PowerCapacity / CHPPowerToHeat,NaN
8,ASHP,p2h,2.80,NaN,PowerCapacity * COP,2.8
9,GSHP,p2h,3.50,NaN,PowerCapacity * COP,3.5


<div style="background-color: black;">
<hr style="border: 0.5px solid skyblue;">
    <div style="text-align: justify; margin-left: 0.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
Fullfilling the CHP features—CHPPowerToHeat; CHPPowerLossFactor; COP—from the CHP parameters dictionary.
    </div>
<hr style="border: 0.5px solid skyblue;">
</div>

In [334]:
# Columns to bring from chp_parameters_dictionary
merge_cols = ['Technology', 'CHPType']
param_cols = ['CHPPowerToHeat', 'CHPPowerLossFactor', 'CHPMaxHeat', 'COP']

for zone in zone_names:
    key = f"{zone}_{data_target_year}"
    df = processed_power_plants_dfs_dict[key]

    # Keep only rows with a non-empty CHPType
    mask = df['CHPType'].notna() & (df['CHPType'] != "")

    # Merge only the rows where merge is needed
    df_merge = df.loc[mask, merge_cols].merge(
        chp_parameters_dict,
        on=merge_cols,
        how='left'
    )

    # Assign merged parameter values back into the main dataframe
    df.loc[mask, param_cols] = df_merge[param_cols].values

    # Save back into the dictionary
    processed_power_plants_dfs_dict[key] = df

# Print the updated columns from the DataFrame
for key, df in processed_power_plants_dfs_dict.items():
    print(f"CHPPowerToHeat,	CHPPowerLossFactor, CHPMaxHeat and COP columns for {key}:")
    print(df[['CHPPowerToHeat'],['CHPPowerLossFactor'],['CHPMaxHeat'],['COP']])

CHPPowerToHeat,	CHPPowerLossFactor, CHPMaxHeat and COP columns for BE_2030:


/tmp/ipykernel_2590693/776889796.py:20: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '['PowerCapacity / CHPPowerToHeat', 'PowerCapacity / CHPPowerToHeat', 'PowerCapacity / CHPPowerToHeat', 'PowerCapacity / CHPPowerToHeat']' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[mask, param_cols] = df_merge[param_cols].values
/tmp/ipykernel_2590693/776889796.py:20: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '['PowerCapacity / CHPPowerToHeat', 'PowerCapacity / CHPPowerToHeat', 'PowerCapacity / CHPPowerToHeat', 'PowerCapacity / CHPPowerToHeat']' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[mask, param_cols] = df_merge[param_cols].values
/tmp/ipykernel_2590693/776889796.py:20: FutureWarning: Setting an item of incompatible dtype i

InvalidIndexError: (['CHPPowerToHeat'], ['CHPPowerLossFactor'], ['CHPMaxHeat'], ['COP'])

<div style="background-color: black;">
<hr style="border: 0.5px solid skyblue;">
    <div style="text-align: justify; margin-left: 0.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
Fullfilling the CHP features—CHPMaxHeat—from the CHP parameters dictionary formulas.
    </div>
<hr style="border: 0.5px solid skyblue;">
</div>

In [255]:
# Pattern to detect words (potential variable names)
formula_pattern = re.compile(r"[A-Za-z_][A-Za-z0-9_]*")

for zone in zone_names:
    key = f"{zone}_{data_target_year}"
    df = processed_power_plants_dfs_dict[key]

    for idx, row in df.iterrows():

        cell = row["CHPMaxHeat"]

        # Skip empty or non-string cells
        if pd.isna(cell) or not isinstance(cell, str) or cell.strip() == "":
            continue

        expression = cell.strip()

        # ---- 1. Extract possible variable names ----
        tokens = set(formula_pattern.findall(expression))
        columns_used = [t for t in tokens if t in df.columns]

        if not columns_used:
            # No valid column names → not a formula
            continue

        # ---- 2. Build evaluation dictionary from row ----
        local_vars = {}
        valid = True

        for col in columns_used:
            val = row[col]
            if pd.isna(val):
                valid = False
                break
            local_vars[col] = float(val)

        if not valid:
            continue

        # ---- 3. Safe evaluation ----
        try:
            # Evaluate with NO builtins and ONLY our variables
            result = eval(
                expression,
                {"__builtins__": None},   # Disable all built-ins
                local_vars                # Only allow numeric column values
            )

            df.at[idx, "CHPMaxHeat"] = float(result)

        except Exception as e:
            print(f"⚠️ Error evaluating formula in {key}, row {idx}: '{expression}' → {e}")
            continue

    processed_power_plants_dfs_dict[key] = df

# Print the updated column from the DataFrame
for key, df in processed_power_plants_dfs_dict.items():
    print(f"CHPMaxHeat column for {key}:")
    print(df[['CHPMaxHeat']])

CHPMaxHeat column for BE_2030:
     CHPMaxHeat
0           NaN
1           NaN
2           NaN
3           NaN
4           NaN
5           NaN
6           NaN
7           NaN
8           NaN
9           NaN
10          NaN
11          NaN
12          NaN
13          NaN
14          NaN
15          NaN
16          NaN
17          NaN
18          NaN
19          NaN
20          NaN
21          NaN
22          NaN
23          NaN
24          NaN
25          NaN
26  1146.936985
27     0.174371
28   136.633488
29  1040.384799
30          NaN
31          NaN
32          NaN
34          NaN
35          NaN
37          NaN
39          NaN
CHPMaxHeat column for FR_2030:
      CHPMaxHeat
0            NaN
1            NaN
2            NaN
3            NaN
4            NaN
5            NaN
6            NaN
7            NaN
8            NaN
9            NaN
10           NaN
11           NaN
12           NaN
13           NaN
14           NaN
15           NaN
16           NaN
17           NaN
18     

<div style="background-color: black;">
<hr style="border: 0.5px solid skyblue;">
<div style="text-align: right; margin-left: 3.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman;color:skyblue">
    Tracking Variables.
</div>
    <div style="text-align: right; margin-left: 1.50em; font-weight: unbold; font-size: 13px; font-family: TimesNewRoman;color:skyblue">
    This cells are just to confirm all the file names, file paths and other information related to the data being processed.
    <br>
  Also are used to ensure the inputs for next cells in order to avoid to re-enter the same information each time.
</div>
<hr style="border: 0.5px solid skyblue;">
</div>

In [230]:
print (f"Name of the DispaSET Unleash folder:                        {dispaSET_unleash_folder_name}\n")
print (f"Path to the DispaSET Unleash folder:                        {dispaSET_unleash_folder_path}\n")
print (f"Name of the Power Plants Base data folder:                  {power_plants_base_data_folder_name}\n")
print (f"Path to the Power Plants Base data folder:                  {power_plants_base_data_folder_path}\n")
print (f"Name of Power Plants_Pypsa Raw data folder:                 {power_plants_pypsa_raw_data_folder_name}\n")
print (f"Path to the Power Plants_Pypsa Raw data folder:             {power_plants_pypsa_raw_data_folder_path}\n")
print (f"Name of the Power Plants_Pypsa Formated data folder:        {power_plants_pypsa_formated_data_folder_name}\n")
print (f"Path to the Power Plants_Pypsa Formated data folder:        {power_plants_pypsa_formated_data_folder_path}\n")
print (f"Name of the zones:                                          {zone_names}\n")
print (f"Target year:                                                {data_target_year}\n")
print (f"Name of the Power Plant DataFrames (dictionary):            {list(power_plants_dfs_dict.keys())}\n")
print (f"Name of the Technologies Equivalences Dictionary:           tech_equivalences_dict\n")
print (f"Name of the Overall Fuel Technology Match DataFrame:        overal_fuel_technologies_match_df\n")
print (f"Name of the Fuel Technology Match Dictionary:               fuel_technologies_match_dict\n")
print (f"Name of the PyPSA Nodal Capacities DataFrames (dictionary): {list(nodal_capacities_dfs_dict.keys())}\n")
print (f"Name of the Processed Power Plant DataFrames (dictionary):  {list(processed_power_plants_dfs_dict.keys())}\n")
print (f"name of the dispaSET CHP technologies list:                 dispaSET_chp_tech_list\n")
print (f"Keys of the CHP type dictionary:                            {list(chp_type_dict.keys())}\n")
print (f"Keys of the CHP parameters dictionady:                      {list(chp_parameters_dict.keys())}\n")

Name of the DispaSET Unleash folder:                        Dispa-SET_Unleash

Path to the DispaSET Unleash folder:                        /home/ray/Dispa-SET_Unleash

Name of the Power Plants Base data folder:                  PowerPlants

Path to the Power Plants Base data folder:                  /home/ray/Dispa-SET_Unleash/Database/PowerPlants

Name of Power Plants_Pypsa Raw data folder:                 PowerPlants

Path to the Power Plants_Pypsa Raw data folder:             /home/ray/Dispa-SET_Unleash/RawData_PyPSA/Reference_Scenario/PowerPlants

Name of the Power Plants_Pypsa Formated data folder:        PowerPlants

Path to the Power Plants_Pypsa Formated data folder:        /home/ray/Dispa-SET_Unleash/Database_PyPSA/Reference_Scenario/PowerPlants

Name of the zones:                                          ['BE', 'FR', 'DE', 'NL', 'UK']

Target year:                                                2030

Name of the Power Plant DataFrames (dictionary):            ['BE_2030', 'FR_

In [254]:
# Define the path where the CSV files will be saved
export_path = "/home/ray/Downloads/Downloads/Test/"

# Ensure the directory exists
os.makedirs(export_path, exist_ok=True)

# Loop through each DataFrame in the dictionary
for country, df in processed_power_plants_dfs_dict.items():
    # Define the full path for each CSV file
    filename = os.path.join(export_path, f"{country}_1.csv")

    # Export the DataFrame to CSV
    df.to_csv(filename, index=False)

    print(f"Exported {country} to {filename}")

Exported BE_2030 to /home/ray/Downloads/Downloads/Test/BE_2030_1.csv
Exported FR_2030 to /home/ray/Downloads/Downloads/Test/FR_2030_1.csv
Exported DE_2030 to /home/ray/Downloads/Downloads/Test/DE_2030_1.csv
Exported NL_2030 to /home/ray/Downloads/Downloads/Test/NL_2030_1.csv
Exported UK_2030 to /home/ray/Downloads/Downloads/Test/UK_2030_1.csv


In [397]:
# Define the path where the CSV files will be saved
export_path = "/home/ray/Downloads/Downloads/Test"

# Ensure the directory exists
os.makedirs(export_path, exist_ok=True)

# Loop through each DataFrame in the dictionary
for country, df in power_plants_dfs_dict.items():
    # Define the full path for each CSV file
    filename = os.path.join(export_path, f"{country}.csv")

    # Export the DataFrame to CSV
    df.to_csv(filename, index=False)

    print(f"Exported {country} to {filename}")

Exported BE_2030 to /home/ray/Downloads/Downloads/Test/BE_2030.csv
Exported FR_2030 to /home/ray/Downloads/Downloads/Test/FR_2030.csv
Exported DE_2030 to /home/ray/Downloads/Downloads/Test/DE_2030.csv
Exported NL_2030 to /home/ray/Downloads/Downloads/Test/NL_2030.csv
Exported UK_2030 to /home/ray/Downloads/Downloads/Test/UK_2030.csv


In [ ]:
# Define the path where the CSV files will be saved
export_path = "/home/ray/Downloads/Downloads"

# Ensure the directory exists
os.makedirs(export_path, exist_ok=True)

# Loop through each DataFrame in the dictionary
for country, df in nodal_capacities_dfs_dict.items():
    # Define the full path for each CSV file
    filename = os.path.join(export_path, f"{country}.csv")

    # Export the DataFrame to CSV
    df.to_csv(filename, index=False)

    print(f"Exported {country} to {filename}")

In [ ]:
# Define the path where the CSV files will be saved
export_path = "/home/ray/Downloads/Downloads/Test"

# Ensure the directory exists
os.makedirs(export_path, exist_ok=True)

# Loop through each DataFrame in the dictionary
for country, df in nodal_capacities_dfs_dict.items():
    # Define the full path for each CSV file
    filename = os.path.join(export_path, f"{country}.csv")

    # Export the DataFrame to CSV
    df.to_csv(filename, index=False)

    print(f"Exported {country} to {filename}")

CHP types updated successfully.
CHP types updated successfully.
CHP types updated successfully.
CHP types updated successfully.
CHP types updated successfully.
CHP types updated successfully.
CHP types updated successfully.
CHP types updated successfully.
CHP types updated successfully.
CHP types updated successfully.
CHP types updated successfully.
CHP types updated successfully.
CHP types updated successfully.
CHP types updated successfully.
CHP types updated successfully.
CHP types updated successfully.
CHP types updated successfully.
CHP types updated successfully.
CHP types updated successfully.
CHP types updated successfully.
CHP types updated successfully.
CHP types updated successfully.
CHP types updated successfully.
CHP types updated successfully.
CHP types updated successfully.
CHP types updated successfully.
CHP types updated successfully.
CHP types updated successfully.
CHP types updated successfully.
CHP types updated successfully.


<div style="background-color: black;">
<hr style="border: 1px solid skyblue;">
<div style="text-align: justify; margin-left: 3.0em; font-weight: bold; font-size: 17px; font-family: TimesNewRoman;color:skyblue">
    7.14. Efficiency Field Fullfilling
</div>
<div style="text-align: justify; margin-left: 0.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman;color:skyblue">
    Finding the closest efficiency value for all the units with the Efficiency field empty or with zero value.
</div>
<div style="text-align: justify; margin-left: 2.0em; font-weight: unbold; font-size: 13px; font-family: TimesNewRoman;color:skyblue">
    The Efficiency technical feauture is taken from the already agregated data base EU_Power_Units_Technical_Features.csv using the notebook: EU_Power_Plant_Technical_Data_Base_Gathering.ipynb and is os sellected in base on the Technology, the Fuel and the PowerCapacity features.
</div>
<div style="text-align: justify; margin-left: 0.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman;color:skyblue">
    This process is done for each zone.
</div>
<br>
<div style="text-align: right; margin-left: 3.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman;color:skyblue">
    Tracking Variables. 
    <br>
    <div style="text-align: right; margin-left: 1.50em; font-weight: unbold; font-size: 13px; font-family: TimesNewRoman;">
    This cells are just to confirm all the file names, file paths and other information related to the data being processed.
    <br>
  Also are used to ensure the inputs for next cells in order to avoid to re-enter the same information each time.
</div>

In [65]:
power_plants_clean_data_file_list

['/home/ray/Dispa-SET_Unleash/RawData/PowerPlants/AT/2024.csv',
 '/home/ray/Dispa-SET_Unleash/RawData/PowerPlants/BE/2024.csv',
 '/home/ray/Dispa-SET_Unleash/RawData/PowerPlants/BG/2024.csv',
 '/home/ray/Dispa-SET_Unleash/RawData/PowerPlants/CH/2024.csv',
 '/home/ray/Dispa-SET_Unleash/RawData/PowerPlants/CY/2024.csv',
 '/home/ray/Dispa-SET_Unleash/RawData/PowerPlants/CZ/2024.csv',
 '/home/ray/Dispa-SET_Unleash/RawData/PowerPlants/DE/2024.csv',
 '/home/ray/Dispa-SET_Unleash/RawData/PowerPlants/DK/2024.csv',
 '/home/ray/Dispa-SET_Unleash/RawData/PowerPlants/EE/2024.csv',
 '/home/ray/Dispa-SET_Unleash/RawData/PowerPlants/EL/2024.csv',
 '/home/ray/Dispa-SET_Unleash/RawData/PowerPlants/ES/2024.csv',
 '/home/ray/Dispa-SET_Unleash/RawData/PowerPlants/FI/2024.csv',
 '/home/ray/Dispa-SET_Unleash/RawData/PowerPlants/FR/2024.csv',
 '/home/ray/Dispa-SET_Unleash/RawData/PowerPlants/HR/2024.csv',
 '/home/ray/Dispa-SET_Unleash/RawData/PowerPlants/HU/2024.csv',
 '/home/ray/Dispa-SET_Unleash/RawData/Po

In [66]:
EU_Power_Units_Technical_Features_file_path

'/home/ray/Dispa-SET_Unleash/RawData/PowerPlants/Dictionaries/EU_Power_Units_Technical_Features.csv'

In [67]:
# Import the function from the external script
from Data_Processing_Functions import copy_technical_values

# Define the paths and variables
#EU_Power_Units_Technical_Features_file_path = '/home/ray/Dispa-SET_Unleash/RawData/PowerPlants/EU_Power_Units_Technical_Features.csv'
#power_plants_clean_data_file_list = [
#    '/home/ray/Dispa-SET_Unleash/RawData/PowerPlants/2020_power_plants_raw_data_sources_20240407_172323/2020.csv',
#    '/home/ray/Dispa-SET_Unleash/RawData/PowerPlants/DK/2020.csv',
#    '/home/ray/Dispa-SET_Unleash/RawData/PowerPlants/CH/2020.csv',
#    '/home/ray/Dispa-SET_Unleash/RawData/PowerPlants/BE/2020.csv'
#]
common_columns = ['PowerCapacity', 'Technology', 'Fuel']
column_to_copy = 'Efficiency'

copy_technical_values(EU_Power_Units_Technical_Features_file_path, power_plants_clean_data_file_list, common_columns, column_to_copy)

Processing target file: /home/ray/Dispa-SET_Unleash/RawData/PowerPlants/AT/2024.csv
Processing target file: /home/ray/Dispa-SET_Unleash/RawData/PowerPlants/BE/2024.csv
Processing target file: /home/ray/Dispa-SET_Unleash/RawData/PowerPlants/BG/2024.csv
Processing target file: /home/ray/Dispa-SET_Unleash/RawData/PowerPlants/CH/2024.csv
Processing target file: /home/ray/Dispa-SET_Unleash/RawData/PowerPlants/CY/2024.csv
Processing target file: /home/ray/Dispa-SET_Unleash/RawData/PowerPlants/CZ/2024.csv
Processing target file: /home/ray/Dispa-SET_Unleash/RawData/PowerPlants/DE/2024.csv
Processing target file: /home/ray/Dispa-SET_Unleash/RawData/PowerPlants/DK/2024.csv
Processing target file: /home/ray/Dispa-SET_Unleash/RawData/PowerPlants/EE/2024.csv
Processing target file: /home/ray/Dispa-SET_Unleash/RawData/PowerPlants/EL/2024.csv


/home/ray/Dispa-SET_Unleash/scripts/Unleash_Raw_Data_Processing/Data_Processing_Functions.py:584: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  first_fuel_filtered['PowerCapacity'] = pd.to_numeric(first_fuel_filtered['PowerCapacity'], errors='coerce')
/home/ray/Dispa-SET_Unleash/scripts/Unleash_Raw_Data_Processing/Data_Processing_Functions.py:584: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  first_fuel_filtered['PowerCapacity'] = pd.to_numeric(first_fuel_filtered['PowerCapacity'], errors='coerce')
/home/ra

Processing target file: /home/ray/Dispa-SET_Unleash/RawData/PowerPlants/ES/2024.csv
Processing target file: /home/ray/Dispa-SET_Unleash/RawData/PowerPlants/FI/2024.csv
Processing target file: /home/ray/Dispa-SET_Unleash/RawData/PowerPlants/FR/2024.csv
Processing target file: /home/ray/Dispa-SET_Unleash/RawData/PowerPlants/HR/2024.csv
Processing target file: /home/ray/Dispa-SET_Unleash/RawData/PowerPlants/HU/2024.csv
Processing target file: /home/ray/Dispa-SET_Unleash/RawData/PowerPlants/IE/2024.csv
Processing target file: /home/ray/Dispa-SET_Unleash/RawData/PowerPlants/IT/2024.csv


/home/ray/Dispa-SET_Unleash/scripts/Unleash_Raw_Data_Processing/Data_Processing_Functions.py:584: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  first_fuel_filtered['PowerCapacity'] = pd.to_numeric(first_fuel_filtered['PowerCapacity'], errors='coerce')
/home/ray/Dispa-SET_Unleash/scripts/Unleash_Raw_Data_Processing/Data_Processing_Functions.py:584: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  first_fuel_filtered['PowerCapacity'] = pd.to_numeric(first_fuel_filtered['PowerCapacity'], errors='coerce')
/home/ra

Processing target file: /home/ray/Dispa-SET_Unleash/RawData/PowerPlants/LT/2024.csv
Processing target file: /home/ray/Dispa-SET_Unleash/RawData/PowerPlants/LU/2024.csv
Processing target file: /home/ray/Dispa-SET_Unleash/RawData/PowerPlants/LV/2024.csv
Processing target file: /home/ray/Dispa-SET_Unleash/RawData/PowerPlants/MT/2024.csv
Processing target file: /home/ray/Dispa-SET_Unleash/RawData/PowerPlants/NL/2024.csv
Processing target file: /home/ray/Dispa-SET_Unleash/RawData/PowerPlants/NO/2024.csv
Processing target file: /home/ray/Dispa-SET_Unleash/RawData/PowerPlants/PL/2024.csv
Processing target file: /home/ray/Dispa-SET_Unleash/RawData/PowerPlants/PT/2024.csv


/home/ray/Dispa-SET_Unleash/scripts/Unleash_Raw_Data_Processing/Data_Processing_Functions.py:584: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  first_fuel_filtered['PowerCapacity'] = pd.to_numeric(first_fuel_filtered['PowerCapacity'], errors='coerce')
/home/ray/Dispa-SET_Unleash/scripts/Unleash_Raw_Data_Processing/Data_Processing_Functions.py:584: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  first_fuel_filtered['PowerCapacity'] = pd.to_numeric(first_fuel_filtered['PowerCapacity'], errors='coerce')
/home/ra

Processing target file: /home/ray/Dispa-SET_Unleash/RawData/PowerPlants/RO/2024.csv
Processing target file: /home/ray/Dispa-SET_Unleash/RawData/PowerPlants/SE/2024.csv
Processing target file: /home/ray/Dispa-SET_Unleash/RawData/PowerPlants/SI/2024.csv
Processing target file: /home/ray/Dispa-SET_Unleash/RawData/PowerPlants/SK/2024.csv
Processing target file: /home/ray/Dispa-SET_Unleash/RawData/PowerPlants/UK/2024.csv
Efficiency Field values copied successfully.


/home/ray/Dispa-SET_Unleash/scripts/Unleash_Raw_Data_Processing/Data_Processing_Functions.py:584: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  first_fuel_filtered['PowerCapacity'] = pd.to_numeric(first_fuel_filtered['PowerCapacity'], errors='coerce')
/home/ray/Dispa-SET_Unleash/scripts/Unleash_Raw_Data_Processing/Data_Processing_Functions.py:584: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  first_fuel_filtered['PowerCapacity'] = pd.to_numeric(first_fuel_filtered['PowerCapacity'], errors='coerce')
/home/ra

<div style="background-color: black;">
<div style="text-align: justify; margin-left: 0.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman;color:skyblue">
    Filling with a value of 1 the efficiency technical feature for all the units with the Efficiency field empty or with zero value.
</div>
<div style="text-align: justify; margin-left: 2.0em; font-weight: unbold; font-size: 13px; font-family: TimesNewRoman;color:skyblue">
    Additionally all this units found with any efficiency value, will be copied to the corresponding no denined units file.
</div>
<div style="text-align: justify; margin-left: 0.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman;color:skyblue">
    This process is done for each zone.
</div>

In [68]:
from Data_Processing_Functions import fill_empty_values_with_specified

# Define the columns to be filled
Column = 'Efficiency'
Value = '1'  # Change this to the desired value

# Iterate over each pair of files
for input_file, output_file in zip(power_plants_clean_data_file_list, power_plants_all_data_not_defined_units_file_list):
    # Call the function for the current pair of files
    fill_empty_values_with_specified(input_file, output_file, Column, Value)

No empty 'Efficiency' or value 0 cells found in '/home/ray/Dispa-SET_Unleash/RawData/PowerPlants/AT/2024.csv'.
No empty 'Efficiency' or value 0 cells found in '/home/ray/Dispa-SET_Unleash/RawData/PowerPlants/BE/2024.csv'.
No empty 'Efficiency' or value 0 cells found in '/home/ray/Dispa-SET_Unleash/RawData/PowerPlants/BG/2024.csv'.
No empty 'Efficiency' or value 0 cells found in '/home/ray/Dispa-SET_Unleash/RawData/PowerPlants/CH/2024.csv'.
No empty 'Efficiency' or value 0 cells found in '/home/ray/Dispa-SET_Unleash/RawData/PowerPlants/CY/2024.csv'.
No empty 'Efficiency' or value 0 cells found in '/home/ray/Dispa-SET_Unleash/RawData/PowerPlants/CZ/2024.csv'.
No empty 'Efficiency' or value 0 cells found in '/home/ray/Dispa-SET_Unleash/RawData/PowerPlants/DE/2024.csv'.
No empty 'Efficiency' or value 0 cells found in '/home/ray/Dispa-SET_Unleash/RawData/PowerPlants/DK/2024.csv'.
No empty 'Efficiency' or value 0 cells found in '/home/ray/Dispa-SET_Unleash/RawData/PowerPlants/EE/2024.csv'.
N

<div style="background-color: black;">
<hr style="border: 1px solid skyblue;">
<div style="text-align: justify; margin-left: 3.0em; font-weight: bold; font-size: 17px; font-family: TimesNewRoman;color:skyblue">
    7.14.  MinDownTime | MinDownTime | RampUpRate | RampDownRate | StartUpCost | NoLoadCost_pu | RampingCost | PartLoadMin | MinEfficiency | StartUpTime | CO2Intensity Fields Fulfilling
</div>
<div style="text-align: justify; margin-left: 0.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman;color:skyblue">
    Finding the closest value for the next technical features:
 </div>
<div style="text-align: justify; margin-left: 2.0em; font-weight: unbold; font-size: 13px; font-family: TimesNewRoman;color:skyblue">
    MinUpTime, MinDownTime, MinDownTime, RampUpRate, RampDownRate, StartUpCost, NoLoadCost_pu, RampingCost, PartLoadMin, MinEfficiency, StartUpTime, CO2Intensity.
 </div>
<div style="text-align: justify; margin-left: 0.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman;color:skyblue">
    This is done for all the units which corresponding field value is empty or with zero.
    </div>
<div style="text-align: justify; margin-left: 2.0em; font-weight: unbold; font-size: 13px; font-family: TimesNewRoman;color:skyblue">
    All the technical feautures are taken from the already agregated data base EU_Power_Units_Technical_Features.csv using the notebook: EU_Power_Plant_Technical_Data_Base_Gathering.ipynb and are sellected in base on the Technology, the Fuel and the PowerCapacity features.
</div>
<div style="text-align: justify; margin-left: 2.0em; font-weight: unbold; font-size: 13px; font-family: TimesNewRoman;color:skyblue">
    This process is done for each zone.
</div>

In [69]:
columns_to_copy = [
    'MinUpTime',
    'MinDownTime',
    'RampUpRate',
    'RampDownRate',
    'StartUpCost',
    'NoLoadCost_pu',
    'RampingCost',
    'PartLoadMin',
    'MinEfficiency',
    'StartUpTime',
    'CO2Intensity'
]
common_columns = ['PowerCapacity', 'Technology', 'Fuel']

In [70]:
for column_to_copy in columns_to_copy:
    copy_technical_values(EU_Power_Units_Technical_Features_file_path, power_plants_clean_data_file_list, common_columns, column_to_copy)

Processing target file: /home/ray/Dispa-SET_Unleash/RawData/PowerPlants/AT/2024.csv
Processing target file: /home/ray/Dispa-SET_Unleash/RawData/PowerPlants/BE/2024.csv
Processing target file: /home/ray/Dispa-SET_Unleash/RawData/PowerPlants/BG/2024.csv
Processing target file: /home/ray/Dispa-SET_Unleash/RawData/PowerPlants/CH/2024.csv
Processing target file: /home/ray/Dispa-SET_Unleash/RawData/PowerPlants/CY/2024.csv
Processing target file: /home/ray/Dispa-SET_Unleash/RawData/PowerPlants/CZ/2024.csv
Processing target file: /home/ray/Dispa-SET_Unleash/RawData/PowerPlants/DE/2024.csv
Processing target file: /home/ray/Dispa-SET_Unleash/RawData/PowerPlants/DK/2024.csv
Processing target file: /home/ray/Dispa-SET_Unleash/RawData/PowerPlants/EE/2024.csv
Processing target file: /home/ray/Dispa-SET_Unleash/RawData/PowerPlants/EL/2024.csv


/home/ray/Dispa-SET_Unleash/scripts/Unleash_Raw_Data_Processing/Data_Processing_Functions.py:584: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  first_fuel_filtered['PowerCapacity'] = pd.to_numeric(first_fuel_filtered['PowerCapacity'], errors='coerce')
/home/ray/Dispa-SET_Unleash/scripts/Unleash_Raw_Data_Processing/Data_Processing_Functions.py:584: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  first_fuel_filtered['PowerCapacity'] = pd.to_numeric(first_fuel_filtered['PowerCapacity'], errors='coerce')
/home/ra

Processing target file: /home/ray/Dispa-SET_Unleash/RawData/PowerPlants/ES/2024.csv
Processing target file: /home/ray/Dispa-SET_Unleash/RawData/PowerPlants/FI/2024.csv
Processing target file: /home/ray/Dispa-SET_Unleash/RawData/PowerPlants/FR/2024.csv
Processing target file: /home/ray/Dispa-SET_Unleash/RawData/PowerPlants/HR/2024.csv
Processing target file: /home/ray/Dispa-SET_Unleash/RawData/PowerPlants/HU/2024.csv
Processing target file: /home/ray/Dispa-SET_Unleash/RawData/PowerPlants/IE/2024.csv


/home/ray/Dispa-SET_Unleash/scripts/Unleash_Raw_Data_Processing/Data_Processing_Functions.py:584: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  first_fuel_filtered['PowerCapacity'] = pd.to_numeric(first_fuel_filtered['PowerCapacity'], errors='coerce')
/home/ray/Dispa-SET_Unleash/scripts/Unleash_Raw_Data_Processing/Data_Processing_Functions.py:584: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  first_fuel_filtered['PowerCapacity'] = pd.to_numeric(first_fuel_filtered['PowerCapacity'], errors='coerce')
/home/ra

Processing target file: /home/ray/Dispa-SET_Unleash/RawData/PowerPlants/IT/2024.csv
Processing target file: /home/ray/Dispa-SET_Unleash/RawData/PowerPlants/LT/2024.csv
Processing target file: /home/ray/Dispa-SET_Unleash/RawData/PowerPlants/LU/2024.csv
Processing target file: /home/ray/Dispa-SET_Unleash/RawData/PowerPlants/LV/2024.csv
Processing target file: /home/ray/Dispa-SET_Unleash/RawData/PowerPlants/MT/2024.csv
Processing target file: /home/ray/Dispa-SET_Unleash/RawData/PowerPlants/NL/2024.csv
Processing target file: /home/ray/Dispa-SET_Unleash/RawData/PowerPlants/NO/2024.csv
Processing target file: /home/ray/Dispa-SET_Unleash/RawData/PowerPlants/PL/2024.csv
Processing target file: /home/ray/Dispa-SET_Unleash/RawData/PowerPlants/PT/2024.csv
Processing target file: /home/ray/Dispa-SET_Unleash/RawData/PowerPlants/RO/2024.csv


...

/home/ray/Dispa-SET_Unleash/scripts/Unleash_Raw_Data_Processing/Data_Processing_Functions.py:584: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  first_fuel_filtered['PowerCapacity'] = pd.to_numeric(first_fuel_filtered['PowerCapacity'], errors='coerce')
/home/ray/Dispa-SET_Unleash/scripts/Unleash_Raw_Data_Processing/Data_Processing_Functions.py:584: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  first_fuel_filtered['PowerCapacity'] = pd.to_numeric(first_fuel_filtered['PowerCapacity'], errors='coerce')
/home/ra

Processing target file: /home/ray/Dispa-SET_Unleash/RawData/PowerPlants/FR/2024.csv
Processing target file: /home/ray/Dispa-SET_Unleash/RawData/PowerPlants/HR/2024.csv
Processing target file: /home/ray/Dispa-SET_Unleash/RawData/PowerPlants/HU/2024.csv
Processing target file: /home/ray/Dispa-SET_Unleash/RawData/PowerPlants/IE/2024.csv
Processing target file: /home/ray/Dispa-SET_Unleash/RawData/PowerPlants/IT/2024.csv
Processing target file: /home/ray/Dispa-SET_Unleash/RawData/PowerPlants/LT/2024.csv
Processing target file: /home/ray/Dispa-SET_Unleash/RawData/PowerPlants/LU/2024.csv
Processing target file: /home/ray/Dispa-SET_Unleash/RawData/PowerPlants/LV/2024.csv
Processing target file: /home/ray/Dispa-SET_Unleash/RawData/PowerPlants/MT/2024.csv
Processing target file: /home/ray/Dispa-SET_Unleash/RawData/PowerPlants/NL/2024.csv


/home/ray/Dispa-SET_Unleash/scripts/Unleash_Raw_Data_Processing/Data_Processing_Functions.py:584: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  first_fuel_filtered['PowerCapacity'] = pd.to_numeric(first_fuel_filtered['PowerCapacity'], errors='coerce')
/home/ray/Dispa-SET_Unleash/scripts/Unleash_Raw_Data_Processing/Data_Processing_Functions.py:584: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  first_fuel_filtered['PowerCapacity'] = pd.to_numeric(first_fuel_filtered['PowerCapacity'], errors='coerce')
/home/ra

Processing target file: /home/ray/Dispa-SET_Unleash/RawData/PowerPlants/NO/2024.csv
Processing target file: /home/ray/Dispa-SET_Unleash/RawData/PowerPlants/PL/2024.csv
Processing target file: /home/ray/Dispa-SET_Unleash/RawData/PowerPlants/PT/2024.csv
Processing target file: /home/ray/Dispa-SET_Unleash/RawData/PowerPlants/RO/2024.csv
Processing target file: /home/ray/Dispa-SET_Unleash/RawData/PowerPlants/SE/2024.csv
Processing target file: /home/ray/Dispa-SET_Unleash/RawData/PowerPlants/SI/2024.csv
Processing target file: /home/ray/Dispa-SET_Unleash/RawData/PowerPlants/SK/2024.csv
Processing target file: /home/ray/Dispa-SET_Unleash/RawData/PowerPlants/UK/2024.csv
CO2Intensity Field values copied successfully.


/home/ray/Dispa-SET_Unleash/scripts/Unleash_Raw_Data_Processing/Data_Processing_Functions.py:584: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  first_fuel_filtered['PowerCapacity'] = pd.to_numeric(first_fuel_filtered['PowerCapacity'], errors='coerce')
/home/ray/Dispa-SET_Unleash/scripts/Unleash_Raw_Data_Processing/Data_Processing_Functions.py:584: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  first_fuel_filtered['PowerCapacity'] = pd.to_numeric(first_fuel_filtered['PowerCapacity'], errors='coerce')
/home/ra

<div style="background-color: black;">
<hr style="border: 2px solid skyblue;">
<div style="text-align: justify; margin-left: 3.0em; font-weight: bold; font-size: 18px; font-family: TimesNewRoman;color:skyblue">
    8. Zone Final Classification
</div>
<div style="text-align: justify; margin-left: 0.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman;color:skyblue">
    Filtering by Zone all the formating files resulting from the downloads files set by the "General" key at the begining of the process.
</div>
<div style="text-align: justify; margin-left: 2.0em; font-weight: unbold; font-size: 13px; font-family: TimesNewRoman;color:skyblue">
    Each row of the files are copied according their zone clasification in their corresponding folder path. 
</div>
<br>
<hr style="border: 0.5px solid skyblue;">
<div style="text-align: justify; margin-left: 3.0em; font-weight: bold; font-size: 17px; font-family: TimesNewRoman;color:skyblue">
    8.1. Zone Field Homogenization
</div>
<div style="text-align: justify; margin-left: 0.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman;color:skyblue">
    Eliminating the space characters from the Zone fileds to avoid errors in the zone classifiaction process.
    <br>
    This process is done for each zone.
</div>

In [71]:
# List of file paths
#power_plants_clean_data_file_list = [
#    '/home/ray/Dispa-SET_Unleash/RawData/PowerPlants/2020_power_plants_raw_data_sources_20240407_201458/2020.csv',
#    '/home/ray/Dispa-SET_Unleash/RawData/PowerPlants/DK/2020.csv',
#    '/home/ray/Dispa-SET_Unleash/RawData/PowerPlants/CH/2020.csv',
#    '/home/ray/Dispa-SET_Unleash/RawData/PowerPlants/BE/2020.csv'
#]

# Iterate over each file
for file_path in power_plants_clean_data_file_list:
    # Read the CSV file into a DataFrame
    df = pd.read_csv(file_path)

    # Trim values in the 'Zone' column
    if 'Zone' in df.columns:
        df['Zone'] = df['Zone'].str.strip()

        # Write the modified DataFrame back to the CSV file
        df.to_csv(file_path, index=False)

        print(f"Trimmed 'Zone' column values in {file_path}")
    else:
        print(f"'Zone' column not found in {file_path}")


Trimmed 'Zone' column values in /home/ray/Dispa-SET_Unleash/RawData/PowerPlants/AT/2024.csv
Trimmed 'Zone' column values in /home/ray/Dispa-SET_Unleash/RawData/PowerPlants/BE/2024.csv
Trimmed 'Zone' column values in /home/ray/Dispa-SET_Unleash/RawData/PowerPlants/BG/2024.csv
Trimmed 'Zone' column values in /home/ray/Dispa-SET_Unleash/RawData/PowerPlants/CH/2024.csv
Trimmed 'Zone' column values in /home/ray/Dispa-SET_Unleash/RawData/PowerPlants/CY/2024.csv
Trimmed 'Zone' column values in /home/ray/Dispa-SET_Unleash/RawData/PowerPlants/CZ/2024.csv
Trimmed 'Zone' column values in /home/ray/Dispa-SET_Unleash/RawData/PowerPlants/DE/2024.csv
Trimmed 'Zone' column values in /home/ray/Dispa-SET_Unleash/RawData/PowerPlants/DK/2024.csv
Trimmed 'Zone' column values in /home/ray/Dispa-SET_Unleash/RawData/PowerPlants/EE/2024.csv
Trimmed 'Zone' column values in /home/ray/Dispa-SET_Unleash/RawData/PowerPlants/EL/2024.csv
Trimmed 'Zone' column values in /home/ray/Dispa-SET_Unleash/RawData/PowerPlants/

<div style="background-color: black;">
<div style="text-align: justify; margin-left: 0.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman;color:skyblue">
    Filtering by Zone all the formating files resulting from the downloads files set by the "General" key at the begining of the process.
<div style="text-align: justify; margin-left: 2.0em; font-weight: unbold; font-size: 13px; font-family: TimesNewRoman;color:skyblue">
    Each row of the files are copied according their zone clasification in their corresponding folder path. 
</div>

In [72]:
# File paths
#power_plants_raw_data_sources_file_path = '/home/ray/Dispa-SET_Unleash/RawData/PowerPlants/2020_power_plants_raw_data_sources_20240407_191506/2020_power_plants_raw_data_sources_20240407_191506.csv'

# Dictionary of created zones
#created_zones = {'DE': '/home/ray/Dispa-SET_Unleash/RawData/PowerPlants/DE',
#                 'DK': '/home/ray/Dispa-SET_Unleash/RawData/PowerPlants/DK',
#                 'CH': '/home/ray/Dispa-SET_Unleash/RawData/PowerPlants/CH',
#                'BE': '/home/ray/Dispa-SET_Unleash/RawData/PowerPlants/BE'}

# Read the first file
first_df = pd.read_csv(power_plants_raw_data_sources_file_path)

# Filter rows with Zone as "General"
general_zones_df = first_df[first_df['Zone'] == 'General']

# Iterate over rows with Zone as "General"
for index, row in general_zones_df.iterrows():
    # Get the path of the corresponding file
    file_path = row['Final_Clean_File_Path']
    
    # Read the second CSV file
    second_df = pd.read_csv(file_path)
    
    # Iterate over zone names
    for zone in zone_names:
        # Filter rows in the second file with matching Zone
        zone_rows = second_df[second_df['Zone'] == zone]
        
        # Check if there are matching rows
        if not zone_rows.empty:
            # Get the corresponding zone path
            zone_path = created_zones.get(zone)
            
            # Create the directory if it does not exist
            if not os.path.exists(zone_path):
                os.makedirs(zone_path)
            
            # Get the file name
            file_name = os.path.basename(file_path)
            zone_file_path = os.path.join(zone_path, file_name)
            
            # Copy the rows to the corresponding zone file
            if not os.path.exists(zone_file_path):
                zone_rows.to_csv(zone_file_path, index=False)
            else:
                zone_rows.to_csv(zone_file_path, mode='a', index=False, header=False)
            
            # Remove the copied rows from the original DataFrame
            second_df.drop(zone_rows.index, inplace=True)
            
            # Print message
            print(f"Copied rows to {zone_file_path} for zone {zone}.")
    
    # Write the updated DataFrame back to the second file
    second_df.to_csv(file_path, index=False)
    print(f"Original file {file_path} updated after row deletion.")

<div style="background-color: black;">
<hr style="border: 2px solid skyblue;">
<div style="text-align: justify; margin-left: 3.0em; font-weight: bold; font-size: 18px; font-family: TimesNewRoman;color:skyblue">
    9. Zone Clustering
</div>
<div style="text-align: justify; margin-left: 0.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman;color:skyblue">
    Grouping power units according the location, the company or the name. 
</div>
<div style="text-align: justify; margin-left: 2.0em; font-weight: unbold; font-size: 13px; font-family: TimesNewRoman;color:skyblue">
    This part is set to make a cluster of all the units that have the same Latitud and Longitud location (if the data is provided)
     <br>
    However, there is the posibility to cluster the units by the name for all the units that has the same name or by the company (if the data is also provided). 
</div>
<div style="text-align: justify; margin-left: 4.0em; font-weight: unbold; font-size: 12px; font-family: TimesNewRoman;color:skyblue">
    To do it, just Uncomment the corresponding line of the following lines of the code.
</div>
<div style="text-align: left; margin-left: 3.0em; font-weight: bold; font-size: 17px; font-family: TimesNewRoman;">
</div>
<br>
<hr style="border: 1px solid skyblue;">
<div style="text-align: justify; margin-left: 3.0em; font-weight: bold; font-size: 18px; font-family: TimesNewRoman;color:skyblue">
    9.1. Lat/Lon Column Creation
</div>
<div style="text-align: justify; margin-left: 0.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman;color:skyblue">
    Joining the filed of the columns Lat and Lon into a new column under the name Lat/Lon to clustering purposes.
</div>

In [73]:
#created_zones = {
#    'DE': '/home/ray/Dispa-SET_Unleash/RawData/PowerPlants/DE',
#    'DK': '/home/ray/Dispa-SET_Unleash/RawData/PowerPlants/DK',
#    'CH': '/home/ray/Dispa-SET_Unleash/RawData/PowerPlants/CH',
#    'BE': '/home/ray/Dispa-SET_Unleash/RawData/PowerPlants/BE'
#}

#data_year = '2020'

power_plants_clean_data_file_path_list = []

# Extract and format paths
for zone, path in created_zones.items():
  full_path = f"{path}/{data_year}.csv"
  power_plants_clean_data_file_path_list.append(full_path)

# Print the list of paths
print(power_plants_clean_data_file_path_list)

['/home/ray/Dispa-SET_Unleash/RawData/PowerPlants/AT/2024.csv', '/home/ray/Dispa-SET_Unleash/RawData/PowerPlants/BE/2024.csv', '/home/ray/Dispa-SET_Unleash/RawData/PowerPlants/BG/2024.csv', '/home/ray/Dispa-SET_Unleash/RawData/PowerPlants/CH/2024.csv', '/home/ray/Dispa-SET_Unleash/RawData/PowerPlants/CY/2024.csv', '/home/ray/Dispa-SET_Unleash/RawData/PowerPlants/CZ/2024.csv', '/home/ray/Dispa-SET_Unleash/RawData/PowerPlants/DE/2024.csv', '/home/ray/Dispa-SET_Unleash/RawData/PowerPlants/DK/2024.csv', '/home/ray/Dispa-SET_Unleash/RawData/PowerPlants/EE/2024.csv', '/home/ray/Dispa-SET_Unleash/RawData/PowerPlants/EL/2024.csv', '/home/ray/Dispa-SET_Unleash/RawData/PowerPlants/ES/2024.csv', '/home/ray/Dispa-SET_Unleash/RawData/PowerPlants/FI/2024.csv', '/home/ray/Dispa-SET_Unleash/RawData/PowerPlants/FR/2024.csv', '/home/ray/Dispa-SET_Unleash/RawData/PowerPlants/HR/2024.csv', '/home/ray/Dispa-SET_Unleash/RawData/PowerPlants/HU/2024.csv', '/home/ray/Dispa-SET_Unleash/RawData/PowerPlants/IE/20

<div style="background-color: black;">
<div style="text-align: right; margin-left: 3.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman;color:skyblue">
    Tracking Variables. 
    <br>
    <div style="text-align: right; margin-left: 1.50em; font-weight: unbold; font-size: 13px; font-family: TimesNewRoman;">
    This cells are just to confirm all the file names, file paths and other information related to the data being processed.
    <br>
  Also are used to ensure the inputs for next cells in order to avoid to re-enter the same information each time.
</div>

In [74]:
power_plants_clean_data_file_path_list

['/home/ray/Dispa-SET_Unleash/RawData/PowerPlants/AT/2024.csv',
 '/home/ray/Dispa-SET_Unleash/RawData/PowerPlants/BE/2024.csv',
 '/home/ray/Dispa-SET_Unleash/RawData/PowerPlants/BG/2024.csv',
 '/home/ray/Dispa-SET_Unleash/RawData/PowerPlants/CH/2024.csv',
 '/home/ray/Dispa-SET_Unleash/RawData/PowerPlants/CY/2024.csv',
 '/home/ray/Dispa-SET_Unleash/RawData/PowerPlants/CZ/2024.csv',
 '/home/ray/Dispa-SET_Unleash/RawData/PowerPlants/DE/2024.csv',
 '/home/ray/Dispa-SET_Unleash/RawData/PowerPlants/DK/2024.csv',
 '/home/ray/Dispa-SET_Unleash/RawData/PowerPlants/EE/2024.csv',
 '/home/ray/Dispa-SET_Unleash/RawData/PowerPlants/EL/2024.csv',
 '/home/ray/Dispa-SET_Unleash/RawData/PowerPlants/ES/2024.csv',
 '/home/ray/Dispa-SET_Unleash/RawData/PowerPlants/FI/2024.csv',
 '/home/ray/Dispa-SET_Unleash/RawData/PowerPlants/FR/2024.csv',
 '/home/ray/Dispa-SET_Unleash/RawData/PowerPlants/HR/2024.csv',
 '/home/ray/Dispa-SET_Unleash/RawData/PowerPlants/HU/2024.csv',
 '/home/ray/Dispa-SET_Unleash/RawData/Po

In [75]:
for file_path in power_plants_clean_data_file_path_list:
  # Read the CSV file
  df = pd.read_csv(file_path)

  # Check if "Lat" and "Lon" columns exist
  if 'Lat' in df.columns and 'Lon' in df.columns:
    # Create a new column named "Lat+Lon" with empty string as default
    df['Lat/Lon'] = np.where((df['Lat'].notna()) & (df['Lon'].notna()), df['Lat'].astype(str) + "/" + df['Lon'].astype(str), '')

    # Save the modified DataFrame back to the CSV file
    df.to_csv(file_path, index=False)
  else:
    print(f"Warning: 'Lat' or 'Lon' column not found in {file_path}")

print("Lat/Lon column added to files (if columns exist).")

Lat/Lon column added to files (if columns exist).


<div style="background-color: black;">
<hr style="border: 1px solid skyblue;">
<div style="text-align: justify; margin-left: 3.0em; font-weight: bold; font-size: 17px; font-family: TimesNewRoman;color:skyblue">
    9.2. Fields Cluster
</div>
<div style="text-align: justify; margin-left: 0.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman;color:skyblue">
    Clustering the units with the repeated values in the Lat/Lon.
</div>
<div style="text-align: justify; margin-left: 2.0em; font-weight: unbold; font-size: 13px; font-family: TimesNewRoman;color:skyblue">
    The next code just clusters the units which Latitud and longitud repeated values under the condition that have the same Technology, Fuel and CHPType as well.
</div>
<div style="text-align: justify; margin-left: 0.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman;color:skyblue">
    This process is made for each zone.
    <br>
    There is the posibility to make a cluster looking for duplicate fields in the name of the units or the company (If the data exists). 
</div>
<div style="text-align: justify; margin-left: 2.0em; font-weight: unbold; font-size: 13px; font-family: TimesNewRoman;color:skyblue">    
For this purpouse just uncomend the corresponding lines.
</div>

In [76]:
# Column to be looked for
column_to_look_for = 'Lat/Lon'  # Change 'YourColumnNameHere' to the actual column name
#column_to_look_for = 'Unit'
#column_to_look_for = 'Company'


# Iterate over each file
for file_path in power_plants_clean_data_file_path_list:
    # Read the CSV file into a DataFrame
    df = pd.read_csv(file_path)

    # Check if the column to look for exists
    if column_to_look_for in df.columns:
        # Iterate over each value in the specified column
        for value in df[column_to_look_for].unique():
            # Check if the field has some value
            if pd.notnull(value):
                # Filter rows where the specified column has the exact same value
                matched_rows = df[df[column_to_look_for] == value]

                # Check if there are more than one row with the same value
                if len(matched_rows) > 1:
                    # Perform additional matching based on Technology, Fuel, and CHPType
                    # You can adjust the conditions as per your requirement
                    matched_rows = matched_rows.groupby(['Technology', 'Fuel', 'CHPType']).filter(lambda x: len(x) > 1)

                    if len(matched_rows) > 1:
                        # Sum the corresponding fields
                        summed_fields = matched_rows[['PowerCapacity', 'Nunits', 'StartUpCost', 'NoLoadCost_pu',
                                                       'RampingCost', 'CHPPowerToHeat', 'CHPMaxHeat', 'STOCapacity',
                                                       'WaterWithdrawal', 'WaterConsumption']].sum()

                        # Calculate the average of the corresponding fields
                        averaged_fields = matched_rows[['Efficiency', 'MinUpTime', 'MinDownTime', 'RampUpRate',
                                                         'RampDownRate', 'PartLoadMin', 'MinEfficiency', 'StartUpTime',
                                                         'CO2Intensity', 'CHPPowerLossFactor', 'CHPMaxHeat', 'COP',
                                                         'Tnominal', 'coef_COP_a', 'coef_COP_b', 'STOSelfDischarge',
                                                         'STOMaxChargingPower', 'STOChargingEfficiency']].mean()

                        # Keep the first row and update its values with the sums and averages
                        first_row_index = matched_rows.index[0]
                        df.loc[first_row_index, ['PowerCapacity', 'Nunits', 'StartUpCost', 'NoLoadCost_pu',
                                                 'RampingCost', 'CHPPowerToHeat', 'CHPMaxHeat', 'STOCapacity',
                                                 'WaterWithdrawal', 'WaterConsumption']] = summed_fields
                        df.loc[first_row_index, ['Efficiency', 'MinUpTime', 'MinDownTime', 'RampUpRate',
                                                 'RampDownRate', 'PartLoadMin', 'MinEfficiency', 'StartUpTime',
                                                 'CO2Intensity', 'CHPPowerLossFactor', 'CHPMaxHeat', 'COP',
                                                 'Tnominal', 'coef_COP_a', 'coef_COP_b', 'STOSelfDischarge',
                                                 'STOMaxChargingPower', 'STOChargingEfficiency']] = averaged_fields

                        # Copy the matched rows to a new DataFrame
                        clustered_df = matched_rows.copy()

                        # Write the matched rows to a new CSV file
                        output_file_path = os.path.splitext(file_path)[0] + '_clustered.csv'
                        clustered_df.to_csv(output_file_path, index=False)

                        # Erase the rows from the original DataFrame
                        df.drop(matched_rows.index, inplace=True)

    # Write the modified DataFrame back to the original CSV file
    df.to_csv(file_path, index=False)

<div style="background-color: black;">
<hr style="border: 2px solid skyblue;">
<div style="text-align: justify; margin-left: 3.0em; font-weight: bold; font-size: 17px; font-family: TimesNewRoman;color:skyblue">
    10. Storage Capacity Data
</div>
<div style="text-align: justify; margin-left: 0.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman;color:skyblue">
    Filling the 'Storage Capacity' fields of each power unit of the power plant data.
</div>
<div style="text-align: justify; margin-left: 2.0em; font-weight: unbold; font-size: 13px; font-family: TimesNewRoman;color:skyblue">
    Data from the 'JRC Hydro-power database' source is used, the same can be dowloaded from:
<div style="text-align: justify; margin-left: 4.0em; font-weight: normal; font-size: 12px; font-family: Times New Roman;color:skyblue">
    <a href="https://zenodo.org/records/5215920" target="_blank">https://zenodo.org/records/5215920</a>
</div>
<div style="text-align: justify; margin-left: 0.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman;color:skyblue">
    The source data is found in the following folder into the Dispa-SET local directory.
</div>
<div style="text-align: justify; margin-left: 2.0em; font-weight: normal; font-size: 13px; font-family: Times New Roman;color:skyblue">
    /local/ditectory/Dispa-SET_Unleash/RawData/PowerPlants/EU_Power_Units_Raw_Data_Source/Hydro_Units
</div>

In [77]:
additional_path_2 = "/RawData/PowerPlants/EU_Power_Units_Raw_Data_Source/Hydro_Units/jrc-hydro-power-plant-database.csv"
hydro_units_raw_data_file_path = dispaSET_unleash_folder_path + additional_path_2
print (f"hydro_units_raw_data_file_path: {hydro_units_raw_data_file_path}")

hydro_units_raw_data_file_path: /home/ray/Dispa-SET_Unleash/RawData/PowerPlants/EU_Power_Units_Raw_Data_Source/Hydro_Units/jrc-hydro-power-plant-database.csv


In [78]:
# Read the hydro units CSV file
hydro_units_df = pd.read_csv(hydro_units_raw_data_file_path)

for file_path_1 in power_plants_clean_data_file_path_list:
    # Read the current CSV file
    file_1_df = pd.read_csv(file_path_1)

    # Extract unique zones from the current file
    unique_zones = file_1_df['Zone'].unique()

    # Filter hydro_units_df by country_code
    filtered_hydro_units_df = hydro_units_df[hydro_units_df['country_code'].isin(unique_zones)]

    # Further filter by type 'HDAM' or 'HPHS'
    filtered_hydro_units_df = filtered_hydro_units_df[filtered_hydro_units_df['type'].isin(['HDAM', 'HPHS'])]

    # Filter file_1_df by Technology 'HDAM' or 'HPHS'
    filtered_file_1_df = file_1_df[file_1_df['Technology'].isin(['HDAM', 'HPHS'])]

    # Create a mapping from installed_capacity_MW to storage_capacity_MWh
    capacity_to_storage = filtered_hydro_units_df.set_index('installed_capacity_MW')['storage_capacity_MWh'].to_dict()

    # Define a function to update 'STOCapacity'
    def update_storage_capacity(row):
        power_capacity = row['PowerCapacity']
        return capacity_to_storage.get(power_capacity, None)

    # Apply the update function
    filtered_file_1_df['STOCapacity'] = filtered_file_1_df.apply(update_storage_capacity, axis=1)

    # Identify rows with no direct match
    no_match_df = filtered_file_1_df[filtered_file_1_df['STOCapacity'].isnull()]

    # Calculate the total storage capacity from unmatched hydro units
    unmatched_hydro_units_df = filtered_hydro_units_df[
        ~filtered_hydro_units_df['installed_capacity_MW'].isin(filtered_file_1_df['PowerCapacity'])
    ]
    total_unmatched_storage_capacity = unmatched_hydro_units_df['storage_capacity_MWh'].sum()

    # Calculate the proportional storage capacity for unmatched rows
    if not no_match_df.empty:
        total_power_capacity = no_match_df['PowerCapacity'].sum()
        if total_power_capacity > 0:
            no_match_df['STOCapacity'] = (no_match_df['PowerCapacity'] / total_power_capacity) * total_unmatched_storage_capacity

        # Check the STOCapacity/PowerCapacity ratio and update if necessary
        def adjust_sto_capacity(row):
            sto_capacity = row['STOCapacity']
            power_capacity = row['PowerCapacity']

            if sto_capacity is not None and power_capacity > 0:
                # Divide STOCapacity by PowerCapacity
                ratio = sto_capacity / power_capacity
                # If ratio exceeds 70, update STOCapacity to PowerCapacity * 70
                if ratio > 70:
                    sto_capacity = power_capacity * 70
            return sto_capacity

        # Apply the adjustment to no_match_df
        no_match_df['STOCapacity'] = no_match_df.apply(adjust_sto_capacity, axis=1)

        # Update the filtered file DataFrame
        filtered_file_1_df.update(no_match_df)

    # Update the original DataFrame
    file_1_df.update(filtered_file_1_df)

    # Save the updated original DataFrame to the same CSV file
    file_1_df.to_csv(file_path_1, index=False)


/tmp/ipykernel_1092850/1376709539.py:29: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_file_1_df['STOCapacity'] = filtered_file_1_df.apply(update_storage_capacity, axis=1)
/tmp/ipykernel_1092850/1376709539.py:44: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  no_match_df['STOCapacity'] = (no_match_df['PowerCapacity'] / total_power_capacity) * total_unmatched_storage_capacity
/tmp/ipykernel_1092850/1376709539.py:60: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a Data

<div style="background-color: black;">
<hr style="border: 2px solid skyblue;">
<div style="text-align: justify; margin-left: 3.0em; font-weight: bold; font-size: 17px; font-family: TimesNewRoman;color:skyblue">
    11. Self-Discharge Rate | Maximum Charging Power | Charging efficiency  Data 
</div>
<div style="text-align: justify; margin-left: 0.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman;color:skyblue">
    Filling the 'STOSelfDischarge', 'STOMaxChargingPower', 'STOChargingEfficiency' fields of each .corresponding Hydro unit of the power plant data.
</div>

In [79]:
def process_csv_files(power_plants_raw_data_sources_file_path):
    """
    Processes CSV files, modifying values for HPHS and HDAM technologies.

    Args:
        power_plants_raw_data_sources_file_path: The path to the CSV file containing file paths.
    """

    df = pd.read_csv(power_plants_raw_data_sources_file_path)

    for index, row in df.iterrows():
        file_path = row['Final_Clean_File_Path']
        if pd.notnull(file_path):
            process_csv_file(file_path)

def process_csv_file(file_path):
    """
    Processes a CSV file, modifying values for HPHS and HDAM technologies.

    Args:
        file_path: The path to the CSV file.
    """

    df = pd.read_csv(file_path)

    # Process HPHS rows
    df.loc[df['Technology'] == 'HPHS', 'STOSelfDischarge'] = 0
    df.loc[df['Technology'] == 'HPHS', 'STOMaxChargingPower'] = df.loc[df['Technology'] == 'HPHS', 'PowerCapacity']
    df.loc[df['Technology'] == 'HPHS', 'STOChargingEfficiency'] = df.loc[df['Technology'] == 'HPHS', 'Efficiency']

    # Process HDAM rows
    df.loc[df['Technology'] == 'HDAM', 'STOSelfDischarge'] = 0
    df.loc[df['Technology'] == 'HDAM', 'STOMaxChargingPower'] = 0
    df.loc[df['Technology'] == 'HDAM', 'STOChargingEfficiency'] = 0

    df.to_csv(file_path, index=False)
    print(f"Processed file: {file_path}")

process_csv_files(power_plants_raw_data_sources_file_path)

Processed file: /home/ray/Dispa-SET_Unleash/RawData/PowerPlants/AT/2024.csv
Processed file: /home/ray/Dispa-SET_Unleash/RawData/PowerPlants/BE/2024.csv
Processed file: /home/ray/Dispa-SET_Unleash/RawData/PowerPlants/BG/2024.csv
Processed file: /home/ray/Dispa-SET_Unleash/RawData/PowerPlants/CH/2024.csv
Processed file: /home/ray/Dispa-SET_Unleash/RawData/PowerPlants/CY/2024.csv
Processed file: /home/ray/Dispa-SET_Unleash/RawData/PowerPlants/CZ/2024.csv
Processed file: /home/ray/Dispa-SET_Unleash/RawData/PowerPlants/DE/2024.csv
Processed file: /home/ray/Dispa-SET_Unleash/RawData/PowerPlants/DE/2024.csv
Processed file: /home/ray/Dispa-SET_Unleash/RawData/PowerPlants/DE/2024.csv
Processed file: /home/ray/Dispa-SET_Unleash/RawData/PowerPlants/DE/2024.csv
Processed file: /home/ray/Dispa-SET_Unleash/RawData/PowerPlants/DK/2024.csv
Processed file: /home/ray/Dispa-SET_Unleash/RawData/PowerPlants/EE/2024.csv
Processed file: /home/ray/Dispa-SET_Unleash/RawData/PowerPlants/EL/2024.csv
Processed fi

<div style="background-color: black;">
<div style="text-align: right; margin-left: 3.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman;color:skyblue">
    Tracking Variables. 
    <br>
    <div style="text-align: right; margin-left: 1.50em; font-weight: unbold; font-size: 13px; font-family: TimesNewRoman;">
    This cells are just to confirm all the file names, file paths and other information related to the data being processed.
    <br>
  Also are used to ensure the inputs for next cells in order to avoid to re-enter the same information each time.
</div>

In [80]:
power_plants_clean_data_file_path_list

['/home/ray/Dispa-SET_Unleash/RawData/PowerPlants/AT/2024.csv',
 '/home/ray/Dispa-SET_Unleash/RawData/PowerPlants/BE/2024.csv',
 '/home/ray/Dispa-SET_Unleash/RawData/PowerPlants/BG/2024.csv',
 '/home/ray/Dispa-SET_Unleash/RawData/PowerPlants/CH/2024.csv',
 '/home/ray/Dispa-SET_Unleash/RawData/PowerPlants/CY/2024.csv',
 '/home/ray/Dispa-SET_Unleash/RawData/PowerPlants/CZ/2024.csv',
 '/home/ray/Dispa-SET_Unleash/RawData/PowerPlants/DE/2024.csv',
 '/home/ray/Dispa-SET_Unleash/RawData/PowerPlants/DK/2024.csv',
 '/home/ray/Dispa-SET_Unleash/RawData/PowerPlants/EE/2024.csv',
 '/home/ray/Dispa-SET_Unleash/RawData/PowerPlants/EL/2024.csv',
 '/home/ray/Dispa-SET_Unleash/RawData/PowerPlants/ES/2024.csv',
 '/home/ray/Dispa-SET_Unleash/RawData/PowerPlants/FI/2024.csv',
 '/home/ray/Dispa-SET_Unleash/RawData/PowerPlants/FR/2024.csv',
 '/home/ray/Dispa-SET_Unleash/RawData/PowerPlants/HR/2024.csv',
 '/home/ray/Dispa-SET_Unleash/RawData/PowerPlants/HU/2024.csv',
 '/home/ray/Dispa-SET_Unleash/RawData/Po

<div style="background-color: black;">
<hr style="border: 2px solid skyblue;">
<div style="text-align: justify; margin-left: 3.0em; font-weight: bold; font-size: 17px; font-family: TimesNewRoman;color:skyblue">
11. Copying Power Plants Formatted Data
</div>
<div style="text-align: justify; margin-left: 0.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman;color:skyblue">
    Moving the already clean and formatted data of the power units to the main Dispa-SET data base directory.
     <br>
    <div style="text-align: justify; margin-left: 2.0em; font-weight: unbold; font-size: 13px; font-family: TimesNewRoman;color:skyblue">
The data base directory has to be identified.
</div>

In [81]:
additional_path_4 = "/Database/PowerPlants/"

# Construct the power_plants_raw_data_folder_path variable
power_plants_data_base_folder_path = dispaSET_unleash_folder_path + additional_path_4

In [82]:
# Iterate over each zone in the list
for zone in zone_names:
    # Define the source file path
    source_file_path = os.path.join(power_plants_raw_data_folder_path, zone, f"{data_year}.csv")
    
    # Define the destination folder path
    destination_folder_path = os.path.join(power_plants_data_base_folder_path, zone)
    
    # Define the destination file path
    destination_file_path = os.path.join(destination_folder_path, f"{data_year}.csv")
    
    # Check if the source file exists
    if os.path.exists(source_file_path):
        # Ensure the destination folder exists
        os.makedirs(destination_folder_path, exist_ok=True)
        
        # Copy the file
        shutil.copyfile(source_file_path, destination_file_path)
        print(f"Copied {source_file_path} to {destination_file_path}")
    else:
        print(f"Source file does not exist: {source_file_path}")

print("Task completed.")

Copied /home/ray/Dispa-SET_Unleash/RawData/PowerPlants/AT/2024.csv to /home/ray/Dispa-SET_Unleash/Database/PowerPlants/AT/2024.csv
Copied /home/ray/Dispa-SET_Unleash/RawData/PowerPlants/BE/2024.csv to /home/ray/Dispa-SET_Unleash/Database/PowerPlants/BE/2024.csv
Copied /home/ray/Dispa-SET_Unleash/RawData/PowerPlants/BG/2024.csv to /home/ray/Dispa-SET_Unleash/Database/PowerPlants/BG/2024.csv
Copied /home/ray/Dispa-SET_Unleash/RawData/PowerPlants/CH/2024.csv to /home/ray/Dispa-SET_Unleash/Database/PowerPlants/CH/2024.csv
Copied /home/ray/Dispa-SET_Unleash/RawData/PowerPlants/CY/2024.csv to /home/ray/Dispa-SET_Unleash/Database/PowerPlants/CY/2024.csv
Copied /home/ray/Dispa-SET_Unleash/RawData/PowerPlants/CZ/2024.csv to /home/ray/Dispa-SET_Unleash/Database/PowerPlants/CZ/2024.csv
Copied /home/ray/Dispa-SET_Unleash/RawData/PowerPlants/DE/2024.csv to /home/ray/Dispa-SET_Unleash/Database/PowerPlants/DE/2024.csv
Copied /home/ray/Dispa-SET_Unleash/RawData/PowerPlants/DK/2024.csv to /home/ray/Dis

<div style="background-color: black;">
<hr style="border: 2px solid skyblue;">
<div style="text-align: justify; margin-left: 3.0em; font-weight: bold; font-size: 17px; font-family: TimesNewRoman; color:skyblue">
12. Power Plants Folder Back Up
</div>
<div style="text-align: justify; margin-left: 0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
Once all the formating process was done the Power Plants Folder is restored to its defoult state.
</div>

In [83]:
if os.path.exists(power_plants_raw_data_folder_path):
    shutil.rmtree(power_plants_raw_data_folder_path)  # Remove the current directory
shutil.copytree(backup_folder_path, power_plants_raw_data_folder_path)

print(f"Directory restored to original state from {backup_folder_path}")

Directory restored to original state from /home/ray/Dispa-SET_Unleash/RawData/PowerPlants_backup/


In [84]:
shutil.rmtree(backup_folder_path)
print(f"Backup folder {backup_folder_path} deleted successfully.")

Backup folder /home/ray/Dispa-SET_Unleash/RawData/PowerPlants_backup/ deleted successfully.


<div style="background-color: black;">
<hr style="border: 3px solid skyblue;">